# Beer Distribution Game: LLM vs Human-Calibrated Benchmark Analysis

End-to-end pipeline:
1. Run the mass-conserving Sterman-style Beer Game simulator.
2. Load pre-extracted LLM simulation results from JSON files.
3. Reconstruct the 11 Sterman-calibrated benchmark sessions.
4. Compare LLM trajectories with the benchmark using order plots, bullwhip, and a **relative log-MSE** score.
5. Test baseline, heuristic-aware, and cognitive-load conditions with session-level statistics.
6. Retain temporal information through peak timing, overshoot-and-collapse, late-period error, zero-order runs, autocorrelation, and individual-session trajectory diagnostics.
7. Generate the figures and CSV tables used in the revised paper and response letter.

The relative log-MSE score is a trajectory-distance comparison, not a likelihood for the LLM decision process.


In [ ]:
# ── Project paths ─────────────────────────────────────────────────
import os
from pathlib import Path

# Input files are expected under:
#   /code/beer data/<model-folder>/<file>
# Generated figures and CSV tables are written under:
#   /results/<filename>
DATA_ROOT = Path("/code/beer data")
RESULTS_ROOT = Path("/results")

# Keep the existing variable name used by downstream plotting/export cells.
PLOTS_DIR = str(RESULTS_ROOT)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# ── Standard scientific stack ─────────────────────────────────────
import json
import math
import random
import datetime
import warnings
from glob import glob
from collections import deque
from typing import (
    Any, Dict, Iterable, List, Optional, Sequence, Tuple, Union,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable

from scipy.special import gammaln
from scipy.stats import (
    pearsonr,
    spearmanr,
    ttest_ind,
    ttest_1samp,
    mannwhitneyu,
    wilcoxon,
    t as student_t,
)

# Roles used throughout
ROLES = ["retailer", "wholesaler", "distributor", "factory"]
ROLE_KEYS = {role: f"{role}_order" for role in ROLES}


def resolve_model_dirs(
    folder_names: Iterable[str],
    *,
    label: str,
    require_any: bool = True,
) -> Dict[str, str]:
    """
    Resolve model folders below DATA_ROOT and keep only folders containing
    at least one extracted session JSON. Missing optional model folders are
    reported but do not stop a partial analysis.
    """
    resolved: Dict[str, str] = {}
    missing: List[str] = []

    for folder_name in folder_names:
        path = DATA_ROOT / folder_name
        has_json = path.is_dir() and any(path.glob("*_extracted.json"))
        if has_json:
            resolved[folder_name] = str(path)
        else:
            missing.append(folder_name)

    if missing:
        print(
            f"[{label}] Skipping {len(missing)} unavailable folder(s): "
            + ", ".join(missing)
        )

    if require_any and not resolved:
        raise FileNotFoundError(
            f"No usable {label} model directories were found below {DATA_ROOT}. "
            "Upload the model directories below /code/beer data or update DATA_ROOT in the setup cell."
        )

    return resolved


def filter_pairs_to_available_models(
    pairs: Iterable[Tuple[str, str, str]],
    data: pd.DataFrame,
) -> List[Tuple[str, str, str]]:
    """Keep only model-pair definitions present in a MultiIndex result table."""
    if not isinstance(data.index, pd.MultiIndex):
        raise ValueError("data must use a MultiIndex containing a 'model' level.")

    available = set(data.index.get_level_values("model"))
    kept = [
        pair
        for pair in pairs
        if pair[1] in available and pair[2] in available
    ]
    omitted = [pair[0] for pair in pairs if pair not in kept]

    if omitted:
        print("Skipping unavailable pair(s): " + ", ".join(omitted))

    return kept


print(f"DATA_ROOT: {DATA_ROOT}")
print(f"PLOTS_DIR: {PLOTS_DIR}")
if not DATA_ROOT.exists():
    print(
        "WARNING: DATA_ROOT does not yet exist. The simulator-only cells can "
        "run, but LLM analyses require the extracted JSON directories."
    )


### Figure style

Shared style and helpers for the paper figures: 7-pt sans-serif text, vector PDF with editable text, keys inside each figure, and individual sessions drawn as dots over the bars. Each figure keeps its original colours; colours that failed a colour-blindness check were replaced by the nearest passing colour. The style is applied only inside the figure functions, so other plots keep matplotlib's defaults.

In [ ]:
# Figure style for the paper figures (Nature format): 7-pt sans-serif text, vector PDF with
# editable text, keys inside the figure, and individual sessions shown as dots aligned in
# their own column. The style is applied only inside the figure functions (via @nature_style),
# so other plots in this notebook keep matplotlib's defaults.
# Colours: each figure keeps its original colours; colours that failed a colour-blindness check
# (every pair at least as distinguishable as the Okabe-Ito palette under simulated deuteranopia,
# protanopia and tritanopia) were replaced by the nearest passing colour.
import matplotlib as mpl
import matplotlib.font_manager
from functools import wraps
from matplotlib.lines import Line2D

MM = 1 / 25.4
FULL_WIDTH, HALF_WIDTH = 180 * MM, 88 * MM
TAB_BLUE, TAB_ORANGE = "#1f77b4", "#ff7f0e"
ROLE_COLORS = {"OVERALL": "#66c2a5", "retailer": "#f0835f", "wholesaler": "#8da0cb",
               "distributor": "#c575a0", "factory": "#a6d854"}   # Set2, two shades repaired
ROLE_LABELS = {"OVERALL": "Overall", "retailer": "Retailer", "wholesaler": "Wholesaler",
               "distributor": "Distributor", "factory": "Factory"}
CATS = ["OVERALL", "retailer", "wholesaler", "distributor", "factory"]
BULLWHIP_COLORS = {"Kimi K2": "#fc8d62", "Qwen-3.6-35B-A3B": "#66c2a5",
                   "low CR": "#756dbd", "high CR": "#e7298a"}
DOT_COLOR = "#222222"
HATCH = "////"
DOT_SEP = {"linear": 0.1, "log": 0.02}   # set per figure by set_dot_spacing()

# Arial where installed (Nature's preferred font); otherwise the closest metric-compatible sans.
_INSTALLED = {f.name for f in mpl.font_manager.fontManager.ttflist}
SANS = next((f for f in ["Arial", "Helvetica", "Liberation Sans"] if f in _INSTALLED), "DejaVu Sans")
NATURE_RC = {
    "font.family": "sans-serif", "font.sans-serif": [SANS, "DejaVu Sans"],
    "mathtext.fontset": "custom", "mathtext.rm": SANS, "mathtext.it": f"{SANS}:italic",
    "mathtext.bf": f"{SANS}:bold", "font.size": 7, "axes.titlesize": 7, "axes.labelsize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6, "legend.title_fontsize": 6,
    "axes.linewidth": 0.5, "xtick.major.width": 0.5, "ytick.major.width": 0.5,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "lines.linewidth": 1.0, "patch.linewidth": 0.5,
    "hatch.linewidth": 0.4, "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "legend.frameon": False, "savefig.dpi": 600,
}


def nature_style(func):
    """Draw (and save) the figure under the paper's figure style."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        with mpl.rc_context(NATURE_RC):
            return func(*args, **kwargs)
    return wrapper


def save_figure(fig, savepath):
    if savepath:
        fig.savefig(savepath, bbox_inches="tight", pad_inches=0.02)


def lighten(color, amount=0.45):
    r, g, b = mpl.colors.to_rgb(color)
    return (1 - amount * (1 - r), 1 - amount * (1 - g), 1 - amount * (1 - b))


def star(p):
    if not np.isfinite(p):
        return ""
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else ""


def sem(v):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    return v.std(ddof=1) / np.sqrt(v.size) if v.size > 1 else np.nan


def geo_ci(v):
    """Geometric mean with a 95% t interval on the log scale."""
    v = np.asarray(v, float)
    v = v[np.isfinite(v) & (v > 0)]
    lg = np.log(v)
    h = student_t.ppf(0.975, v.size - 1) * lg.std(ddof=1) / np.sqrt(v.size)
    return np.exp(lg.mean()), np.exp(lg.mean() - h), np.exp(lg.mean() + h)


def set_dot_spacing(all_values, log=False, frac=0.012):
    """Vertical distance below which two dots in the same column are shifted sideways."""
    v = np.asarray([x for x in np.ravel(all_values) if np.isfinite(x) and (x > 0 or not log)], float)
    v = np.log10(v) if log else v
    DOT_SEP["log" if log else "linear"] = max((v.max() - v.min()) * frac, 1e-9)


def swarm_offsets(values, y_sep, x_step, max_half):
    """Deterministic beeswarm: each dot sits on the column centre unless it would overlap another."""
    order = np.argsort(values)
    placed, offs = [], np.zeros(len(values))
    for idx in order:
        y = values[idx]
        cand = 0.0
        for k in range(1, 60):
            if all(abs(y - py) >= y_sep or abs(cand - px) >= x_step * 0.999 for px, py in placed):
                break
            cand = ((k + 1) // 2) * x_step * (1 if k % 2 else -1)
            if abs(cand) > max_half:
                cand = float(np.clip(cand, -max_half, max_half))
                break
        placed.append((cand, y))
        offs[idx] = cand
    return offs


def dots(ax, x, values, width, size=1.0, log=False):
    values = np.asarray(values, float)
    values = values[np.isfinite(values) & ((values > 0) if log else True)]
    yv = np.log10(values) if log else values
    offs = swarm_offsets(yv, DOT_SEP["log" if log else "linear"], width * 0.2, width * 0.42)
    ax.scatter(x + offs, values, s=size, color=DOT_COLOR, alpha=0.75, linewidths=0, zorder=4, clip_on=False)


def bar_with_dots(ax, x, values, color, width, baseline=False, err="sem", center=None, dot_size=3, show_dots=True):
    """Bar at the mean (or a supplied centre) with an error bar and the individual values as dots.

    err: "sem" (s.e.m. of `values`), None (no error bar) or an explicit (lower, upper) pair.
    Baseline bars are lighter and hatched.
    """
    values = np.asarray(values, float)
    if center is None:
        center = np.nanmean(values)
    ax.bar(x, center, width=width, color=lighten(color) if baseline else color,
           edgecolor="#555555" if baseline else "black", linewidth=0.4,
           hatch=HATCH if baseline else None, zorder=2)
    if err == "sem":
        e = sem(values)
        ax.errorbar(x, center, yerr=e, fmt="none", ecolor="black", elinewidth=0.5, capsize=1.2, zorder=5)
    elif err is not None:
        lo, hi = err
        ax.errorbar(x, center, yerr=[[center - lo], [hi - center]], fmt="none", ecolor="black",
                    elinewidth=0.5, capsize=1.2, zorder=5)
    if show_dots:
        dots(ax, x, values, width, size=dot_size, log=(ax.get_yscale() == "log"))


def condition_handles(labels=("Baseline", "Heuristic-aware"), dot_label="Session ($n=11$)", dot_size=2.5):
    handles = [Patch(facecolor="white", edgecolor="#555555", hatch=HATCH, label=labels[0]),
               Patch(facecolor="#999999", edgecolor="black", label=labels[1])]
    if dot_label:
        handles.append(Line2D([], [], marker="o", ls="", color=DOT_COLOR, alpha=0.75, markeredgewidth=0,
                              markersize=dot_size, label=dot_label))
    return handles


def session_values(per_sim, model, category, metric="Relative_log_MSE"):
    """Per-session values of `metric`; OVERALL averages the four roles within each session."""
    d = per_sim.copy()
    d.index = d.index.set_names(["model", "sim", "role"])
    d = d.reset_index()
    sub = d[d["model"] == model] if category == "OVERALL" else d[(d["model"] == model) & (d["role"] == category)]
    return sub.groupby("sim")[metric].mean().to_numpy(float)


def role_grouped_figure(groups, values_fn, *, conditions, ylabel, star_fn=None, center_fn=None, err_fn=None,
                        condition_labels=None, dot_note="Session ($n=11$)", height=80, width=FULL_WIDTH,
                        legend_loc="lower left", star_position="bar"):
    """Groups (models) on the x axis, five role categories within each group, one or two conditions per role.

    groups: [(label, key)]. conditions: [(condition_key, is_baseline)].
    values_fn(key, category, condition) -> per-session values (the dots).
    center_fn / err_fn override the bar height (mean) and the error bar (s.e.m.; heuristic-aware bars only).
    star_fn(key, category, condition) -> adjusted P value or a star string; stars are printed in one row,
    over their bar (star_position="bar") or centred over the category's pair of bars ("pair").
    legend_loc: a location inside the axes, (location, anchor) to place it at a given point,
    or "above" for one row above the plot.
    """
    fig, ax = plt.subplots(figsize=(width, height * MM))
    k = len(conditions)
    bw = 0.8 / (len(CATS) * k + 0.5)
    allv = np.concatenate([np.asarray(values_fn(g, c, cond), float) for _, g in groups for c in CATS
                           for cond, _ in conditions])
    allv = allv[np.isfinite(allv)]
    set_dot_spacing(allv)
    span = allv.max() - allv.min()
    star_y = max(allv.max(), 0) + span * 0.06
    any_star = False
    for gi, (_, g) in enumerate(groups):
        for ci, cat in enumerate(CATS):
            for j, (cond, base) in enumerate(conditions):
                x = gi + (ci - (len(CATS) - 1) / 2) * k * bw + (j - (k - 1) / 2) * bw
                v = values_fn(g, cat, cond)
                center = center_fn(g, cat, cond) if center_fn else None
                if err_fn:
                    c0 = center if center is not None else np.nanmean(v)
                    e = err_fn(g, cat, cond)
                    err = (c0 - e, c0 + e) if np.isfinite(e) else None
                else:
                    err = None if base else "sem"
                bar_with_dots(ax, x, v, ROLE_COLORS[cat], bw * 0.9, baseline=base, err=err, center=center, dot_size=1.0)
                if star_fn is not None:
                    s = star_fn(g, cat, cond)
                    s = s if isinstance(s, str) else star(s)
                    if s:
                        any_star = True
                        x_star = x if star_position == "bar" else gi + (ci - (len(CATS) - 1) / 2) * k * bw
                        ax.text(x_star, star_y, s, ha="center", va="bottom", fontsize=4.2, rotation=90)
    ax.set_xlim(-0.5, len(groups) - 0.5)
    ax.set_ylim(allv.min() - span * 0.03, (star_y + span * 0.07) if any_star else allv.max() + span * 0.04)
    ax.axhline(0, color="black", linewidth=0.5, zorder=1)
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels([label for label, _ in groups])
    ax.set_ylabel(ylabel)
    handles = [Patch(facecolor=ROLE_COLORS[c], edgecolor="black", label=ROLE_LABELS[c]) for c in CATS]
    if k == 2:
        handles += condition_handles(condition_labels or ("Baseline", "Heuristic-aware"), dot_label=dot_note, dot_size=1.5)
    else:
        handles += [Line2D([], [], marker="o", ls="", color=DOT_COLOR, alpha=0.75, markeredgewidth=0,
                           markersize=1.5, label=dot_note)]
    if legend_loc == "above":
        ncol = len(handles) if width >= FULL_WIDTH else 4
        ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=ncol,
                  columnspacing=1.2, handletextpad=0.5)
    elif isinstance(legend_loc, tuple):   # (loc, anchor in axes coordinates)
        ax.legend(handles=handles, loc=legend_loc[0], bbox_to_anchor=legend_loc[1], ncol=2)
    else:
        ax.legend(handles=handles, loc=legend_loc, ncol=2)
    return fig


def trajectory_panels(series, *, rounds, band=None, benchmark=None, benchmark_color="#7570b3",
                      human_curves=None, ylabel="Mean orders", height=100, ncol=3):
    """Four role panels of order trajectories.

    series: [dict(label, values={role: array}, color, ls)]. band / benchmark: {role: array} for the
    empirical benchmark (mean LOHO) and its 95% band as {role: (lower, upper)}.
    human_curves: [(label, {role: array}, colour)] x 2 for the low- and high-CR medians.
    """
    band_color = lighten(benchmark_color, 0.2)
    fig, axes = plt.subplots(2, 2, figsize=(FULL_WIDTH, height * MM))
    for ax, r in zip(axes.ravel(), ROLES):
        x = np.arange(rounds)   # round index (0-based), as in the original figures
        if benchmark is not None:
            lo, hi = band[r]
            ax.fill_between(x, lo[:rounds], hi[:rounds], color=band_color, linewidth=0, zorder=1)
            ax.plot(x, benchmark[r][:rounds], color=benchmark_color, lw=1.2, zorder=3)
        if human_curves:
            (_, c1, col1), (_, c2, col2) = human_curves
            ax.fill_between(x, c1[r][:rounds], c2[r][:rounds], color=lighten(col1, 0.15), linewidth=0, zorder=1)
            ax.plot(x, c1[r][:rounds], color=col1, ls=":", lw=1.2, zorder=3)
            ax.plot(x, c2[r][:rounds], color=col2, ls=":", lw=1.2, zorder=3)
        for s in series:
            ax.plot(x, s["values"][r][:rounds], color=s["color"], ls=s.get("ls", "-"), lw=1.1, zorder=4)
        if r == "retailer":
            ax.plot(x, np.where(x < 4, 4.0, 8.0), color="#555555", ls="-.", lw=0.8, zorder=2)
        ax.set_title(r.capitalize())
        ax.set_xlabel("Round")
        ax.set_ylabel(ylabel)
        ax.set_xlim(0, rounds - 1)
    handles = [Line2D([], [], color=s["color"], ls=s.get("ls", "-"), lw=1.1, label=s["label"]) for s in series]
    if benchmark is not None:
        handles += [Line2D([], [], color=benchmark_color, lw=1.2, label="Empirical benchmark (mean LOHO)"),
                    Patch(facecolor=band_color, label="95% band")]
    if human_curves:
        handles += [Line2D([], [], color=human_curves[0][2], ls=":", lw=1.2, label=human_curves[0][0]),
                    Line2D([], [], color=human_curves[1][2], ls=":", lw=1.2, label=human_curves[1][0]),
                    Patch(facecolor=lighten(human_curves[0][2], 0.15), label="Low–high CR range")]
    handles.append(Line2D([], [], color="#555555", ls="-.", lw=0.8, label="Customer demand"))
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=ncol)
    fig.tight_layout(h_pad=1.2, w_pad=1.5)
    return fig


def bullwhip_bars(spec, ticks, tick_labels, ylabel):
    """Bullwhip bars on a log axis: geometric mean, 95% t interval on the log scale, one dot per session.

    spec: [(x, label, colour, is_baseline, values)]; a single value is drawn as a plain bar.
    """
    fig, ax = plt.subplots(figsize=(HALF_WIDTH, 62 * MM))
    ax.set_yscale("log")
    all_values = np.concatenate([np.ravel(np.asarray(v, float)) for *_, v in spec])
    set_dot_spacing(all_values, log=True, frac=0.018)
    for x, _, color, base, values in spec:
        values = np.asarray(values, float)
        if values.size == 1:
            ax.bar(x, values[0], width=0.66, color=color, edgecolor="black", linewidth=0.4, zorder=2)
            continue
        g, lo, hi = geo_ci(values)
        bar_with_dots(ax, x, values, color, 0.66, baseline=base, err=(lo, hi), center=g, dot_size=2.2)
    ax.set_xticks(ticks)
    ax.set_xticklabels(tick_labels)
    positive = all_values[np.isfinite(all_values) & (all_values > 0)]
    ax.set_ylim(bottom=positive.min() * 0.7)
    ax.set_ylabel(ylabel)
    ax.legend(handles=condition_handles(), loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3,
              handlelength=1.4, columnspacing=1.0)
    return fig

## 1. Beer-Game Agent-Based Simulator

Mass-conserving ABM aligned with the latest LLM simulator:
- 35 rounds; demand = 4 for first 4 rounds, then 8.
- 2-round order delay, 2-round shipment delay, 2-round factory production lead time.
- Seeded equilibrium: 12 units on hand, 4-unit pipelines.
- First 4 rounds are forced warm-up (no ABM decisions).
- ABM decision rule after warm-up:
  $$IO_t = L_e + \alpha_s (S' - S_t - \beta\, SL_t) + \epsilon_t$$


In [ ]:
def simulate_beer_game(
    # Retailer parameters
    theta_r=0.36, alpha_s_r=0.26, beta_r=0.34, s_prime_r=17,
    # Wholesaler parameters
    theta_w=0.36, alpha_s_w=0.26, beta_w=0.34, s_prime_w=17,
    # Distributor parameters
    theta_d=0.36, alpha_s_d=0.26, beta_d=0.34, s_prime_d=17,
    # Factory parameters
    theta_f=0.36, alpha_s_f=0.26, beta_f=0.34, s_prime_f=17,
    # Noise + RNG
    sigma_eps=1,
    seed=42,
    # Environment
    weeks=35,
    initial_inventory=12,
    initial_throughput=4,
    initial_order=None,        # backward-compat alias for initial_throughput
    order_delay=2,
    shipment_lead_time=2,
    production_lead_time=2,
    forced_warmup_rounds=4,
    run_sanity_check=True,
):
    """
    Mass-conserving ABM Beer Game aligned with the latest LLM simulator.

    Returns a dict with: data (DataFrame), total_costs, total_team_cost,
    amplification, full_horizon_amplification, parameters, orders_placed, summary.
    """
    # Backward-compat: old code passes initial_order
    if initial_order is not None:
        initial_throughput = initial_order

    # ---- Validation ------------------------------------------------
    if weeks < 1:                 raise ValueError("weeks must be >= 1.")
    if order_delay < 1:           raise ValueError("order_delay must be >= 1.")
    if shipment_lead_time < 1:    raise ValueError("shipment_lead_time must be >= 1.")
    if production_lead_time < 1:  raise ValueError("production_lead_time must be >= 1.")
    if forced_warmup_rounds < 0:  raise ValueError("forced_warmup_rounds must be nonnegative.")

    rng = np.random.default_rng(seed)

    # ---- Per-role behavioral parameters ----------------------------
    params = {
        "retailer":    {"theta": theta_r, "alpha_s": alpha_s_r, "beta": beta_r, "s_prime": s_prime_r},
        "wholesaler":  {"theta": theta_w, "alpha_s": alpha_s_w, "beta": beta_w, "s_prime": s_prime_w},
        "distributor": {"theta": theta_d, "alpha_s": alpha_s_d, "beta": beta_d, "s_prime": s_prime_d},
        "factory":     {"theta": theta_f, "alpha_s": alpha_s_f, "beta": beta_f, "s_prime": s_prime_f},
    }
    # Box-constrain parameters for stability
    for p in params.values():
        p["theta"]   = min(1.0, max(0.0, p["theta"]))
        p["alpha_s"] = max(0.0, p["alpha_s"])
        p["beta"]    = min(1.0, max(0.0, p["beta"]))
        p["s_prime"] = max(0.0, p["s_prime"])

    # ---- Utility helpers ------------------------------------------
    def to_nonnegative_int(value, default=0):
        try:
            return max(0, int(round(float(value))))
        except (TypeError, ValueError):
            return default

    initial_inventory  = to_nonnegative_int(initial_inventory)
    initial_throughput = to_nonnegative_int(initial_throughput)

    # Exogenous customer demand: 4 during warmup, then 8
    customer_demand = [
        initial_throughput if w < forced_warmup_rounds else 8
        for w in range(weeks)
    ]

    # ---- Pipeline helpers (delays of orders & physical shipments) -
    def make_seeded_delay_pipeline(delay_length, qty=initial_throughput):
        # Pipelines are advanced at the start of each round
        return [{"qty": to_nonnegative_int(qty), "eta": eta}
                for eta in range(1, int(delay_length) + 1)]

    def advance_pipeline(pipeline):
        for item in pipeline:
            item["eta"] -= 1

    def pop_arrivals(pipeline):
        arriving  = [it for it in pipeline if it["eta"] <= 0]
        remaining = [it for it in pipeline if it["eta"]  > 0]
        pipeline.clear()
        pipeline.extend(remaining)
        return sum(to_nonnegative_int(it.get("qty", 0)) for it in arriving)

    def add_to_pipeline(pipeline, qty, eta):
        qty = to_nonnegative_int(qty)
        if qty > 0:
            pipeline.append({"qty": qty, "eta": int(eta)})

    def pipeline_sum(pipeline):
        return sum(to_nonnegative_int(it.get("qty", 0)) for it in pipeline)

    def eta_breakdown(pipeline, max_eta=4):
        return {
            f"eta_{eta}": sum(to_nonnegative_int(it.get("qty", 0))
                              for it in pipeline if int(it.get("eta", 0)) == eta)
            for eta in range(1, max_eta + 1)
        }

    # ---- Initialize each agent at Sterman equilibrium -------------
    agents = {}
    for role in ROLES:
        agents[role] = {
            "inventory": initial_inventory,
            "backlog": 0,
            "order_pipeline": [],
            "shipment_pipeline": [],
            "last_order": 0,
            "orders_placed": [],
            "expected_loss": float(initial_throughput),
            "orders_history": [initial_throughput] * 4,
            "total_cost": 0.0,
            "outstanding_to_supplier": 0,
        }

    # Seed downstream order-delay pipelines (W/D/F see order slips)
    for role in ["wholesaler", "distributor", "factory"]:
        agents[role]["order_pipeline"] = make_seeded_delay_pipeline(order_delay, initial_throughput)

    # Seed inbound physical shipment pipelines for R/W/D
    for role in ["retailer", "wholesaler", "distributor"]:
        agents[role]["shipment_pipeline"] = make_seeded_delay_pipeline(shipment_lead_time, initial_throughput)
        # Outstanding = orders slips + in-transit shipments
        agents[role]["outstanding_to_supplier"] = (order_delay + shipment_lead_time) * initial_throughput

    # Factory: production-in-progress
    agents["factory"]["shipment_pipeline"] = make_seeded_delay_pipeline(production_lead_time, initial_throughput)
    agents["factory"]["outstanding_to_supplier"] = production_lead_time * initial_throughput

    # ---- Agent decision rule --------------------------------------
    def calc_expected(agent, theta):
        """Adaptive expectation: exponential smoothing on incoming orders."""
        last_actual = agent["orders_history"][-1] if agent["orders_history"] else float(initial_throughput)
        return max(0.0, theta * last_actual + (1.0 - theta) * agent["expected_loss"])

    def append_observed_incoming(agent, incoming):
        """Update incoming-order history (used in both warmup and ABM rounds)."""
        agent["orders_history"].append(to_nonnegative_int(incoming))
        if len(agent["orders_history"]) > 8:
            agent["orders_history"] = agent["orders_history"][-8:]

    def receive(agent, qty):
        """Physically receive units: decrements outstanding, increments inventory."""
        qty = to_nonnegative_int(qty)
        if qty <= 0: return
        agent["outstanding_to_supplier"] = max(0, agent["outstanding_to_supplier"] - qty)
        agent["inventory"] += qty

    def fill(agent, demand_qty):
        """Mass-conserving fulfillment. Returns actual shipped units."""
        demand_qty = to_nonnegative_int(demand_qty)
        agent["backlog"] += demand_qty
        shipped = min(agent["inventory"], agent["backlog"])
        agent["inventory"] -= shipped
        agent["backlog"]   -= shipped
        return shipped

    def cost_breakdown(agent):
        """Holding cost 0.5/unit; backlog cost 1.0/unit."""
        inv_cost = 0.5 * agent["inventory"]
        bl_cost  = 1.0 * agent["backlog"]
        total    = inv_cost + bl_cost
        agent["total_cost"] += total
        return total, inv_cost, bl_cost

    def calc_order(agent, incoming, p):
        """ABM ordering rule used after the forced-warmup rounds."""
        incoming = to_nonnegative_int(incoming)
        agent["expected_loss"] = calc_expected(agent, p["theta"])

        effective_inv = agent["inventory"] - agent["backlog"]
        SL_t          = agent["outstanding_to_supplier"]
        intended      = agent["expected_loss"] + p["alpha_s"] * (
            p["s_prime"] - effective_inv - p["beta"] * SL_t
        )
        if sigma_eps > 0:
            intended += rng.normal(0.0, sigma_eps)

        order = max(0, int(round(intended)))
        append_observed_incoming(agent, incoming)
        agent["orders_placed"].append(order)
        return order

    # ---- Main simulation loop -------------------------------------
    weekly_data = []
    for week in range(weeks):
        round_number = week + 1
        demand = customer_demand[week]

        # 1) Advance all pipelines
        for role in ROLES:
            advance_pipeline(agents[role]["order_pipeline"])
            advance_pipeline(agents[role]["shipment_pipeline"])

        # 2) Collect arriving downstream orders
        order_received = {
            "retailer":    demand,
            "wholesaler":  pop_arrivals(agents["wholesaler"]["order_pipeline"]),
            "distributor": pop_arrivals(agents["distributor"]["order_pipeline"]),
            "factory":     pop_arrivals(agents["factory"]["order_pipeline"]),
        }
        _ = pop_arrivals(agents["retailer"]["order_pipeline"])  # retailer has no meaningful pipeline

        # 3) Collect arriving physical shipments / production
        shipment_received = {
            r: pop_arrivals(agents[r]["shipment_pipeline"]) for r in ROLES
        }

        # 4) Physically receive
        for role in ROLES:
            receive(agents[role], shipment_received[role])

        # 5) Fulfill downstream demand
        shipped = {r: fill(agents[r], order_received[r]) for r in ROLES}

        # State visible to the decision rule BEFORE the new order is placed
        SL_before_order              = {r: agents[r]["outstanding_to_supplier"] for r in ROLES}
        visible_pipeline_before_order = {r: pipeline_sum(agents[r]["shipment_pipeline"]) for r in ROLES}
        visible_eta_before_order      = {r: eta_breakdown(agents[r]["shipment_pipeline"], 4) for r in ROLES}

        # 6) Place orders (forced 4 during warmup; ABM rule afterwards)
        orders, decision_source = {}, {}
        for role in ROLES:
            if week < forced_warmup_rounds:
                order = initial_throughput
                append_observed_incoming(agents[role], order_received[role])
                agents[role]["orders_placed"].append(order)
                source = "forced_warmup"
            else:
                order = calc_order(agents[role], order_received[role], params[role])
                source = "abm"
            orders[role] = to_nonnegative_int(order)
            decision_source[role] = source
            agents[role]["last_order"] = orders[role]
            # Outstanding bumps up immediately after placing the order
            agents[role]["outstanding_to_supplier"] += orders[role]

        # 7) Inject new order slips into upstream order pipelines
        add_to_pipeline(agents["wholesaler"]["order_pipeline"],  orders["retailer"],    order_delay)
        add_to_pipeline(agents["distributor"]["order_pipeline"], orders["wholesaler"],  order_delay)
        add_to_pipeline(agents["factory"]["order_pipeline"],     orders["distributor"], order_delay)

        # 8) Schedule physical shipments downstream (using ACTUAL shipped amounts)
        add_to_pipeline(agents["retailer"]["shipment_pipeline"],    shipped["wholesaler"],  shipment_lead_time)
        add_to_pipeline(agents["wholesaler"]["shipment_pipeline"],  shipped["distributor"], shipment_lead_time)
        add_to_pipeline(agents["distributor"]["shipment_pipeline"], shipped["factory"],     shipment_lead_time)

        # 9) Factory production
        add_to_pipeline(agents["factory"]["shipment_pipeline"], orders["factory"], production_lead_time)

        # 10) Costs
        costs = {r: cost_breakdown(agents[r]) for r in ROLES}

        # 11) Record this week's state
        row = {
            "week": week, "round": round_number, "customer_demand": demand,
            **{f"{r}_order":            orders[r]              for r in ROLES},
            **{f"{r}_decision_source":  decision_source[r]     for r in ROLES},
            **{f"{r}_order_received":   order_received[r]      for r in ROLES},
            "retailer_shipment_received":    shipment_received["retailer"],
            "wholesaler_shipment_received":  shipment_received["wholesaler"],
            "distributor_shipment_received": shipment_received["distributor"],
            "factory_production_received":   shipment_received["factory"],
            "retailer_shipped_to_customer":     shipped["retailer"],
            "wholesaler_shipped_to_retailer":   shipped["wholesaler"],
            "distributor_shipped_to_wholesaler":shipped["distributor"],
            "factory_shipped_to_distributor":   shipped["factory"],
            **{f"{r}_shipped": shipped[r] for r in ROLES},
        }
        for role in ROLES:
            total_cost, inv_cost, bl_cost = costs[role]
            row[f"{role}_inventory"]            = agents[role]["inventory"]
            row[f"{role}_backlog"]              = agents[role]["backlog"]
            row[f"{role}_effective_inventory"]  = agents[role]["inventory"] - agents[role]["backlog"]
            row[f"{role}_cost"]                 = total_cost
            row[f"{role}_inventory_cost"]       = inv_cost
            row[f"{role}_backlog_cost"]         = bl_cost
            row[f"{role}_SL_before_order"]            = SL_before_order[role]
            row[f"{role}_visible_pipeline_before_order"] = visible_pipeline_before_order[role]
            for eta_label, qty in visible_eta_before_order[role].items():
                row[f"{role}_visible_pipeline_{eta_label}_before_order"] = qty
            row[f"{role}_SL_inbound"]        = agents[role]["outstanding_to_supplier"]
            row[f"{role}_order_pipeline"]    = pipeline_sum(agents[role]["order_pipeline"])
            row[f"{role}_shipment_pipeline"] = pipeline_sum(agents[role]["shipment_pipeline"])
            row[f"{role}_supply_line"]       = pipeline_sum(agents[role]["shipment_pipeline"])
            row[f"{role}_expected_loss"]     = agents[role]["expected_loss"]
        row["factory_production_pipeline"] = pipeline_sum(agents["factory"]["shipment_pipeline"])
        weekly_data.append(row)

    df = pd.DataFrame(weekly_data)

    # ---- Warm-up sanity check -------------------------------------
    warmup_sanity_passed = True
    if run_sanity_check and forced_warmup_rounds > 0:
        warmup_df = df[df["week"] < forced_warmup_rounds]
        for role in ROLES:
            if not (warmup_df[f"{role}_order"]      == initial_throughput).all(): warmup_sanity_passed = False
            if not (warmup_df[f"{role}_inventory"]  == initial_inventory).all():  warmup_sanity_passed = False
            if not (warmup_df[f"{role}_backlog"]    == 0).all():                  warmup_sanity_passed = False
            if not (warmup_df[f"{role}_decision_source"] == "forced_warmup").all(): warmup_sanity_passed = False
        if not warmup_sanity_passed:
            raise AssertionError(
                "Warm-up equilibrium failed: expected forced orders=4, inventory=12, backlog=0."
            )

    # ---- Aggregates -----------------------------------------------
    total_costs     = {r: float(agents[r]["total_cost"]) for r in ROLES}
    total_team_cost = float(sum(total_costs.values()))

    if weeks > 20:
        steady = df[df["week"] >= 10]
        cust_var = float(steady["customer_demand"].var())
        fact_var = float(steady["factory_order"].var())
        amplification = (fact_var / cust_var) if cust_var > 0 else fact_var
    else:
        amplification = 0.0

    full_cust_var    = float(df["customer_demand"].var())
    full_factory_var = float(df["factory_order"].var())
    full_horizon_amplification = (
        full_factory_var / full_cust_var if full_cust_var > 0 else full_factory_var
    )

    orders_placed = {r: list(agents[r]["orders_placed"]) for r in ROLES}

    summary = {
        "cost_vs_optimal":   total_team_cost / 204.0,
        "cost_vs_human_avg": total_team_cost / 2028.0,
        "max_retailer_order":    float(df["retailer_order"].max()),
        "max_wholesaler_order":  float(df["wholesaler_order"].max()),
        "max_distributor_order": float(df["distributor_order"].max()),
        "max_factory_order":     float(df["factory_order"].max()),
        "min_retailer_inventory": float(df["retailer_inventory"].min()),
        "max_retailer_backlog":   float(df["retailer_backlog"].max()),
        "total_backlog_weeks": int((df[[f"{r}_backlog" for r in ROLES]].sum(axis=1) > 0).sum()),
        "warmup_sanity_passed": warmup_sanity_passed,
    }

    return {
        "data": df,
        "total_costs": total_costs,
        "total_team_cost": total_team_cost,
        "amplification": amplification,
        "full_horizon_amplification": full_horizon_amplification,
        "parameters": {
            "role_parameters": params, "sigma_eps": sigma_eps, "seed": seed,
            "weeks": weeks, "initial_inventory": initial_inventory,
            "initial_throughput": initial_throughput,
            "initial_order": initial_throughput,  # backward-compat
            "order_delay": order_delay,
            "shipment_lead_time": shipment_lead_time,
            "production_lead_time": production_lead_time,
            "forced_warmup_rounds": forced_warmup_rounds,
            "initial_pipelines_seeded": True,
            "mass_conserving_shipments": True,
            "separate_backlog": True,
            "nonnegative_inventory": True,
        },
        "orders_placed": orders_placed,
        "summary": summary,
    }


In [ ]:
def plot_simulation_results(result):
    """Plot orders, effective inventory, and weekly costs with per-role params in the title."""
    data  = result["data"]
    weeks = data["week"]
    p     = result["parameters"]["role_parameters"]

    colors = {
        "customer":    "black",
        "retailer":    "#1f77b4",
        "wholesaler":  "#ff7f0e",
        "distributor": "#2ca02c",
        "factory":     "#d62728",
    }

    def _fmt(pp):  # compact role-param formatter
        return f"θ={pp['theta']:.2f}, αS={pp['alpha_s']:.2f}, β={pp['beta']:.2f}, S′={int(pp['s_prime'])}"

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

    # --- Orders ---
    ax1.plot(weeks, data["customer_demand"],   color=colors["customer"],    lw=3, ls="--", alpha=0.9, label="Customer Demand")
    for r in ROLES:
        ax1.plot(weeks, data[f"{r}_order"], color=colors[r], lw=2, alpha=0.85, label=r.capitalize())
    ax1.set_title("Orders by Supply Chain Level", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Week"); ax1.set_ylabel("Orders (Cases/Week)")
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, max(data["factory_order"].max() * 1.1, 20))

    # --- Effective Inventory ---
    for r in ROLES:
        ax2.plot(weeks, data[f"{r}_inventory"] - data[f"{r}_backlog"],
                 color=colors[r], lw=2, alpha=0.85, label=r.capitalize())
    ax2.axhline(0,  color="red",  ls=":", alpha=0.7, label="Stockout Line")
    ax2.axhline(12, color="gray", ls=":", alpha=0.5, label="Initial Level")
    ax2.set_title("Effective Inventory (Inventory − Backlog)", fontsize=14, fontweight="bold")
    ax2.set_xlabel("Week"); ax2.set_ylabel("Cases")
    ax2.legend(); ax2.grid(True, alpha=0.3)

    # --- Weekly Costs ---
    for r in ROLES:
        ax3.plot(weeks, data[f"{r}_cost"], color=colors[r], lw=2, alpha=0.85, label=r.capitalize())
    ax3.set_title("Weekly Costs", fontsize=14, fontweight="bold")
    ax3.set_xlabel("Week"); ax3.set_ylabel("$/Week")
    ax3.legend(); ax3.grid(True, alpha=0.3)

    # Title with summary stats and role parameters
    role_str = " | ".join([
        f"R({_fmt(p['retailer'])})", f"W({_fmt(p['wholesaler'])})",
        f"D({_fmt(p['distributor'])})", f"F({_fmt(p['factory'])})",
    ])
    fig.suptitle(
        f"Beer Game • Total Cost ${result['total_team_cost']:.0f} "
        f"({result['summary']['cost_vs_optimal']:.1f}× optimal) • "
        f"Amplification {result['amplification']:.1f}×\n{role_str}",
        fontsize=12, fontweight="bold",
    )
    plt.tight_layout()
    plt.show()


## 2. Data Loading

Loader for the pre-extracted JSON files produced from raw LLM runs.
Each file is named `sim_{i}_extracted.json` and contains order/inventory/backlog
arrays for all four roles.


In [ ]:
REQUIRED_KEYS: List[str] = [
    "retailer_order",     "wholesaler_order",     "distributor_order",     "factory_order",
    "retailer_inventory", "wholesaler_inventory", "distributor_inventory", "factory_inventory",
    "retailer_backlog",   "wholesaler_backlog",   "distributor_backlog",   "factory_backlog",
]


def load_simulated_data(extracted_dir: str, sim_name: Optional[str] = None) -> Dict[str, Any]:
    """Load a single extracted simulation from `extracted_dir`. If `sim_name` is None, uses first match."""
    extracted_dir_abs = os.path.abspath(extracted_dir)

    if sim_name:
        file_path = os.path.join(extracted_dir_abs, f"{sim_name}_extracted.json")
        if not os.path.isfile(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")
    else:
        matches = sorted(glob(os.path.join(extracted_dir_abs, "*_extracted.json")))
        if not matches:
            raise FileNotFoundError(f"No '*_extracted.json' files in: {extracted_dir_abs}")
        file_path = matches[0]

    with open(file_path, "r") as f:
        payload: Dict[str, Any] = json.load(f)

    missing = [k for k in REQUIRED_KEYS if k not in payload]
    if missing:
        raise ValueError(f"{os.path.basename(file_path)} missing keys: {', '.join(missing)}")

    # Coerce list fields to numpy arrays
    simulated = {}
    for key in REQUIRED_KEYS:
        v = payload[key]
        if isinstance(v, list):
            try:    simulated[key] = np.array(v, dtype=int)
            except Exception: simulated[key] = np.array(v, dtype=object)
        else:
            simulated[key] = v
    return simulated


## 3. Sterman's 11 Human Experiments

The 11 calibrated (θ, αS, β, S′) parameter sets per role, from Sterman's
empirical fits. We build a 16×11 table (16 params × 11 sims) used as
"human" simulator input throughout the rest of the notebook.


In [ ]:
# Sterman's 11 fitted experiments (θ, αS, β, S′ per role).
# R² and RMSE columns from the original table are intentionally omitted.
_STERMAN_11 = [
    # 0 Bassbeer
    {"Retailer":   {"theta": 0.90, "alpha_s": 0.10, "beta": 0.65, "s_prime": 20},
     "Wholesaler": {"theta": 0.00, "alpha_s": 0.25, "beta": 0.50, "s_prime": 27},
     "Distributor":{"theta": 0.15, "alpha_s": 0.05, "beta": 0.35, "s_prime": 14},
     "Factory":    {"theta": 1.00, "alpha_s": 0.65, "beta": 0.40, "s_prime": 15}},
    # 1 Budweiser
    {"Retailer":   {"theta": 0.00, "alpha_s": 0.40, "beta": 0.10, "s_prime": 7},
     "Wholesaler": {"theta": 0.00, "alpha_s": 0.40, "beta": 0.75, "s_prime": 30},
     "Distributor":{"theta": 0.00, "alpha_s": 0.30, "beta": 0.10, "s_prime": 10},
     "Factory":    {"theta": 0.25, "alpha_s": 0.25, "beta": 0.10, "s_prime": 9}},
    # 2 Coors
    {"Retailer":   {"theta": 0.00, "alpha_s": 0.20, "beta": 0.00, "s_prime": 25},
     "Wholesaler": {"theta": 0.00, "alpha_s": 0.15, "beta": 0.50, "s_prime": 38},
     "Distributor":{"theta": 0.90, "alpha_s": 0.30, "beta": 0.20, "s_prime": 10},
     "Factory":    {"theta": 0.25, "alpha_s": 0.30, "beta": 0.00, "s_prime": 18}},
    # 3 Freebeer
    {"Retailer":   {"theta": 0.40, "alpha_s": 0.35, "beta": 0.45, "s_prime": 15},
     "Wholesaler": {"theta": 0.30, "alpha_s": 0.05, "beta": 0.00, "s_prime": 30},
     "Distributor":{"theta": 0.05, "alpha_s": 0.35, "beta": 1.00, "s_prime": 18},
     "Factory":    {"theta": 0.25, "alpha_s": 0.25, "beta": 0.00, "s_prime": 19}},
    # 4 Grin & Beer It
    {"Retailer":   {"theta": 0.10, "alpha_s": 0.35, "beta": 0.65, "s_prime": 13},
     "Wholesaler": {"theta": 0.95, "alpha_s": 0.15, "beta": 0.55, "s_prime": 14},
     "Distributor":{"theta": 0.20, "alpha_s": 0.20, "beta": 0.30, "s_prime": 19},
     "Factory":    {"theta": 0.25, "alpha_s": 0.35, "beta": 0.55, "s_prime": 24}},
    # 5 Grizzly
    {"Retailer":   {"theta": 0.05, "alpha_s": 0.30, "beta": 0.65, "s_prime": 31},
     "Wholesaler": {"theta": 0.30, "alpha_s": 0.20, "beta": 0.35, "s_prime": 27},
     "Distributor":{"theta": 0.15, "alpha_s": 0.05, "beta": 0.25, "s_prime": 15},
     "Factory":    {"theta": 0.55, "alpha_s": 0.65, "beta": 0.00, "s_prime": 9}},
    # 6 Heineken1
    {"Retailer":   {"theta": 0.95, "alpha_s": 0.15, "beta": 0.00, "s_prime": 9},
     "Wholesaler": {"theta": 0.50, "alpha_s": 0.00, "beta": 0.34, "s_prime": 17},
     "Distributor":{"theta": 0.20, "alpha_s": 0.30, "beta": 0.05, "s_prime": 8},
     "Factory":    {"theta": 0.80, "alpha_s": 0.00, "beta": 0.34, "s_prime": 17}},
    # 7 Heineken2
    {"Retailer":   {"theta": 0.50, "alpha_s": 0.05, "beta": 0.60, "s_prime": 6},
     "Wholesaler": {"theta": 0.40, "alpha_s": 0.10, "beta": 0.30, "s_prime": 16},
     "Distributor":{"theta": 1.00, "alpha_s": 0.15, "beta": 0.60, "s_prime": 14},
     "Factory":    {"theta": 0.55, "alpha_s": 0.80, "beta": 0.00, "s_prime": 9}},
    # 8 Heineken3
    {"Retailer":   {"theta": 0.05, "alpha_s": 0.30, "beta": 0.45, "s_prime": 5},
     "Wholesaler": {"theta": 0.20, "alpha_s": 0.00, "beta": 0.34, "s_prime": 17},
     "Distributor":{"theta": 0.30, "alpha_s": 0.10, "beta": 0.30, "s_prime": 12},
     "Factory":    {"theta": 0.00, "alpha_s": 0.30, "beta": 0.15, "s_prime": 17}},
    # 9 Suds
    {"Retailer":   {"theta": 1.00, "alpha_s": 0.00, "beta": 0.34, "s_prime": 17},
     "Wholesaler": {"theta": 0.05, "alpha_s": 0.30, "beta": 0.20, "s_prime": 20},
     "Distributor":{"theta": 0.15, "alpha_s": 0.60, "beta": 0.35, "s_prime": 0},
     "Factory":    {"theta": 0.40, "alpha_s": 0.35, "beta": 1.05, "s_prime": 32}},
    # 10 Twoborg
    {"Retailer":   {"theta": 0.75, "alpha_s": 0.35, "beta": 0.00, "s_prime": 4},
     "Wholesaler": {"theta": 0.00, "alpha_s": 0.25, "beta": 0.05, "s_prime": 18},
     "Distributor":{"theta": 0.05, "alpha_s": 0.50, "beta": 0.00, "s_prime": 15},
     "Factory":    {"theta": 0.95, "alpha_s": 0.30, "beta": 0.20, "s_prime": 26}},
]

_ROLE_SUFFIX = {"Retailer": "r", "Distributor": "d", "Wholesaler": "w", "Factory": "f"}


def build_human_experiments_df() -> pd.DataFrame:
    """Build a 16 × 11 params-by-sims DataFrame for Sterman's experiments."""
    rows = [
        "theta_r","alpha_s_r","beta_r","s_prime_r",
        "theta_d","alpha_s_d","beta_d","s_prime_d",
        "theta_w","alpha_s_w","beta_w","s_prime_w",
        "theta_f","alpha_s_f","beta_f","s_prime_f",
    ]
    cols = [f"sim_{i}" for i in range(11)]
    df = pd.DataFrame(index=rows, columns=cols, dtype=float)
    for i, exp in enumerate(_STERMAN_11):
        for role, ps in exp.items():
            sfx = _ROLE_SUFFIX[role]
            df.loc[f"theta_{sfx}",   f"sim_{i}"] = ps["theta"]
            df.loc[f"alpha_s_{sfx}", f"sim_{i}"] = ps["alpha_s"]
            df.loc[f"beta_{sfx}",    f"sim_{i}"] = ps["beta"]
            df.loc[f"s_prime_{sfx}", f"sim_{i}"] = ps["s_prime"]
    return df


human_df_sims     = build_human_experiments_df()
human_mean_series = human_df_sims.mean(axis=1)  # sanity check


In [ ]:
def append_humans_to_outputs(all_samples_by_dir, overall_means_by_dir):
    """Append the human parameter samples to an existing `compute_means_for_dirs` output."""
    human_stacked = human_df_sims.copy()
    human_stacked["directory"] = "human"
    human_stacked["parameter"] = human_stacked.index
    human_stacked = human_stacked.set_index(["directory", "parameter"])
    all_samples_by_dir_with_human = pd.concat([all_samples_by_dir, human_stacked], axis=0)

    overall_means_with_human = overall_means_by_dir.copy()
    overall_means_with_human["human"] = human_mean_series
    return all_samples_by_dir_with_human, overall_means_with_human


## 4. Parameter → simulator kwargs helpers

Small helpers used everywhere downstream to turn a column of `human_df_sims`
into simulator kwargs (and to build a leave-one-human-out – "LOHO" – series).


In [ ]:
_PARAM_KEYS = [
    "theta_r","alpha_s_r","beta_r","s_prime_r",
    "theta_w","alpha_s_w","beta_w","s_prime_w",
    "theta_d","alpha_s_d","beta_d","s_prime_d",
    "theta_f","alpha_s_f","beta_f","s_prime_f",
]


def _kwargs_from_param_series(series: pd.Series, *, weeks: int,
                              initial_inventory: int = 12,
                              initial_order: int = 4) -> Dict[str, Any]:
    """Convert a Series indexed by _PARAM_KEYS into simulator kwargs."""
    missing = [k for k in _PARAM_KEYS if k not in series.index]
    if missing:
        raise KeyError(f"Missing parameters: {missing}")
    out = {k: float(series[k]) for k in _PARAM_KEYS}
    out.update(weeks=weeks, initial_inventory=initial_inventory, initial_order=initial_order)
    return out


def _human_kwargs_for_sim(human_param_df: pd.DataFrame, sim_idx: int, *, weeks: int,
                          initial_inventory: int = 12,
                          initial_order: int = 4) -> Dict[str, Any]:
    """Kwargs for human experiment `sim_idx` (column `sim_{idx}`)."""
    return _kwargs_from_param_series(
        human_param_df[f"sim_{sim_idx}"], weeks=weeks,
        initial_inventory=initial_inventory, initial_order=initial_order,
    )


def _loho_series(human_param_df: pd.DataFrame, leave_out_sim: int) -> pd.Series:
    """Leave-One-Human-Out: mean parameters across all sims except `leave_out_sim`."""
    cols = [c for c in human_param_df.columns if c != f"sim_{leave_out_sim}"]
    if not cols:
        raise ValueError("LOHO requires at least 2 human sims.")
    return human_param_df[cols].mean(axis=1)


def _human_baseline_series(human_param_df: pd.DataFrame, sim_idx: int, mode: str) -> pd.Series:
    """`avg` = mean across all sims; `loho` = mean across all sims except `sim_idx`."""
    assert mode in {"avg", "loho"}
    if mode == "avg":
        return human_param_df.mean(axis=1)
    return _loho_series(human_param_df, sim_idx)


## 5. Orders vs Human — LOHO band plot

For each role, plot LLM order trajectories alongside the Leave-One-Human-Out
(LOHO) human benchmark with a shaded confidence band.


In [ ]:
@nature_style
def plot_model_orders_vs_human_with_shading(
    *,
    model_dirs: Dict[str, str],
    human_param_df: pd.DataFrame,
    simulate_fn,
    series: List[Dict[str, Any]],
    sim_indices: Iterable[int] = range(11),
    distribution: str = "gaussian",
    gaussian_sigma: Union[None, float, Dict[str, float]] = None,
    nb_dispersion: Union[None, float, Dict[str, float]] = None,
    conf_level: float = 0.95,
    weeks_fallback: int = 35,
    show_loho_band: bool = True,
    loho_color: str = "#7570b3",
    savepath: Optional[str] = None,
):
    """
    Plot per-sim mean order trajectory for one or more LLM model directories,
    with a LOHO benchmark and a Gaussian/Poisson/NegBin confidence band.

    `series` is a list of dicts. Required keys:
        - "label":     legend name
        - "model_key": key into `model_dirs`
    Optional keys (auto-filled if missing):
        - "color":     matplotlib color (default: Dark2/Set2 palette)
        - "linestyle": '-', '--', etc. (default '-')
    """
    distribution = distribution.lower()
    assert distribution in {"gaussian", "poisson", "negbin"}

    default_palette = list(plt.colormaps["Dark2"].colors) + list(plt.colormaps["Set2"].colors)
    norm_series: List[Dict[str, Any]] = []
    for i, s in enumerate(series):
        if "label" not in s or "model_key" not in s:
            raise KeyError(f"series[{i}] must define 'label' and 'model_key'.")
        if s["model_key"] not in model_dirs:
            raise KeyError(f"series[{i}]['model_key']={s['model_key']!r} not in model_dirs.")
        norm_series.append({
            "label":     s["label"],
            "model_key": s["model_key"],
            "color":     s.get("color",     default_palette[i % len(default_palette)]),
            "linestyle": s.get("linestyle", "-"),
        })

    def _load(model_key: str, role: str, sim_i: int) -> np.ndarray:
        payload = load_simulated_data(model_dirs[model_key], sim_name=f"sim_{sim_i}")
        return np.asarray(payload[ROLE_KEYS[role]], dtype=float)

    series_sims = {s["model_key"]: {r: [] for r in ROLES} for s in norm_series}
    human_sims  = {r: [] for r in ROLES}
    loho_sims   = {r: [] for r in ROLES}

    for i in sim_indices:
        weeks_i = None
        for s in norm_series:
            try:
                weeks_i = len(_load(s["model_key"], "retailer", i))
                break
            except Exception:
                pass
        weeks_i = int(weeks_i if weeks_i is not None else weeks_fallback)

        hdf = simulate_fn(**_human_kwargs_for_sim(human_param_df, i, weeks=weeks_i))["data"]
        ldf = simulate_fn(**_kwargs_from_param_series(_loho_series(human_param_df, i),
                                                       weeks=weeks_i))["data"]

        for r in ROLES:
            human_sims[r].append(hdf[f"{r}_order"].to_numpy(float))
            loho_sims[r].append(ldf[f"{r}_order"].to_numpy(float))
            for s in norm_series:
                try:
                    series_sims[s["model_key"]][r].append(_load(s["model_key"], r, i))
                except Exception:
                    pass

    def _pad_to(arrs, L):
        out = []
        for a in arrs:
            a = np.asarray(a, float)
            if len(a) < L:
                a = np.concatenate([a, np.full(L - len(a), np.nan)])
            out.append(a)
        return np.vstack(out) if out else np.empty((0, L))

    role_len: Dict[str, int] = {}
    for r in ROLES:
        L = 0
        for bunch in [human_sims[r], loho_sims[r]] + [series_sims[s["model_key"]][r] for s in norm_series]:
            if bunch:
                L = max(L, max(len(x) for x in bunch))
        role_len[r] = L if L > 0 else weeks_fallback

    def _mean_across_sims(sim_list, r):
        M = _pad_to(sim_list, role_len[r])
        return (np.nanmean(M, axis=0) if M.size else np.full(role_len[r], np.nan), M)

    means: Dict[Tuple[str, str], np.ndarray] = {}
    mats:  Dict[Tuple[str, str], np.ndarray] = {}
    for r in ROLES:
        means[(r, "human")], mats[(r, "human")] = _mean_across_sims(human_sims[r], r)
        means[(r, "loho")],  mats[(r, "loho")]  = _mean_across_sims(loho_sims[r], r)
        for s in norm_series:
            means[(r, s["model_key"])], _ = _mean_across_sims(series_sims[s["model_key"]][r], r)

    alpha = 1.0 - conf_level
    bands = {}
    for r in ROLES:
        mu = means[(r, "loho")]
        if not show_loho_band:
            bands[r] = (np.full_like(mu, np.nan), np.full_like(mu, np.nan))
        elif distribution == "gaussian":
            if isinstance(gaussian_sigma, dict):
                sigma = float(gaussian_sigma.get(r, np.nan))
            elif isinstance(gaussian_sigma, (int, float)):
                sigma = float(gaussian_sigma)
            else:
                resid = (mats[(r, "human")] - mats[(r, "loho")])
                resid = resid[np.isfinite(resid)]
                sigma = float(np.sqrt(np.nanmean(resid ** 2))) if resid.size else np.nan
            if np.isfinite(sigma):
                resid_n = int(np.isfinite(mats[(r, "human")] - mats[(r, "loho")]).sum()) - 1
                try:    tcrit = float(student_t.ppf(1 - alpha / 2, max(resid_n, 1)))
                except: tcrit = 1.9599639845
                half = tcrit * sigma
                bands[r] = (mu - half, mu + half)
            else:
                bands[r] = (np.full_like(mu, np.nan), np.full_like(mu, np.nan))
        elif distribution == "poisson":
            mu_clip = np.clip(mu, 0, None)
            tcrit   = float(student_t.ppf(1 - alpha / 2, 10**6))
            half    = tcrit * np.sqrt(mu_clip)
            bands[r] = (mu_clip - half, mu_clip + half)
        else:
            if nb_dispersion is None:
                raise ValueError("For negbin, provide nb_dispersion.")
            k = nb_dispersion.get(r, np.nan) if isinstance(nb_dispersion, dict) else float(nb_dispersion)
            if not np.isfinite(k) or k <= 0:
                raise ValueError(f"Invalid NB dispersion k for role '{r}'.")
            mu_clip = np.clip(mu, 0, None)
            var     = mu_clip + (mu_clip ** 2) / k
            tcrit   = float(student_t.ppf(1 - alpha / 2, 10**6))
            half    = tcrit * np.sqrt(var)
            bands[r] = (mu_clip - half, mu_clip + half)

    fig = trajectory_panels(
        [dict(label=s["label"], values={r: means[(r, s["model_key"])] for r in ROLES},
              color=s["color"], ls=s["linestyle"]) for s in norm_series],
        rounds=role_len[ROLES[0]],
        band=bands,
        benchmark={r: means[(r, "loho")] for r in ROLES},
        benchmark_color=loho_color,
    )
    save_figure(fig, savepath)
    plt.show()


# ── Call ─────────────────────────────────────────────────────────
# All available baseline and heuristic-aware model folders are resolved
# automatically. The displayed figure uses a readable four-line subset.
model_dirs = resolve_model_dirs(
    [
        "llama2-baseline", "llama2-prompt",
        "llama3-baseline", "llama3-prompt",
        "llama70b-baseline", "llama70b-prompt",
        "kimi-baseline", "kimi-prompt",
        "qwen3-baseline", "qwen3-prompt",
        "mistral-baseline", "mistral-prompt",
        "oss20b-baseline", "oss20b-prompt",
        "oss120b-baseline", "oss120b-prompt",
    ],
    label="trajectory-plot",
)

# Baseline dashed, heuristic-aware solid.
candidate_order_series = [
    {
        "label": "Kimi K2 (baseline)",
        "model_key": "kimi-baseline",
        "color": "#1d9275",
        "linestyle": "--",
    },
    {
        "label": "Kimi K2 (heuristic-aware)",
        "model_key": "kimi-prompt",
        "color": "#cc545e",
    },
    {
        "label": "Qwen-3.6-35B-A3B (baseline)",
        "model_key": "qwen3-baseline",
        "color": "#66a61e",
        "linestyle": "--",
    },
    {
        "label": "Qwen-3.6-35B-A3B (heuristic-aware)",
        "model_key": "qwen3-prompt",
        "color": "#a87ce9",
    },
]

orders_series = [
    entry
    for entry in candidate_order_series
    if entry["model_key"] in model_dirs
]

if len(orders_series) < 2:
    raise RuntimeError(
        "At least two of the requested trajectory-series folders are needed "
        "to build orders.pdf."
    )

plot_model_orders_vs_human_with_shading(
    model_dirs=model_dirs,
    human_param_df=human_df_sims,
    simulate_fn=simulate_beer_game,
    series=orders_series,
    distribution="gaussian",
    gaussian_sigma=None,
    conf_level=0.95,
    savepath=f"{PLOTS_DIR}/orders.pdf",
)

## 6. Bullwhip Effect

Two complementary views:
- **Per-role**: 4 subplots, each model's variance ratio (Var(orders)/Var(demand)).
- **Aggregate**: 1 bar per model, averaged across roles.


In [ ]:
# ============================================================================
# OPTION A — geometric mean with a log-space t interval
#
# Drop-in replacement for plot_bullwhip_effect_aggregate.
# Same name, same signature (plus one new kwarg: ci_mode), so your existing
# call site keeps working unchanged.
#
# Why this fixes the Kimi-raw distributor/factory whiskers:
#   A variance ratio is a multiplicative quantity. mean +/- t*SE on the raw
#   ratios can produce a lower bound at or below zero whenever the CV of the
#   per-sim ratios approaches 1, which is exactly what happens upstream where
#   a couple of sims carry ratios in the hundreds. Computing the interval on
#   log(ratio) and back-transforming guarantees a strictly positive lower
#   bound, so nothing needs clipping and no whisker can span the whole panel.
#
# NOTE: this reports the GEOMETRIC mean, not the arithmetic mean. Bar heights
# will drop for the heavy-tailed upstream roles. Say so in your caption.
# ============================================================================

def plot_bullwhip_effect_aggregate(
    *,
    model_dirs: Dict[str, str],
    human_param_df: pd.DataFrame,
    simulate_fn,
    baseline_model: str,
    prompt_model: str,
    second_baseline: Optional[str] = None,
    second_prompt: Optional[str] = None,
    primary_label: str = "Kimi K2",
    second_label: str = "Qwen-3.6-35B-A3B",
    sim_indices: Iterable[int] = range(11),
    demand_keys: Tuple[str, ...] = ("retailer_demand", "customer_demand", "demand", "retailer_order"),
    weeks_fallback: int = 35,
    figsize: Tuple[int, int] = (12, 8),
    title: str = "Bullwhip (whole-game variance ratio)",
    ylim:  Optional[Tuple[float, float]] = None,
    ylims: Optional[Union[Dict[str, Tuple[float, float]],
                          Tuple[Tuple[float, float], ...],
                          List[Tuple[float, float]]]] = None,
    yscale: str = "linear",          # "linear" | "log"
    log_floor: float = 0.1,          # log-axis bottom (whiskers no longer clipped to it)
    ci_mode: str = "auto",           # "auto" | "geometric" | "arithmetic"
    annotate: bool = False,           # footnote describing the statistic
    savepath: Optional[str] = None,
):
    """4 subplots, one per role: bars for primary/second baseline+heuristic + LOHO.

    On a log axis this reports the geometric mean of the per-sim variance
    ratios, with a 95% t interval computed on log(ratio) and exponentiated
    back. The interval is therefore asymmetric on a linear ruler but symmetric
    in log space, always strictly positive, and never needs clipping.

    ci_mode:
      "auto"       -> geometric when yscale="log", arithmetic when "linear"
      "geometric"  -> always geometric/log-space
      "arithmetic" -> always mean +/- t*SE (original behaviour)
    """
    assert yscale in {"linear", "log"}
    assert ci_mode in {"auto", "geometric", "arithmetic"}
    sec_b_key = second_baseline if (second_baseline and second_baseline in model_dirs) else None
    sec_p_key = second_prompt   if (second_prompt   and second_prompt   in model_dirs) else None

    def _load(model_name: str, role: str, sim_i: int) -> np.ndarray:
        payload = load_simulated_data(model_dirs[model_name], sim_name=f"sim_{sim_i}")
        return np.asarray(payload[ROLE_KEYS[role]], dtype=float)

    def _var_ratio(y: np.ndarray, d: np.ndarray) -> float:
        y, d = np.asarray(y, float), np.asarray(d, float)
        L = min(len(y), len(d))
        if L < 2: return np.nan
        vy, vd = np.var(y[:L], ddof=0), np.var(d[:L], ddof=0)
        return float(vy / vd) if vd > 0 else np.nan

    def _mean_ci(x, alpha=0.05):
        """Arithmetic mean, returns (center, lo, hi)."""
        arr = np.asarray(x, float); arr = arr[np.isfinite(arr)]
        n = len(arr)
        if n == 0: return np.nan, np.nan, np.nan
        mu = float(np.mean(arr))
        sd = float(np.std(arr, ddof=1)) if n > 1 else 0.0
        if n > 1 and sd > 0:
            h = float(student_t.ppf(1 - alpha/2, n-1)) * sd / np.sqrt(n)
        else:
            h = 0.0
        return mu, mu - h, mu + h

    def _geo_mean_ci(x, alpha=0.05):
        """Geometric mean with a log-space t interval, returns (center, lo, hi).

        Non-positive and non-finite values are dropped: log() is undefined on
        them and a variance ratio of exactly 0 means a degenerate order series.
        """
        arr = np.asarray(x, float)
        arr = arr[np.isfinite(arr) & (arr > 0)]
        n = len(arr)
        if n == 0: return np.nan, np.nan, np.nan
        lg = np.log(arr)
        mu = float(np.mean(lg))
        sd = float(np.std(lg, ddof=1)) if n > 1 else 0.0
        if n > 1 and sd > 0:
            h = float(student_t.ppf(1 - alpha/2, n-1)) * sd / np.sqrt(n)
        else:
            h = 0.0
        return float(np.exp(mu)), float(np.exp(mu - h)), float(np.exp(mu + h))

    bw = {
        "primary_raw":  {r: [] for r in ROLES},
        "primary_heur": {r: [] for r in ROLES},
        "second_raw":   {r: [] for r in ROLES} if sec_b_key else None,
        "second_heur":  {r: [] for r in ROLES} if sec_p_key else None,
        "loho":         {r: [] for r in ROLES},
    }

    for i in sim_indices:
        weeks_i = None
        for probe in [baseline_model, prompt_model, sec_b_key, sec_p_key]:
            if probe is None: continue
            try: weeks_i = len(_load(probe, "retailer", i)); break
            except Exception: pass
        weeks_i = int(weeks_i if weeks_i is not None else weeks_fallback)

        hdf = simulate_fn(**_human_kwargs_for_sim(human_param_df, i, weeks=weeks_i))["data"]
        ldf = simulate_fn(**_kwargs_from_param_series(_loho_series(human_param_df, i),
                                                       weeks=weeks_i))["data"]

        demand = None
        for k in demand_keys:
            if k in hdf.columns: demand = hdf[k].to_numpy(float); break
        if demand is None:
            for k in demand_keys:
                if k in ldf.columns: demand = ldf[k].to_numpy(float); break
        if demand is None:
            demand = hdf["retailer_order"].to_numpy(float)

        for r in ROLES:
            bw["loho"][r].append(_var_ratio(ldf[f"{r}_order"].to_numpy(float), demand))
            for store_key, model_name in [("primary_raw", baseline_model), ("primary_heur", prompt_model)]:
                try: bw[store_key][r].append(_var_ratio(_load(model_name, r, i), demand))
                except Exception: bw[store_key][r].append(np.nan)
            if sec_b_key:
                try: bw["second_raw"][r].append(_var_ratio(_load(sec_b_key, r, i), demand))
                except Exception: bw["second_raw"][r].append(np.nan)
            if sec_p_key:
                try: bw["second_heur"][r].append(_var_ratio(_load(sec_p_key, r, i), demand))
                except Exception: bw["second_heur"][r].append(np.nan)

    set2 = plt.colormaps["Dark2"]
    series_order = [
        ("primary_raw",  f"{primary_label} (raw)",            set2.colors[0], 0.6),
        ("primary_heur", f"{primary_label} (heuristic-aware)", "#cc545e",     1.0),
    ]
    if sec_b_key: series_order.append(("second_raw",  f"{second_label} (raw)",            set2.colors[4], 0.6))
    if sec_p_key: series_order.append(("second_heur", f"{second_label} (heuristic-aware)", "#9671c3",     1.0))
    series_order.append(("loho", "LOHO", set2.colors[2], 1.0))

    def _get_role_ylim(role):
        if ylims is None: return None
        if isinstance(ylims, dict): return ylims.get(role, None)
        try:
            seq = list(ylims)
            if len(seq) == 4: return tuple(seq[ROLES.index(role)])
        except Exception: pass
        return None

    fig, axes = plt.subplots(2, 2, figsize=figsize); axes = axes.ravel()
    x0 = np.arange(len(series_order))
    is_log = (yscale == "log")
    use_geo = (ci_mode == "geometric") or (ci_mode == "auto" and is_log)
    _ci = _geo_mean_ci if use_geo else _mean_ci

    global_lo = []   # to pick a sensible log-axis bottom

    for ax, r in zip(axes, ROLES):
        means, los, his, colors, alphas = [], [], [], [], []
        for key, _lab, col, a in series_order:
            mu, lo, hi = _ci(bw[key][r])
            means.append(mu); los.append(lo); his.append(hi)
            colors.append(col); alphas.append(a)

        means = np.asarray(means, float)
        los   = np.asarray(los,   float)
        his   = np.asarray(his,   float)
        global_lo.extend(los[np.isfinite(los) & (los > 0)].tolist())

        # Asymmetric about the bar on a linear ruler, symmetric in log space.
        yerr = np.vstack([means - los, his - means])
        yerr = np.nan_to_num(np.clip(yerr, 0.0, None), nan=0.0, posinf=0.0, neginf=0.0)

        bars = ax.bar(x0, means, yerr=yerr, width=0.12, capsize=3,
                      color=colors, edgecolor="none")
        for b, a in zip(bars, alphas): b.set_alpha(a)

        ax.set_xticks(x0)
        ax.set_xticklabels([lab for _, lab, _, _ in series_order], rotation=20, ha="right")
        ax.set_title(r.capitalize())
        ax.set_ylabel("Bullwhip (Var orders / Var demand)")

        rlim = _get_role_ylim(r)
        if is_log:
            ax.set_yscale("log")
            top = (rlim[1] if rlim is not None else (ylim[1] if ylim is not None else None))
            # Provisional bottom; harmonised across panels after the loop.
            bottom = (rlim[0] if rlim is not None else log_floor)
            ax.set_ylim(bottom=bottom, top=top)
        else:
            if   rlim is not None: ax.set_ylim(*rlim)
            elif ylim is not None: ax.set_ylim(*ylim)
        ax.grid(axis="y", alpha=0.2)

    # If a genuine lower bound sits below log_floor, drop the floor to show it
    # rather than hiding the whisker end. Panels with an explicit ylim/ylims
    # are left alone.
    if is_log and global_lo:
        need = min(log_floor, min(global_lo) * 0.8)
        if need < log_floor:
            for ax, r in zip(axes, ROLES):
                if _get_role_ylim(r) is None and ylim is None:
                    ax.set_ylim(bottom=need, top=ax.get_ylim()[1])

    if annotate:
        if use_geo:
            note = ("Bars: geometric mean across sims. Whiskers: 95% t interval on "
                    "log(ratio), back-transformed (symmetric in log space).")
        else:
            note = "Bars: arithmetic mean across sims. Whiskers: 95% t interval."
        fig.text(0.5, 0.005, note, ha="center", fontsize=8, color="0.35")

    fig.suptitle(title, y=0.98, fontsize=12)
    fig.tight_layout(rect=(0, 0.03 if annotate else 0, 1, 0.94))
    if savepath: plt.savefig(savepath, bbox_inches="tight")
    plt.show()


# ── Call: per-role bullwhip on a log axis (Kimi + Qwen-3) ─────────
plot_bullwhip_effect_aggregate(
    model_dirs=model_dirs,
    human_param_df=human_df_sims,
    simulate_fn=simulate_beer_game,
    baseline_model="kimi-baseline",
    prompt_model="kimi-prompt",
    second_baseline="qwen3-baseline",
    second_prompt="qwen3-prompt",
    primary_label="Kimi K2",
    second_label="Qwen3.6 35B-A3B",
    sim_indices=range(11),
    yscale="linear",
    log_floor=0.1,
    ci_mode="geometric",
    savepath=f"{PLOTS_DIR}/bullwhip_aggregate.pdf",
)

In [ ]:
# ============================================================================
# MAIN AGGREGATE BULLWHIP FIGURE
#
# Model colors:
#   - Kimi K2: orange (#FC8D62)
#   - Qwen-3.6-35B-A3B: teal (#66C2A5)
#
# Treatment styling:
#   - Baseline: lighter model color with hatching
#   - Heuristic-aware: same model color, solid
#
# The empirical benchmark remains a separate solid color. Bars show the
# geometric mean across sessions with a 95% t interval on the log scale;
# dots show the individual sessions. The vertical axis is logarithmic.
# ============================================================================

from typing import Dict, Iterable, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import t as student_t

@nature_style
def plot_bullwhip_effect_aggregate_over_roles(
    *,
    model_dirs: Dict[str, str],
    human_param_df: pd.DataFrame,
    simulate_fn,
    baseline_model: str,
    prompt_model: str,
    second_baseline: Optional[str] = None,
    second_prompt: Optional[str] = None,
    primary_label: str = "Kimi K2",
    second_label: str = "Qwen-3.6-35B-A3B",
    sim_indices: Iterable[int] = range(11),
    demand_keys: Tuple[str, ...] = (
        "retailer_demand",
        "customer_demand",
        "demand",
        "retailer_order",
    ),
    weeks_fallback: int = 35,
    savepath: Optional[str] = None,
    role_weights: Optional[
        Union[
            Dict[str, float],
            Tuple[float, ...],
            List[float],
        ]
    ] = None,
    role_agg: str = "geometric",
    ci_mode: str = "geometric",
):
    """
    Compute the bullwhip ratio for each role and simulation, collapse the
    role-specific ratios into one simulation-level value, and summarize the
    simulation-level values.

    Parameters
    ----------
    role_agg:
        "geometric" or "arithmetic"

    ci_mode:
        "geometric" calculates a t interval in log space.
        "arithmetic" calculates a conventional mean-based t interval.
    """
    if role_agg not in {
        "geometric",
        "arithmetic",
    }:
        raise ValueError(
            "role_agg must be 'geometric' or 'arithmetic'."
        )

    if ci_mode not in {
        "geometric",
        "arithmetic",
    }:
        raise ValueError(
            "ci_mode must be 'geometric' or 'arithmetic'."
        )


    second_baseline_key = (
        second_baseline
        if (
            second_baseline
            and second_baseline in model_dirs
        )
        else None
    )

    second_prompt_key = (
        second_prompt
        if (
            second_prompt
            and second_prompt in model_dirs
        )
        else None
    )


    def _load(
        model_name,
        role,
        simulation_index,
    ):
        payload = load_simulated_data(
            model_dirs[model_name],
            sim_name=f"sim_{simulation_index}",
        )

        return np.asarray(
            payload[
                ROLE_KEYS[role]
            ],
            dtype=float,
        )


    def _var_ratio(
        orders,
        demand,
    ):
        orders = np.asarray(
            orders,
            dtype=float,
        )
        demand = np.asarray(
            demand,
            dtype=float,
        )

        length = min(
            len(orders),
            len(demand),
        )

        if length < 2:
            return np.nan

        order_variance = np.var(
            orders[:length],
            ddof=0,
        )
        demand_variance = np.var(
            demand[:length],
            ddof=0,
        )

        if demand_variance <= 0:
            return np.nan

        return float(
            order_variance
            / demand_variance
        )


    # ── Collect role-level bullwhip ratios ─────────────────────────
    bullwhip = {
        "primary_baseline": {
            role: []
            for role in ROLES
        },
        "primary_heuristic": {
            role: []
            for role in ROLES
        },
        "second_baseline": (
            {
                role: []
                for role in ROLES
            }
            if second_baseline_key
            else None
        ),
        "second_heuristic": (
            {
                role: []
                for role in ROLES
            }
            if second_prompt_key
            else None
        ),
        "empirical": {
            role: []
            for role in ROLES
        },
    }


    for simulation_index in sim_indices:
        number_of_weeks = None

        for model_key in [
            baseline_model,
            prompt_model,
            second_baseline_key,
            second_prompt_key,
        ]:
            if model_key is None:
                continue

            try:
                number_of_weeks = len(
                    _load(
                        model_key,
                        "retailer",
                        simulation_index,
                    )
                )
                break

            except Exception:
                pass

        number_of_weeks = int(
            number_of_weeks
            if number_of_weeks is not None
            else weeks_fallback
        )


        human_data = simulate_fn(
            **_human_kwargs_for_sim(
                human_param_df,
                simulation_index,
                weeks=number_of_weeks,
            )
        )["data"]


        empirical_data = simulate_fn(
            **_kwargs_from_param_series(
                _loho_series(
                    human_param_df,
                    simulation_index,
                ),
                weeks=number_of_weeks,
            )
        )["data"]


        demand = None

        for demand_key in demand_keys:
            if demand_key in human_data.columns:
                demand = human_data[
                    demand_key
                ].to_numpy(
                    dtype=float
                )
                break

        if demand is None:
            for demand_key in demand_keys:
                if demand_key in empirical_data.columns:
                    demand = empirical_data[
                        demand_key
                    ].to_numpy(
                        dtype=float
                    )
                    break

        if demand is None:
            demand = human_data[
                "retailer_order"
            ].to_numpy(
                dtype=float
            )


        for role in ROLES:
            bullwhip[
                "empirical"
            ][role].append(
                _var_ratio(
                    empirical_data[
                        f"{role}_order"
                    ].to_numpy(
                        dtype=float
                    ),
                    demand,
                )
            )


            try:
                bullwhip[
                    "primary_baseline"
                ][role].append(
                    _var_ratio(
                        _load(
                            baseline_model,
                            role,
                            simulation_index,
                        ),
                        demand,
                    )
                )
            except Exception:
                bullwhip[
                    "primary_baseline"
                ][role].append(
                    np.nan
                )


            try:
                bullwhip[
                    "primary_heuristic"
                ][role].append(
                    _var_ratio(
                        _load(
                            prompt_model,
                            role,
                            simulation_index,
                        ),
                        demand,
                    )
                )
            except Exception:
                bullwhip[
                    "primary_heuristic"
                ][role].append(
                    np.nan
                )


            if second_baseline_key:
                try:
                    bullwhip[
                        "second_baseline"
                    ][role].append(
                        _var_ratio(
                            _load(
                                second_baseline_key,
                                role,
                                simulation_index,
                            ),
                            demand,
                        )
                    )
                except Exception:
                    bullwhip[
                        "second_baseline"
                    ][role].append(
                        np.nan
                    )


            if second_prompt_key:
                try:
                    bullwhip[
                        "second_heuristic"
                    ][role].append(
                        _var_ratio(
                            _load(
                                second_prompt_key,
                                role,
                                simulation_index,
                            ),
                            demand,
                        )
                    )
                except Exception:
                    bullwhip[
                        "second_heuristic"
                    ][role].append(
                        np.nan
                    )


    # ── Role weights ───────────────────────────────────────────────
    if role_weights is None:
        weights = np.ones(
            len(ROLES),
            dtype=float,
        )

    elif isinstance(
        role_weights,
        dict,
    ):
        weights = np.asarray(
            [
                float(
                    role_weights.get(
                        role,
                        0.0,
                    )
                )
                for role in ROLES
            ],
            dtype=float,
        )

    else:
        role_weights_list = list(
            role_weights
        )

        if len(role_weights_list) != len(ROLES):
            raise ValueError(
                "role_weights must contain one value per role."
            )

        weights = np.asarray(
            role_weights_list,
            dtype=float,
        )


    def _aggregate_over_roles(series_key):
        """
        Collapse the four role-specific ratios into one value per simulation.
        """
        role_series = bullwhip[
            series_key
        ]

        number_of_simulations = max(
            len(
                role_series[role]
            )
            for role in ROLES
        )

        aggregated_values = []

        for simulation_index in range(
            number_of_simulations
        ):
            values = np.asarray(
                [
                    (
                        role_series[
                            role
                        ][simulation_index]
                        if simulation_index
                        < len(
                            role_series[role]
                        )
                        else np.nan
                    )
                    for role in ROLES
                ],
                dtype=float,
            )

            if role_agg == "geometric":
                valid = (
                    np.isfinite(values)
                    & (values > 0)
                    & (weights > 0)
                )
            else:
                valid = (
                    np.isfinite(values)
                    & (weights > 0)
                )

            if not np.any(valid):
                aggregated_values.append(
                    np.nan
                )
                continue

            effective_weights = (
                weights[valid]
                / np.sum(
                    weights[valid]
                )
            )

            if role_agg == "geometric":
                value = float(
                    np.exp(
                        np.sum(
                            effective_weights
                            * np.log(
                                values[valid]
                            )
                        )
                    )
                )
            else:
                value = float(
                    np.sum(
                        values[valid]
                        * effective_weights
                    )
                )

            aggregated_values.append(
                value
            )

        return aggregated_values


    # ── Labels and series order ────────────────────────────────────
    primary_baseline_label = (
        f"{primary_label} (baseline)"
    )
    primary_heuristic_label = (
        f"{primary_label} (heuristic-aware)"
    )

    second_baseline_label = (
        f"{second_label} (baseline)"
    )
    second_heuristic_label = (
        f"{second_label} (heuristic-aware)"
    )

    empirical_label = (
        "Empirical benchmark"
    )


    per_model_series = {
        primary_baseline_label: (
            _aggregate_over_roles(
                "primary_baseline"
            )
        ),
    }

    if second_baseline_key:
        per_model_series[
            second_baseline_label
        ] = _aggregate_over_roles(
            "second_baseline"
        )

    per_model_series[
        empirical_label
    ] = _aggregate_over_roles(
        "empirical"
    )

    if second_prompt_key:
        per_model_series[
            second_heuristic_label
        ] = _aggregate_over_roles(
            "second_heuristic"
        )

    per_model_series[
        primary_heuristic_label
    ] = _aggregate_over_roles(
        "primary_heuristic"
    )


    # ── Confidence intervals across simulations ────────────────────
    def _arithmetic_mean_ci(
        values,
        alpha=0.05,
    ):
        values = np.asarray(
            values,
            dtype=float,
        )

        values = values[
            np.isfinite(values)
        ]

        n = len(values)

        if n == 0:
            return (
                np.nan,
                np.nan,
                np.nan,
            )

        mean = float(
            np.mean(values)
        )

        standard_deviation = (
            float(
                np.std(
                    values,
                    ddof=1,
                )
            )
            if n > 1
            else 0.0
        )

        if n > 1 and standard_deviation > 0:
            half_width = (
                float(
                    student_t.ppf(
                        1 - alpha / 2,
                        n - 1,
                    )
                )
                * standard_deviation
                / np.sqrt(n)
            )
        else:
            half_width = 0.0

        return (
            mean,
            mean - half_width,
            mean + half_width,
        )


    def _geometric_mean_ci(
        values,
        alpha=0.05,
    ):
        values = np.asarray(
            values,
            dtype=float,
        )

        values = values[
            np.isfinite(values)
            & (values > 0)
        ]

        n = len(values)

        if n == 0:
            return (
                np.nan,
                np.nan,
                np.nan,
            )

        log_values = np.log(
            values
        )

        log_mean = float(
            np.mean(
                log_values
            )
        )

        log_standard_deviation = (
            float(
                np.std(
                    log_values,
                    ddof=1,
                )
            )
            if n > 1
            else 0.0
        )

        if n > 1 and log_standard_deviation > 0:
            half_width = (
                float(
                    student_t.ppf(
                        1 - alpha / 2,
                        n - 1,
                    )
                )
                * log_standard_deviation
                / np.sqrt(n)
            )
        else:
            half_width = 0.0

        return (
            float(
                np.exp(
                    log_mean
                )
            ),
            float(
                np.exp(
                    log_mean
                    - half_width
                )
            ),
            float(
                np.exp(
                    log_mean
                    + half_width
                )
            ),
        )


    confidence_interval_function = (
        _geometric_mean_ci
        if ci_mode == "geometric"
        else _arithmetic_mean_ci
    )


    labels = list(
        per_model_series.keys()
    )

    statistics = [
        confidence_interval_function(
            per_model_series[label]
        )
        for label in labels
    ]

    means = np.asarray(
        [
            statistic[0]
            for statistic in statistics
        ],
        dtype=float,
    )

    lower_bounds = np.asarray(
        [
            statistic[1]
            for statistic in statistics
        ],
        dtype=float,
    )

    upper_bounds = np.asarray(
        [
            statistic[2]
            for statistic in statistics
        ],
        dtype=float,
    )

    # ── Figure ────────────────────────────────────────────────────
    bar_style = {
        # label: (x position, colour, is baseline)
        empirical_label: (0.0, "#7570B3", False),
        primary_baseline_label: (1.25, "#FC8D62", True),
        primary_heuristic_label: (2.0, "#FC8D62", False),
        second_baseline_label: (3.25, "#66C2A5", True),
        second_heuristic_label: (4.0, "#66C2A5", False),
    }
    spec = [
        (bar_style[label][0], bar_style[label][1], bar_style[label][2],
         per_model_series[label], statistics[labels.index(label)])
        for label in [empirical_label, primary_baseline_label, primary_heuristic_label,
                      second_baseline_label, second_heuristic_label]
        if label in per_model_series
    ]
    fig, axis = plt.subplots(figsize=(HALF_WIDTH, 62 * MM))
    axis.set_yscale("log")
    all_values = np.concatenate([np.asarray(values, float) for *_, values, _ in spec])
    set_dot_spacing(all_values, log=True, frac=0.018)
    for x, color, is_baseline, values, (center, lower, upper) in spec:
        bar_with_dots(axis, x, values, color, 0.66, baseline=is_baseline, err=(lower, upper), center=center, dot_size=2.2)
    axis.set_xticks([0.0, 1.625, 3.625])
    axis.set_xticklabels(["Empirical\nbenchmark", primary_label, second_label])
    positive = all_values[np.isfinite(all_values) & (all_values > 0)]
    axis.set_ylim(bottom=positive.min() * 0.7)
    axis.set_ylabel("Bullwhip effect (order variance /\ndemand variance, log scale)")
    axis.legend(handles=condition_handles(), loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3,
                handlelength=1.4, columnspacing=1.0)

    save_figure(fig, savepath)
    plt.show()


plot_bullwhip_effect_aggregate_over_roles(
    model_dirs=model_dirs,
    human_param_df=human_df_sims,
    simulate_fn=simulate_beer_game,
    baseline_model="kimi-baseline",
    prompt_model="kimi-prompt",
    second_baseline="qwen3-baseline",
    second_prompt="qwen3-prompt",
    primary_label="Kimi K2",
    second_label="Qwen-3.6-35B-A3B",
    sim_indices=range(11),
    role_agg="geometric",
    ci_mode="geometric",
    savepath=(
        f"{PLOTS_DIR}/"
        "bullwhip_aggregate_over_roles.pdf"
    ),
)

## 7. Narayanan & Moritz Table 7 — Human Median Orders

Digitised median weekly orders (low and high cost-ratio conditions) from Fig 1 / Table 7.
We use the first 24 rounds throughout.


In [ ]:
# Narayanan & Moritz Table 7 / Fig 1: median weekly orders, weeks 1–29
# (digitised from the published plots). This is the ONLY 29-round horizon
# used in the notebook; LLM simulations are 35 rounds (the ABM default).
fig1_low_cr = {
    "retailer":    np.array([5,4,4,4,6,9,8,10,9,10,10,10,12,12,13,15,14,18,14,10,12,12,8,8,8,8,1,0,0], dtype=float),
    "wholesaler":  np.array([5,4,5,4,4,3,4,8,8,10,10,10,12,20,20,15,20,25,20,30,20,20,10,14,8,4,0,0,0], dtype=float),
    "distributor": np.array([5,4,4,4,4,4,4,2,4,6,10,13,16,20,20,20,20,20,29,24,20,25,10,8,10,9,4,1,0], dtype=float),
    "factory":     np.array([5,2,4,4,6,4,4,4,4,4,4,5,4,6,15,20,15,18,20,22,25,40,25,20,5,5,4,0,0], dtype=float),
}
fig1_high_cr = {
    "retailer":    np.array([5,5,5,5,6,8,9,9,10,10,12,10,8,10,10,8,8,8,8,8,8,8,8,8,8,8,7,8,8], dtype=float),
    "wholesaler":  np.array([5,4,5,5,5,4,5,10,8,10,10,12,11,12,12,11,15,10,8,10,9,10,9,8,8,5,5,5,6], dtype=float),
    "distributor": np.array([5,4,4,4,5,5,4,3,5,7,9,8,11,13,17,13,12,13,15,18,12,8,10,8,9,5,4,3,2], dtype=float),
    "factory":     np.array([5,4,4,6,4,6,5,4,4,4,1,1,4,6,8,12,14,15,14,12,10,9,9,6,2,3,5,4,0], dtype=float),
}

# Horizon constants for the Table-7 comparison cells (orders/bullwhip/final fig).
T7_ROUNDS = 29                                     # length of the digitised human data
WARMUP    = 4                                       # skip first 4 weeks (demand = 4 everywhere)
demand_t7 = np.array([4]*4 + [8]*(T7_ROUNDS - 4), dtype=float)


In [ ]:
def avg_model_orders(model_name: str, model_dirs: Dict[str, str],
                     sim_indices: Iterable[int] = range(11)) -> Dict[str, np.ndarray]:
    """Mean order trajectory across all sims for a given model (one per role)."""
    out = {r: [] for r in ROLES}
    for i in sim_indices:
        try:
            p = load_simulated_data(model_dirs[model_name], sim_name=f"sim_{i}")
            for r in ROLES:
                out[r].append(np.asarray(p[ROLE_KEYS[r]], dtype=float))
        except Exception:
            pass
    means = {}
    for r in ROLES:
        if out[r]:
            L = max(len(a) for a in out[r])
            padded = [np.concatenate([a, np.full(L - len(a), np.nan)]) if len(a) < L else a
                      for a in out[r]]
            means[r] = np.nanmean(np.vstack(padded), axis=0)
        else:
            means[r] = np.full(T7_ROUNDS, np.nan)
    return means


# Compute averaged LLM trajectories
kimi_base_avg    = avg_model_orders("kimi-baseline",  model_dirs)
kimi_prompt_avg  = avg_model_orders("kimi-prompt",    model_dirs)
qwen_base_avg    = avg_model_orders("qwen3-baseline", model_dirs)
qwen_prompt_avg  = avg_model_orders("qwen3-prompt",   model_dirs)

# ── Plot: LLM averages vs digitised Fig 1 (low/high CR), T7_ROUNDS rounds ─
# Baseline dashed, heuristic-aware solid; human medians dotted.
@nature_style
def plot_llm_orders_vs_cr_medians(series, low_cr, high_cr, *, rounds, savepath=None):
    fig = trajectory_panels(
        series,
        rounds=rounds,
        human_curves=[("Median human (low CR)", low_cr, "#756daf"),
                      ("Median human (high CR)", high_cr, "#e377c2")],
        ylabel="Orders",
    )
    save_figure(fig, savepath)
    plt.show()


plot_llm_orders_vs_cr_medians(
    [
        dict(label="Kimi K2 (baseline)", values=kimi_base_avg, color="#1d9275", ls="--"),
        dict(label="Kimi K2 (heuristic-aware)", values=kimi_prompt_avg, color="#cc545e"),
        dict(label="Qwen-3.6-35B-A3B (baseline)", values=qwen_base_avg, color="#66a61e", ls="--"),
        dict(label="Qwen-3.6-35B-A3B (heuristic-aware)", values=qwen_prompt_avg, color="#a86df0"),
    ],
    fig1_low_cr,
    fig1_high_cr,
    rounds=T7_ROUNDS,
    savepath=f"{PLOTS_DIR}/orders_table7_cr.pdf",
)

## 8. Table 7 Bullwhip plots
Per-role bar chart and aggregate-over-roles plot, comparing each LLM
against the two human CR medians (which act as deterministic points).


In [ ]:
# ============================================================================
# TABLE-7 BULLWHIP PLOTS
#
# Model colors:
#   - Kimi K2: orange (#FC8D62)
#   - Qwen3.6 35B-A3B: teal (#66C2A5)
#
# Treatment styling:
#   - Baseline: same model color, alpha=0.35, diagonal hatch
#   - Heuristic-aware: same model color, fully opaque, no hatch
#
# Produces:
#   1. bullwhip_per_role_table7.pdf
#   2. bullwhip_aggregate_table7.pdf
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import to_rgba
from scipy.stats import t as student_t


# ── Config ─────────────────────────────────────────────────────────
YSCALE = "linear"          # "linear" | "log"
LOG_FLOOR = 0.1            # Used only when YSCALE == "log"
CI_MODE = "geometric"      # "geometric" | "arithmetic"
ROLE_AGG = "geometric"     # "geometric" | "arithmetic"
ANNOTATE = False

SECOND_LABEL = "Qwen-3.6-35B-A3B"
SECOND_DIRS = (
    "qwen3-baseline",
    "qwen3-prompt",
)


# ── Shared visual configuration ────────────────────────────────────
# Strongly separated colors from the Set2 palette.
KIMI_COLOR = "#FC8D62"      # Orange
QWEN_COLOR = "#66C2A5"      # Teal

# Separate benchmark colors.
LOW_HUMAN_COLOR = "#7570B3"
HIGH_HUMAN_COLOR = "#E7298A"

BASELINE_ALPHA = 0.35
BASELINE_HATCH = "//"

BAR_EDGE_COLOR = "black"
BAR_EDGE_WIDTH = 0.6
ERROR_BAR_COLOR = "black"


def _facecolor(base_color, is_baseline):
    """
    Return the same RGB color for both treatments, changing only opacity.
    """
    alpha = BASELINE_ALPHA if is_baseline else 1.0
    return to_rgba(base_color, alpha)


# ── Statistical helpers ────────────────────────────────────────────
def _var_ratio(orders, demand):
    """
    Variance ratio used as the bullwhip measure.
    """
    orders = np.asarray(orders, dtype=float)
    demand = np.asarray(demand, dtype=float)

    length = min(len(orders), len(demand))

    if length < 2:
        return np.nan

    order_variance = np.var(
        orders[:length],
        ddof=0,
    )
    demand_variance = np.var(
        demand[:length],
        ddof=0,
    )

    if demand_variance <= 0:
        return np.nan

    return float(
        order_variance / demand_variance
    )


def _mean_ci(values, alpha=0.05):
    """
    Arithmetic mean with a t confidence interval.

    Returns:
        center, lower_bound, upper_bound
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    n = len(values)

    if n == 0:
        return np.nan, np.nan, np.nan

    mean = float(
        np.mean(values)
    )

    standard_deviation = (
        float(
            np.std(
                values,
                ddof=1,
            )
        )
        if n > 1
        else 0.0
    )

    if n > 1 and standard_deviation > 0:
        half_width = (
            float(
                student_t.ppf(
                    1 - alpha / 2,
                    n - 1,
                )
            )
            * standard_deviation
            / np.sqrt(n)
        )
    else:
        half_width = 0.0

    return (
        mean,
        mean - half_width,
        mean + half_width,
    )


def _geo_mean_ci(values, alpha=0.05):
    """
    Geometric mean with a t confidence interval calculated in log space.

    Non-positive and non-finite observations are removed because log is
    undefined for those values.

    Returns:
        center, lower_bound, upper_bound
    """
    values = np.asarray(values, dtype=float)
    values = values[
        np.isfinite(values)
        & (values > 0)
    ]

    n = len(values)

    if n == 0:
        return np.nan, np.nan, np.nan

    log_values = np.log(values)

    log_mean = float(
        np.mean(log_values)
    )

    log_standard_deviation = (
        float(
            np.std(
                log_values,
                ddof=1,
            )
        )
        if n > 1
        else 0.0
    )

    if n > 1 and log_standard_deviation > 0:
        half_width = (
            float(
                student_t.ppf(
                    1 - alpha / 2,
                    n - 1,
                )
            )
            * log_standard_deviation
            / np.sqrt(n)
        )
    else:
        half_width = 0.0

    return (
        float(np.exp(log_mean)),
        float(np.exp(log_mean - half_width)),
        float(np.exp(log_mean + half_width)),
    )


_ci = (
    _geo_mean_ci
    if CI_MODE == "geometric"
    else _mean_ci
)


def _yerr_from(
    means,
    lower_bounds,
    upper_bounds,
):
    """
    Convert explicit lower and upper confidence bounds to Matplotlib yerr.
    """
    means = np.asarray(means, dtype=float)
    lower_bounds = np.asarray(
        lower_bounds,
        dtype=float,
    )
    upper_bounds = np.asarray(
        upper_bounds,
        dtype=float,
    )

    yerr = np.vstack(
        [
            means - lower_bounds,
            upper_bounds - means,
        ]
    )

    return np.nan_to_num(
        np.clip(
            yerr,
            0.0,
            None,
        ),
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )


def _log_bottom(lower_bounds):
    """
    Lower the log-axis floor when a real confidence bound lies below
    LOG_FLOOR.
    """
    lower_bounds = np.asarray(
        lower_bounds,
        dtype=float,
    )

    valid_lower_bounds = lower_bounds[
        np.isfinite(lower_bounds)
        & (lower_bounds > 0)
    ]

    if valid_lower_bounds.size == 0:
        return LOG_FLOOR

    return min(
        LOG_FLOOR,
        float(
            valid_lower_bounds.min()
        )
        * 0.8,
    )


def _collapse_roles(values, method):
    """
    Collapse the four role-specific bullwhip ratios into one value.
    """
    values = np.asarray(
        values,
        dtype=float,
    )

    if method == "geometric":
        valid = (
            np.isfinite(values)
            & (values > 0)
        )

        if not np.any(valid):
            return np.nan

        return float(
            np.exp(
                np.mean(
                    np.log(
                        values[valid]
                    )
                )
            )
        )

    valid = np.isfinite(values)

    if not np.any(valid):
        return np.nan

    return float(
        np.mean(
            values[valid]
        )
    )


# ── Human bullwhip ─────────────────────────────────────────────────
# One deterministic value per CR condition and role.
human_bw = {
    condition: {
        role: _var_ratio(
            role_data[role],
            demand_t7,
        )
        for role in ROLES
    }
    for condition, role_data in [
        (
            "low_cr",
            fig1_low_cr,
        ),
        (
            "high_cr",
            fig1_high_cr,
        ),
    ]
}


# ── LLM bullwhip per simulation ────────────────────────────────────
bw = {
    model_key: {
        role: []
        for role in ROLES
    }
    for model_key in (
        "kimi_raw",
        "kimi_heur",
        "qwen_raw",
        "qwen_heur",
    )
}

model_sources = {
    "kimi_raw": "kimi-baseline",
    "kimi_heur": "kimi-prompt",
    "qwen_raw": SECOND_DIRS[0],
    "qwen_heur": SECOND_DIRS[1],
}

sim_indices = range(11)

for simulation_index in sim_indices:
    for role in ROLES:
        for model_key, directory_key in model_sources.items():
            try:
                payload = load_simulated_data(
                    model_dirs[directory_key],
                    sim_name=f"sim_{simulation_index}",
                )

                orders = np.asarray(
                    payload[
                        ROLE_KEYS[role]
                    ],
                    dtype=float,
                )[:T7_ROUNDS]

                bw[model_key][role].append(
                    _var_ratio(
                        orders,
                        demand_t7,
                    )
                )

            except Exception:
                bw[model_key][role].append(
                    np.nan
                )


# ============================================================================
# PLOT 1: Bullwhip by role
# ============================================================================

# Tuple format:
#   data key, display label, model/benchmark color, is baseline
series_order = [
    (
        "kimi_raw",
        "Kimi K2 (baseline)",
        KIMI_COLOR,
        True,
    ),
    (
        "kimi_heur",
        "Kimi K2 (heuristic-aware)",
        KIMI_COLOR,
        False,
    ),
    (
        "qwen_raw",
        f"{SECOND_LABEL} (baseline)",
        QWEN_COLOR,
        True,
    ),
    (
        "qwen_heur",
        f"{SECOND_LABEL} (heuristic-aware)",
        QWEN_COLOR,
        False,
    ),
    (
        "low_cr",
        "Median human (low CR)",
        LOW_HUMAN_COLOR,
        False,
    ),
    (
        "high_cr",
        "Median human (high CR)",
        HIGH_HUMAN_COLOR,
        False,
    ),
]

is_log = (
    YSCALE == "log"
)

figure, axes = plt.subplots(
    2,
    2,
    figsize=(12, 8),
)

axes = axes.ravel()

x_positions = np.arange(
    len(series_order)
)

all_lower_bounds = []

for axis, role in zip(
    axes,
    ROLES,
):
    means = []
    lower_bounds = []
    upper_bounds = []
    facecolors = []
    hatches = []

    for (
        model_key,
        _display_label,
        base_color,
        is_baseline,
    ) in series_order:

        if model_key in (
            "low_cr",
            "high_cr",
        ):
            # Human bars are deterministic, so no interval is shown.
            value = human_bw[
                model_key
            ][role]

            mean = value
            lower_bound = value
            upper_bound = value

        else:
            (
                mean,
                lower_bound,
                upper_bound,
            ) = _ci(
                bw[
                    model_key
                ][role]
            )

        means.append(mean)
        lower_bounds.append(
            lower_bound
        )
        upper_bounds.append(
            upper_bound
        )

        facecolors.append(
            _facecolor(
                base_color,
                is_baseline,
            )
        )

        hatches.append(
            (
                BASELINE_HATCH
                if is_baseline
                else None
            )
        )

    means = np.asarray(
        means,
        dtype=float,
    )
    lower_bounds = np.asarray(
        lower_bounds,
        dtype=float,
    )
    upper_bounds = np.asarray(
        upper_bounds,
        dtype=float,
    )

    valid_lower_bounds = lower_bounds[
        np.isfinite(lower_bounds)
        & (lower_bounds > 0)
    ]

    all_lower_bounds.extend(
        valid_lower_bounds.tolist()
    )

    bars = axis.bar(
        x_positions,
        means,
        yerr=_yerr_from(
            means,
            lower_bounds,
            upper_bounds,
        ),
        width=0.55,
        capsize=3,
        color=facecolors,
        edgecolor=BAR_EDGE_COLOR,
        linewidth=BAR_EDGE_WIDTH,
        ecolor=ERROR_BAR_COLOR,
        zorder=2,
    )

    for bar, hatch in zip(
        bars,
        hatches,
    ):
        if hatch:
            bar.set_hatch(
                hatch
            )

    axis.set_xticks(
        x_positions
    )

    axis.set_xticklabels(
        [
            display_label
            for (
                _,
                display_label,
                _,
                _,
            ) in series_order
        ],
        rotation=25,
        ha="right",
        fontsize=8,
    )

    axis.set_title(
        role.capitalize()
    )

    axis.set_ylabel(
        "Bullwhip (Var orders / Var demand)"
    )

    if is_log:
        axis.set_yscale(
            "log"
        )

        axis.set_ylim(
            bottom=LOG_FLOOR
        )

    axis.spines[
        "top"
    ].set_visible(False)

    axis.spines[
        "right"
    ].set_visible(False)

    axis.set_axisbelow(True)

    axis.grid(
        axis="y",
        alpha=0.2,
    )


# Harmonize the log-axis floor across all role panels.
if is_log and all_lower_bounds:
    required_bottom = _log_bottom(
        all_lower_bounds
    )

    if required_bottom < LOG_FLOOR:
        for axis in axes:
            axis.set_ylim(
                bottom=required_bottom,
                top=axis.get_ylim()[1],
            )


figure.suptitle(
    f"Bullwhip by Role ({T7_ROUNDS} rounds)",
    y=0.98,
    fontsize=12,
)

if ANNOTATE:
    if CI_MODE == "geometric":
        note = (
            "LLM bars: geometric mean across simulations with a "
            "95% t interval on log(ratio). Human bars: one "
            "deterministic value with no interval."
        )
    else:
        note = (
            "LLM bars: arithmetic mean across simulations with a "
            "95% t interval. Human bars: one deterministic value "
            "with no interval."
        )

    figure.text(
        0.5,
        0.005,
        note,
        ha="center",
        fontsize=8,
        color="0.35",
    )

figure.tight_layout(
    rect=(
        0,
        0.03 if ANNOTATE else 0,
        1,
        0.94,
    )
)

plt.savefig(
    f"{PLOTS_DIR}/bullwhip_per_role_table7.pdf",
    bbox_inches="tight",
)

plt.show()


# ============================================================================
# PLOT 2: Aggregate bullwhip across roles
# ============================================================================

def _aggregate_llm_over_roles(model_key):
    """
    Collapse each LLM simulation's four role ratios into one value.
    """
    number_of_simulations = max(
        len(
            bw[model_key][role]
        )
        for role in ROLES
    )

    aggregated_values = []

    for simulation_index in range(
        number_of_simulations
    ):
        role_values = [
            (
                bw[
                    model_key
                ][role][simulation_index]
                if simulation_index
                < len(
                    bw[
                        model_key
                    ][role]
                )
                else np.nan
            )
            for role in ROLES
        ]

        aggregated_values.append(
            _collapse_roles(
                role_values,
                ROLE_AGG,
            )
        )

    return aggregated_values


def _aggregate_human_over_roles(condition):
    """
    Collapse the four deterministic human role values.
    """
    return [
        _collapse_roles(
            [
                human_bw[
                    condition
                ][role]
                for role in ROLES
            ],
            ROLE_AGG,
        )
    ]


kimi_baseline_label = (
    "Kimi K2 (baseline)"
)
kimi_heuristic_label = (
    "Kimi K2 (heuristic-aware)"
)

qwen_baseline_label = (
    f"{SECOND_LABEL} (baseline)"
)
qwen_heuristic_label = (
    f"{SECOND_LABEL} (heuristic-aware)"
)

low_human_label = (
    "Median human (low CR)"
)
high_human_label = (
    "Median human (high CR)"
)


aggregate_series = {
    kimi_baseline_label: (
        _aggregate_llm_over_roles(
            "kimi_raw"
        )
    ),
    qwen_baseline_label: (
        _aggregate_llm_over_roles(
            "qwen_raw"
        )
    ),
    low_human_label: (
        _aggregate_human_over_roles(
            "low_cr"
        )
    ),
    high_human_label: (
        _aggregate_human_over_roles(
            "high_cr"
        )
    ),
    qwen_heuristic_label: (
        _aggregate_llm_over_roles(
            "qwen_heur"
        )
    ),
    kimi_heuristic_label: (
        _aggregate_llm_over_roles(
            "kimi_heur"
        )
    ),
}


# Tuple format:
#   base color, is baseline, is deterministic human value
aggregate_style = {
    kimi_baseline_label: (
        KIMI_COLOR,
        True,
        False,
    ),
    kimi_heuristic_label: (
        KIMI_COLOR,
        False,
        False,
    ),
    qwen_baseline_label: (
        QWEN_COLOR,
        True,
        False,
    ),
    qwen_heuristic_label: (
        QWEN_COLOR,
        False,
        False,
    ),
    low_human_label: (
        LOW_HUMAN_COLOR,
        False,
        True,
    ),
    high_human_label: (
        HIGH_HUMAN_COLOR,
        False,
        True,
    ),
}


labels = list(
    aggregate_series.keys()
)

statistics = []

for label in labels:
    values = aggregate_series[
        label
    ]

    is_human = aggregate_style[
        label
    ][2]

    if is_human:
        # Human aggregate is one deterministic value.
        value = (
            float(values[0])
            if len(values)
            else np.nan
        )

        statistics.append(
            (
                value,
                value,
                value,
            )
        )

    else:
        statistics.append(
            _ci(values)
        )


# Bars in paper order: human medians, then each model's baseline and heuristic-aware runs.
# LLM bars: geometric mean across sessions with a 95% t interval on log(ratio), dots = sessions.
# Human bars: one deterministic value with no interval.
aggregate_positions = {
    low_human_label: 0.0,
    high_human_label: 0.75,
    kimi_baseline_label: 2.0,
    kimi_heuristic_label: 2.75,
    qwen_baseline_label: 4.0,
    qwen_heuristic_label: 4.75,
}
aggregate_colors = {
    KIMI_COLOR: "#FC8D62",
    QWEN_COLOR: "#66C2A5",
    LOW_HUMAN_COLOR: "#756DBD",    # colour-blind-safe shade of the original
    HIGH_HUMAN_COLOR: "#E7298A",
}


@nature_style
def plot_bullwhip_aggregate_table7(savepath=None):
    fig, axis = plt.subplots(figsize=(HALF_WIDTH, 62 * MM))
    axis.set_yscale("log")
    all_values = np.concatenate([np.asarray(aggregate_series[label], float) for label in labels])
    set_dot_spacing(all_values, log=True, frac=0.018)
    for label, (center, lower, upper) in zip(labels, statistics):
        color, is_baseline, is_human = aggregate_style[label]
        x = aggregate_positions[label]
        if is_human:
            axis.bar(x, center, width=0.66, color=aggregate_colors[color], edgecolor="black",
                     linewidth=0.4, zorder=2)
        else:
            bar_with_dots(axis, x, aggregate_series[label], aggregate_colors[color], 0.66, baseline=is_baseline,
                err=(lower, upper), center=center, dot_size=2.2)
    axis.set_xticks([0.0, 0.75, 2.375, 4.375])
    axis.set_xticklabels(["Human\nlow CR", "Human\nhigh CR", "Kimi K2", SECOND_LABEL])
    positive = all_values[np.isfinite(all_values) & (all_values > 0)]
    axis.set_ylim(bottom=positive.min() * 0.7)
    axis.set_ylabel("Bullwhip effect (order variance /\ndemand variance, log scale)")
    axis.legend(handles=condition_handles(), loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3,
                handlelength=1.4, columnspacing=1.0)
    save_figure(fig, savepath)
    plt.show()


plot_bullwhip_aggregate_table7(f"{PLOTS_DIR}/bullwhip_aggregate_table7.pdf")

## 9. Relative log-MSE evaluation — LLM vs human-calibrated target

The primary scalar score is

\[
\mathrm{Relative\ log\text{-}MSE}
=
\log\!\left(
\frac{\mathrm{MSE}_{\mathrm{LOHO}}}
     {\mathrm{MSE}_{\mathrm{LLM}}}
\right).
\]

Positive values indicate lower trajectory MSE than the leave-one-human-out
predictor, zero indicates equal MSE, and negative values indicate higher MSE.
This is a trajectory-distance score. It is **not** a likelihood for the LLM
decision process.


In [ ]:
def _safe_corrs(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> Tuple[float, float]:
    """Return Pearson r and Spearman rho; return NaNs when undefined."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if y_true.size < 2:
        return np.nan, np.nan

    try:
        pearson_result = pearsonr(y_pred, y_true)
        pearson_value = (
            pearson_result.statistic
            if hasattr(pearson_result, "statistic")
            else pearson_result[0]
        )
    except Exception:
        pearson_value = np.nan

    try:
        spearman_value = spearmanr(y_pred, y_true).correlation
    except Exception:
        spearman_value = np.nan

    return float(pearson_value), float(spearman_value)


def _align_three_sequences(
    target: np.ndarray,
    predictor: np.ndarray,
    loho: np.ndarray,
    *,
    warmup_weeks: int = 0,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Trim, align, and jointly mask target, predictor, and LOHO sequences."""
    target = np.asarray(target, dtype=float)
    predictor = np.asarray(predictor, dtype=float)
    loho = np.asarray(loho, dtype=float)

    if warmup_weeks > 0:
        target = target[warmup_weeks:]
        predictor = predictor[warmup_weeks:]
        loho = loho[warmup_weeks:]

    n = min(len(target), len(predictor), len(loho))
    target = target[:n]
    predictor = predictor[:n]
    loho = loho[:n]

    mask = (
        np.isfinite(target)
        & np.isfinite(predictor)
        & np.isfinite(loho)
    )
    return target[mask], predictor[mask], loho[mask]


In [ ]:
def evaluate_llm_vs_human_sequences_with_log_mse(
    model_extracted_dir: str,
    human_param_df: pd.DataFrame,
    simulate_fn,
    sim_indices: Iterable[int] = range(11),
    *,
    series: str = "order",
    warmup_weeks: int = 0,
    human_baseline: str = "loho",
    initial_inventory: int = 12,
    initial_order: int = 4,
    verbose: bool = True,
    eps: float = 1e-12,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compare one LLM directory with session-specific Sterman targets.

    Primary score
    -------------
    Relative_log_MSE = log(MSE_LOHO / MSE_LLM)

    Interpretation
    --------------
    > 0 : LLM has lower MSE than the LOHO predictor.
    = 0 : LLM and LOHO have equal MSE, but may differ temporally.
    < 0 : LLM has higher MSE than the LOHO predictor.
    """
    if series not in {"order", "inventory", "backlog"}:
        raise ValueError("series must be 'order', 'inventory', or 'backlog'.")
    if human_baseline not in {"avg", "loho"}:
        raise ValueError("human_baseline must be 'avg' or 'loho'.")

    records: List[Dict[str, Any]] = []

    for sim_index in sim_indices:
        sim_name = f"sim_{sim_index}"
        if verbose:
            print(f"Processing {sim_name}...")

        try:
            llm_payload = load_simulated_data(
                model_extracted_dir,
                sim_name=sim_name,
            )
        except Exception as error:
            if verbose:
                print(f"  Skipping {sim_name}: {error}")
            continue

        weeks = int(len(llm_payload["retailer_order"]))

        target_df = simulate_fn(
            **_human_kwargs_for_sim(
                human_param_df,
                sim_index,
                weeks=weeks,
                initial_inventory=initial_inventory,
                initial_order=initial_order,
            )
        )["data"]

        baseline_parameters = _human_baseline_series(
            human_param_df,
            sim_index,
            mode=human_baseline,
        )
        loho_df = simulate_fn(
            **_kwargs_from_param_series(
                baseline_parameters,
                weeks=weeks,
                initial_inventory=initial_inventory,
                initial_order=initial_order,
            )
        )["data"]

        for role in ROLES:
            key = f"{role}_{series}"

            target, model, loho = _align_three_sequences(
                target_df[key].to_numpy(float),
                np.asarray(llm_payload[key], dtype=float),
                loho_df[key].to_numpy(float),
                warmup_weeks=warmup_weeks,
            )

            n = int(target.size)
            if n == 0:
                continue

            model_error = model - target
            loho_error = loho - target

            mse_model = float(np.mean(model_error ** 2))
            mse_loho = float(np.mean(loho_error ** 2))
            mse_model_safe = max(mse_model, eps)
            mse_loho_safe = max(mse_loho, eps)

            relative_log_mse = float(
                np.log(mse_loho_safe)
                - np.log(mse_model_safe)
            )
            mse_ratio_to_loho = float(
                mse_model_safe / mse_loho_safe
            )

            pearson_value, spearman_value = _safe_corrs(
                target,
                model,
            )

            records.append({
                "sim": sim_name,
                "role": role,
                "series": series,
                "n": n,
                "MAE": float(np.mean(np.abs(model_error))),
                "RMSE": float(np.sqrt(mse_model)),
                "Max_abs_error": float(np.max(np.abs(model_error))),
                "Pearson_r": pearson_value,
                "Spearman_rho": spearman_value,
                "MSE_model": mse_model,
                "MSE_LOHO": mse_loho,
                "MSE_ratio_to_LOHO": mse_ratio_to_loho,
                "Relative_log_MSE": relative_log_mse,
            })

    if not records:
        raise RuntimeError(
            f"No comparisons were produced for {model_extracted_dir}. "
            "Check the directory, sim_indices, and JSON contents."
        )

    per_sim_df = (
        pd.DataFrame.from_records(records)
        .set_index(["sim", "role"])
        .sort_index()
    )
    per_sim_df.index = per_sim_df.index.set_names(["sim", "role"])

    columns_to_average = [
        "MAE",
        "RMSE",
        "Max_abs_error",
        "Pearson_r",
        "Spearman_rho",
        "MSE_model",
        "MSE_LOHO",
        "MSE_ratio_to_LOHO",
        "Relative_log_MSE",
    ]
    aggregate_df = (
        per_sim_df
        .groupby("role")[columns_to_average]
        .mean(numeric_only=True)
        .reindex(ROLES)
    )

    overall = pd.DataFrame(
        aggregate_df.mean(numeric_only=True)
    ).T
    overall.index = pd.Index(["OVERALL"], name="role")
    aggregate_df = pd.concat([aggregate_df, overall], axis=0)
    aggregate_df.index.name = "role"

    return per_sim_df, aggregate_df


In [ ]:
def evaluate_many_models_vs_human_with_log_mse(
    model_dirs: Union[
        Dict[str, str],
        Iterable[Tuple[str, str]],
    ],
    human_param_df: pd.DataFrame,
    simulate_fn,
    sim_indices: Iterable[int] = range(11),
    *,
    series: str = "order",
    warmup_weeks: int = 0,
    human_baseline: str = "loho",
    initial_inventory: int = 12,
    initial_order: int = 4,
    verbose: bool = True,
    rank_by: str = "Relative_log_MSE",
    rank_ascending: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Evaluate multiple model directories using relative log-MSE."""
    if not isinstance(model_dirs, dict):
        model_dirs = dict(model_dirs)

    all_per_sim: List[pd.DataFrame] = []
    all_aggregate: List[pd.DataFrame] = []

    for model_name, model_path in model_dirs.items():
        if verbose:
            print(f"\n=== Evaluating model: {model_name} ===")

        try:
            per_sim_df, aggregate_df = (
                evaluate_llm_vs_human_sequences_with_log_mse(
                    model_extracted_dir=model_path,
                    human_param_df=human_param_df,
                    simulate_fn=simulate_fn,
                    sim_indices=sim_indices,
                    series=series,
                    warmup_weeks=warmup_weeks,
                    human_baseline=human_baseline,
                    initial_inventory=initial_inventory,
                    initial_order=initial_order,
                    verbose=verbose,
                )
            )
        except RuntimeError as error:
            warnings.warn(str(error))
            continue

        per_sim_df = per_sim_df.copy()
        per_sim_df["model"] = model_name
        per_sim_df = (
            per_sim_df
            .set_index("model", append=True)
            .reorder_levels(["model", "sim", "role"])
        )
        all_per_sim.append(per_sim_df)

        aggregate_df = aggregate_df.copy()
        aggregate_df["model"] = model_name
        aggregate_df = (
            aggregate_df
            .set_index("model", append=True)
            .reorder_levels(["model", "role"])
        )
        all_aggregate.append(aggregate_df)

    if not all_per_sim:
        raise RuntimeError("No model evaluations completed.")

    per_sim_all = pd.concat(all_per_sim).sort_index()
    per_sim_all.index = per_sim_all.index.set_names(
        ["model", "sim", "role"]
    )

    per_model_role = pd.concat(all_aggregate).sort_index()
    per_model_role.index = per_model_role.index.set_names(
        ["model", "role"]
    )

    overall = per_model_role[
        per_model_role.index.get_level_values("role") == "OVERALL"
    ].droplevel("role")

    if rank_by not in overall.columns:
        raise KeyError(
            f"rank_by={rank_by!r} not found in {list(overall.columns)}"
        )

    leaderboard = overall.sort_values(
        rank_by,
        ascending=rank_ascending,
    )
    return per_sim_all, per_model_role, leaderboard


## 10. Shared statistical helpers

Consolidated once: BH-FDR correction, Welch CI, Hedges' g, and significance stars.
The original script defined these 4+ times — we use a single canonical version below.


In [ ]:
def _bh_fdr(pvals):
    """Benjamini–Hochberg FDR-adjusted q-values."""
    p = np.asarray(pvals, float)
    m = p.size
    order = np.argsort(p)
    ranked = p[order]
    q = ranked * m / np.arange(1, m + 1, dtype=float)
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0.0, 1.0)
    out = np.empty_like(q); out[order] = q
    return out


def _stars(p):
    """Significance stars: *** <1e-3, ** <1e-2, * <5e-2."""
    if not np.isfinite(p): return ""
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 5e-2 else ""


def _welch_ci(mean_diff, s1, n1, s2, n2, alpha=0.05) -> Tuple[float, float, float]:
    """95% CI for (group2 − group1) using Welch–Satterthwaite df. Returns (df, lo, hi)."""
    n1 = max(int(n1), 1); n2 = max(int(n2), 1)
    se = np.sqrt((s1 ** 2) / n1 + (s2 ** 2) / n2)
    if not np.isfinite(se) or se == 0 or (n1 < 2 and n2 < 2):
        return np.nan, np.nan, np.nan
    num = (s1 ** 2 / n1 + s2 ** 2 / n2) ** 2
    den = ((s1 ** 2 / n1) ** 2 / (n1 - 1 if n1 > 1 else 1)) \
        + ((s2 ** 2 / n2) ** 2 / (n2 - 1 if n2 > 1 else 1))
    df = num / den if den > 0 else np.nan
    tcrit = float(student_t.ppf(1 - alpha / 2, df)) if np.isfinite(df) else np.nan
    halfw = tcrit * se if np.isfinite(tcrit) else np.nan
    return df, mean_diff - halfw, mean_diff + halfw


def _hedges_g(mean1, mean2, s1, s2, n1, n2):
    """Hedges' g (small-sample-corrected Cohen's d) for (mean2 − mean1)."""
    if n1 < 2 or n2 < 2: return np.nan
    sp2 = ((n1 - 1) * s1 ** 2 + (n2 - 1) * s2 ** 2) / (n1 + n2 - 2)
    if not np.isfinite(sp2) or sp2 <= 0: return np.nan
    d = (mean2 - mean1) / np.sqrt(sp2)
    J = 1 - 3 / (4 * (n1 + n2) - 9)
    return float(J * d)


def _apply_correction(out_df: pd.DataFrame, col_in: str, col_out: str, method: str):
    """In-place: add a corrected-pvalue column."""
    method = method.lower()
    if method == "bonferroni":
        m = int(out_df[col_in].notna().sum())
        out_df[col_out] = np.minimum(out_df[col_in] * m, 1.0)
    elif method == "fdr_bh":
        mask = out_df[col_in].notna().to_numpy()
        q = np.full(len(out_df), np.nan)
        if mask.any():
            q[mask] = _bh_fdr(out_df.loc[mask, col_in].to_list())
        out_df[col_out] = q
    elif method == "none":
        out_df[col_out] = out_df[col_in]
    else:
        raise ValueError("correction must be 'fdr_bh', 'bonferroni', or 'none'.")


In [ ]:
def tests_model_vs_loho_parity(
    per_sim_all: pd.DataFrame,
    *,
    metric: str = "Relative_log_MSE",
    categories: Sequence[str] = (
        "OVERALL",
        "retailer",
        "wholesaler",
        "distributor",
        "factory",
    ),
    correction: str = "fdr_bh",
    alternative: str = "two-sided",
    min_n: int = 3,
) -> pd.DataFrame:
    """
    Test session-level relative log-MSE against zero.

    The result schema remains compatible with the grouped plotting helper:
    `diff_base_minus_model` is the mean relative log-MSE. Positive values
    indicate lower MSE than LOHO.
    """
    if not isinstance(per_sim_all.index, pd.MultiIndex):
        raise ValueError(
            "per_sim_all must be indexed by (model, sim, role)."
        )
    if alternative not in {"two-sided", "greater", "less"}:
        raise ValueError("Invalid alternative.")

    data = per_sim_all.copy()
    data.index = data.index.set_names(["model", "sim", "role"])
    data = data.reset_index()

    rows: List[Dict[str, Any]] = []

    for model in data["model"].drop_duplicates():
        for category in categories:
            if category == "OVERALL":
                values = (
                    data[data["model"] == model]
                    .groupby("sim", as_index=False)[metric]
                    .mean()[metric]
                    .to_numpy(float)
                )
            else:
                values = (
                    data[
                        (data["model"] == model)
                        & (data["role"] == category)
                    ]
                    .groupby("sim", as_index=False)[metric]
                    .mean()[metric]
                    .to_numpy(float)
                )

            values = values[np.isfinite(values)]
            n = int(values.size)
            mean_value = float(np.mean(values)) if n else np.nan
            sd_value = (
                float(np.std(values, ddof=1))
                if n > 1
                else np.nan
            )
            sem_value = (
                sd_value / np.sqrt(n)
                if n > 1 and np.isfinite(sd_value)
                else np.nan
            )

            if n >= min_n:
                t_stat, p_requested = ttest_1samp(
                    values,
                    popmean=0.0,
                    alternative=alternative,
                )
                _, p_two_sided = ttest_1samp(
                    values,
                    popmean=0.0,
                    alternative="two-sided",
                )
                t_critical = float(student_t.ppf(0.975, n - 1))
                ci_low = mean_value - t_critical * sem_value
                ci_high = mean_value + t_critical * sem_value

                try:
                    wilcoxon_result = wilcoxon(
                        values,
                        alternative=alternative,
                        zero_method="wilcox",
                    )
                    p_wilcoxon = float(wilcoxon_result.pvalue)
                except Exception:
                    p_wilcoxon = np.nan
            else:
                t_stat = np.nan
                p_requested = np.nan
                p_two_sided = np.nan
                p_wilcoxon = np.nan
                ci_low = np.nan
                ci_high = np.nan

            standardized_mean = (
                mean_value / sd_value
                if n > 1 and np.isfinite(sd_value) and sd_value > 0
                else np.nan
            )

            rows.append({
                "model": model,
                "category": category,
                "n_model": n,
                "n_baseline": n,
                "mean_model": mean_value,
                "mean_baseline": 0.0,
                "diff_base_minus_model": mean_value,
                "t_welch": (
                    float(t_stat)
                    if np.isfinite(t_stat)
                    else np.nan
                ),
                "df_welch": float(n - 1) if n > 1 else np.nan,
                "p_welch": (
                    float(p_requested)
                    if np.isfinite(p_requested)
                    else np.nan
                ),
                "p_welch_2s": (
                    float(p_two_sided)
                    if np.isfinite(p_two_sided)
                    else np.nan
                ),
                "p_wilcoxon": p_wilcoxon,
                "ci95_low": ci_low,
                "ci95_high": ci_high,
                "hedges_g": standardized_mean,
            })

    output = pd.DataFrame(rows)
    _apply_correction(
        output,
        "p_welch",
        "p_welch_corr",
        correction,
    )
    _apply_correction(
        output,
        "p_welch_2s",
        "p_welch_2s_corr",
        correction,
    )
    _apply_correction(
        output,
        "p_wilcoxon",
        "p_wilcoxon_corr",
        correction,
    )

    output["sig"] = output["p_welch_corr"].apply(_stars)
    output["sig_2s"] = output["p_welch_2s_corr"].apply(_stars)

    return output.sort_values(
        ["category", "diff_base_minus_model"],
        ascending=[True, False],
    ).reset_index(drop=True)


## 11. Run relative log-MSE evaluations for all model families


In [ ]:
model_dirs_baseline = resolve_model_dirs(
    [
        "llama2-baseline",
        "llama3-baseline",
        "llama70b-baseline",
        "kimi-baseline",
        "qwen3-baseline",
        "mistral-baseline",
        "oss20b-baseline",
        "oss120b-baseline",
    ],
    label="baseline",
)

(
    per_sim_all_baseline,
    per_model_role_baseline,
    leaderboard_baseline,
) = evaluate_many_models_vs_human_with_log_mse(
    model_dirs=model_dirs_baseline,
    human_param_df=human_df_sims,
    simulate_fn=simulate_beer_game,
    sim_indices=range(11),
    series="order",
    warmup_weeks=4,
    human_baseline="loho",
    rank_by="Relative_log_MSE",
    rank_ascending=False,
)

display(leaderboard_baseline.round(4))


In [ ]:
model_dirs_prompt = resolve_model_dirs(
    [
        "llama2-prompt",
        "llama3-prompt",
        "llama70b-prompt",
        "kimi-prompt",
        "qwen3-prompt",
        "mistral-prompt",
        "oss20b-prompt",
        "oss120b-prompt",
    ],
    label="heuristic-aware",
)

(
    per_sim_all_prompt,
    per_model_role_prompt,
    leaderboard_prompt,
) = evaluate_many_models_vs_human_with_log_mse(
    model_dirs=model_dirs_prompt,
    human_param_df=human_df_sims,
    simulate_fn=simulate_beer_game,
    sim_indices=range(11),
    series="order",
    warmup_weeks=4,
    human_baseline="loho",
    rank_by="Relative_log_MSE",
    rank_ascending=False,
)

display(leaderboard_prompt.round(4))


In [ ]:
model_dirs_alt = resolve_model_dirs(
    [
        "llama2-alt",
        "llama3-alt",
        "llama70b-alt",
        "kimi-alt",
        "qwen3-alt",
        "mistral-alt",
        "oss20b-alt",
        "oss120b-alt",
    ],
    label="cognitive-load",
)

(
    per_sim_all_alt,
    per_model_role_alt,
    leaderboard_alt,
) = evaluate_many_models_vs_human_with_log_mse(
    model_dirs=model_dirs_alt,
    human_param_df=human_df_sims,
    simulate_fn=simulate_beer_game,
    sim_indices=range(11),
    series="order",
    warmup_weeks=4,
    human_baseline="loho",
    rank_by="Relative_log_MSE",
    rank_ascending=False,
)

display(leaderboard_alt.round(4))


In [ ]:
per_sim_all = pd.concat(
    [
        per_sim_all_baseline,
        per_sim_all_prompt,
        per_sim_all_alt,
    ],
    axis=0,
).sort_index()

res_one_sided_baseline = tests_model_vs_loho_parity(
    per_sim_all_baseline,
    metric="Relative_log_MSE",
    alternative="two-sided",
    correction="fdr_bh",
)
res_one_sided_prompt = tests_model_vs_loho_parity(
    per_sim_all_prompt,
    metric="Relative_log_MSE",
    alternative="two-sided",
    correction="fdr_bh",
)
res_one_sided_alt = tests_model_vs_loho_parity(
    per_sim_all_alt,
    metric="Relative_log_MSE",
    alternative="two-sided",
    correction="fdr_bh",
)

display(res_one_sided_baseline.round(4))


## 12. Grouped bar plot — model vs human baseline

Originally the script defined three near-identical versions of this function
(one per scenario). Here it's a single function with `savepath` and `legend_loc`
as parameters, called three times.


In [ ]:
@nature_style
def plot_model_vs_human_grouped_from_results(
    res_df: pd.DataFrame,
    *,
    per_sim: pd.DataFrame,
    models: Optional[List[str]] = None,
    categories: Sequence[str] = ("OVERALL", "retailer", "wholesaler", "distributor", "factory"),
    value_col:    str = "diff_base_minus_model",
    metric:       str = "Relative_log_MSE",
    star_source:  str = "one-sided",           # 'one-sided' → 'sig'; 'two-sided' → 'sig_2s'
    ylabel:       str = "Relative log-MSE\n(positive = lower MSE than LOHO)",
    model_name_map: Optional[Dict[str, str]] = None,
    savepath:   Optional[str] = None,
    width:      float = FULL_WIDTH,
    height:     float = 70,
    legend_loc: str = "lower left",
):
    """Grouped bars (mean ± s.e.m. across sessions) with one dot per session and significance stars.

    `res_df` supplies the model order and the stars; `per_sim` supplies the per-session scores.
    """
    assert star_source in {"one-sided", "two-sided"}
    star_col = "sig" if star_source == "one-sided" else "sig_2s"
    model_name_map = model_name_map or {}

    # Filter + ordering
    df = res_df.copy()
    df = df[df["category"].isin(categories)].copy()
    df["category"] = pd.Categorical(df["category"], categories=categories, ordered=True)
    if models is None:
        overall = (df[df["category"] == "OVERALL"]
                   .sort_values(value_col, ascending=False)["model"])
        models = overall.drop_duplicates().tolist()
    df["model"] = pd.Categorical(df["model"], categories=models, ordered=True)
    df = df.sort_values(["model", "category"])

    stars = {(row["model"], row["category"]): (row[star_col] or "") for _, row in df.iterrows()}

    fig = role_grouped_figure(
        [(model_name_map.get(m, m), m) for m in models],
        lambda m, cat, _: session_values(per_sim, m, cat, metric),
        conditions=[("model", False)],
        star_fn=lambda m, cat, _: stars.get((m, cat), ""),
        ylabel=ylabel,
        width=width,
        height=height,
        legend_loc=legend_loc,
    )
    save_figure(fig, savepath)
    plt.show()
    return df


# Category renaming, shared across all 3 calls
CAT_MAP = {"OVERALL": "Overall", "retailer": "Retailer",
           "wholesaler": "Wholesaler", "distributor": "Distributor", "factory": "Factory"}

In [ ]:
model_map_baseline = {
    "llama2-baseline":   "Llama-2 7B",
    "llama3-baseline":   "Llama-3.1 8B",
    "llama70b-baseline": "Llama-3.3 70B",
    "kimi-baseline":     "Kimi K2",
    "qwen3-baseline":    "Qwen-3.6-35B-A3B",
    "mistral-baseline":  "Mistral Small",
    "oss20b-baseline":   "GPT-OSS 20B",
    "oss120b-baseline":  "GPT-OSS 120B",
}

_ = plot_model_vs_human_grouped_from_results(
    res_one_sided_baseline,
    per_sim=per_sim_all,
    models=None,
    star_source="two-sided",
    ylabel="Relative log-MSE\n(positive = lower MSE than LOHO)",
    model_name_map=model_map_baseline,
    savepath=f"{PLOTS_DIR}/baseline_vs_human.pdf",
)

In [ ]:
model_map_prompt = {
    "llama2-prompt":   "Llama-2 7B",
    "llama3-prompt":   "Llama-3.1 8B",
    "llama70b-prompt": "Llama-3.3 70B",
    "kimi-prompt":     "Kimi K2",
    "qwen3-prompt":    "Qwen-3.6-35B-A3B",
    "mistral-prompt":  "Mistral Small",
    "oss20b-prompt":   "GPT-OSS 20B",
    "oss120b-prompt":  "GPT-OSS 120B",
}

_ = plot_model_vs_human_grouped_from_results(
    res_one_sided_prompt,
    per_sim=per_sim_all,
    models=None,
    star_source="two-sided",
    ylabel="Relative log-MSE\n(positive = lower MSE than LOHO)",
    model_name_map=model_map_prompt,
    savepath=f"{PLOTS_DIR}/prompt_vs_human.pdf",
)

In [ ]:
model_map_alt = {
    "llama2-alt":   "Llama-2 7B",
    "llama3-alt":   "Llama-3.1 8B",
    "llama70b-alt": "Llama-3.3 70B",
    "kimi-alt":     "Kimi K2",
    "qwen3-alt":    "Qwen-3.6-35B-A3B",
    "mistral-alt":  "Mistral Small",
    "oss20b-alt":   "GPT-OSS 20B",
    "oss120b-alt":  "GPT-OSS 120B",
}

_ = plot_model_vs_human_grouped_from_results(
    res_one_sided_alt,
    per_sim=per_sim_all,
    models=None,
    star_source="two-sided",
    ylabel="Relative log-MSE\n(positive = lower MSE than LOHO)",
    model_name_map=model_map_alt,
    savepath=f"{PLOTS_DIR}/alt_vs_human.pdf",
)

## 13. Baseline vs Prompt (& Baseline vs Alt) — independent tests + plots


In [ ]:
def independent_tests_baseline_vs_prompt(
    per_sim_all: pd.DataFrame,
    pairs: List[Tuple[str, str, str]],          # [(family, baseline_model, prompt_model), ...]
    *,
    metric: str = "Relative_log_MSE",
    categories = ("OVERALL", "retailer", "wholesaler", "distributor", "factory"),
    correction: str = "fdr_bh",
    alternative: str = "two-sided",
    min_n_per_group: int = 3,
) -> pd.DataFrame:
    """Treat baseline/prompt as independent samples; Welch + Mann–Whitney per (family, category)."""
    if not isinstance(per_sim_all.index, pd.MultiIndex):
        raise ValueError("per_sim_all must be indexed by (model, sim, role).")

    df = per_sim_all.copy()
    df.index = df.index.set_names(["model", "sim", "role"])
    df = df.reset_index()

    rows = []
    for fam, base_model, prompt_model in pairs:
        for cat in categories:
            if cat == "OVERALL":
                base_vals = (df[df["model"] == base_model]
                             .groupby(["model", "sim"], as_index=False)[metric].mean()
                             [metric].to_numpy(float))
                prompt_vals = (df[df["model"] == prompt_model]
                               .groupby(["model", "sim"], as_index=False)[metric].mean()
                               [metric].to_numpy(float))
            else:
                base_vals = (df[(df["model"] == base_model) & (df["role"] == cat)]
                             .groupby(["model", "sim"], as_index=False)[metric].mean()
                             [metric].to_numpy(float))
                prompt_vals = (df[(df["model"] == prompt_model) & (df["role"] == cat)]
                               .groupby(["model", "sim"], as_index=False)[metric].mean()
                               [metric].to_numpy(float))

            x = base_vals[np.isfinite(base_vals)]
            y = prompt_vals[np.isfinite(prompt_vals)]
            n1, n2 = int(x.size), int(y.size)
            m1 = float(np.mean(x)) if n1 else np.nan
            m2 = float(np.mean(y)) if n2 else np.nan
            s1 = float(np.std(x, ddof=1)) if n1 > 1 else np.nan
            s2 = float(np.std(y, ddof=1)) if n2 > 1 else np.nan
            mean_diff = (m2 - m1) if (np.isfinite(m1) and np.isfinite(m2)) else np.nan

            if n1 >= min_n_per_group and n2 >= min_n_per_group and np.isfinite(m1) and np.isfinite(m2):
                t_stat, p_welch = ttest_ind(y, x, equal_var=False, alternative=alternative)
                df_w, ci_lo, ci_hi = _welch_ci(mean_diff,
                                                s1 if np.isfinite(s1) else 0.0, n1,
                                                s2 if np.isfinite(s2) else 0.0, n2, alpha=0.05)
            else:
                t_stat = p_welch = df_w = ci_lo = ci_hi = np.nan

            if n1 >= 1 and n2 >= 1:
                try:    u_stat, p_mw = mannwhitneyu(y, x, alternative=alternative)
                except Exception: u_stat, p_mw = np.nan, np.nan
            else:
                u_stat, p_mw = np.nan, np.nan

            g = _hedges_g(m1, m2,
                          s1 if np.isfinite(s1) else 0.0,
                          s2 if np.isfinite(s2) else 0.0, n1, n2)
            rows.append({
                "family": fam, "category": cat,
                "n_base": n1, "n_prompt": n2,
                "baseline_mean": m1, "prompt_mean": m2, "mean_diff": mean_diff,
                "t_welch": float(t_stat) if np.isfinite(t_stat) else np.nan,
                "df_welch": float(df_w)  if np.isfinite(df_w)  else np.nan,
                "p_welch": float(p_welch) if np.isfinite(p_welch) else np.nan,
                "U_mw": float(u_stat) if np.isfinite(u_stat) else np.nan,
                "p_mw": float(p_mw) if np.isfinite(p_mw) else np.nan,
                "ci95_low": ci_lo, "ci95_high": ci_hi,
                "hedges_g": g,
            })

    out = pd.DataFrame(rows)
    _apply_correction(out, "p_welch", "p_welch_corr", correction)
    out["sig"] = out["p_welch_corr"].apply(_stars)
    return out.sort_values(["category", "family"]).reset_index(drop=True)


# Pairs for baseline-vs-prompt — all 8 families
pairs_baseline_prompt = [
    ("Llama-2 7B",    "llama2-baseline",   "llama2-prompt"),
    ("Llama-3.1 8B",  "llama3-baseline",   "llama3-prompt"),
    ("Llama-3.3 70B", "llama70b-baseline", "llama70b-prompt"),
    ("Kimi K2",       "kimi-baseline",     "kimi-prompt"),
    ("Qwen-3.6-35B-A3B",   "qwen3-baseline",    "qwen3-prompt"),
    ("Mistral Small", "mistral-baseline",  "mistral-prompt"),
    ("GPT-OSS 20B",   "oss20b-baseline",   "oss20b-prompt"),
    ("GPT-OSS 120B",  "oss120b-baseline",  "oss120b-prompt"),
]

pairs_baseline_prompt = filter_pairs_to_available_models(
    pairs_baseline_prompt,
    per_sim_all,
)

indep_results = independent_tests_baseline_vs_prompt(
    per_sim_all, pairs_baseline_prompt,
    metric="Relative_log_MSE", correction="fdr_bh",
    alternative="greater", min_n_per_group=3,
)
display(indep_results[[
    "family", "category", "n_base", "n_prompt", "baseline_mean", "prompt_mean",
    "mean_diff", "t_welch", "df_welch", "p_welch", "p_welch_corr", "sig",
    "ci95_low", "ci95_high", "hedges_g",
]].round(4))


In [ ]:
def _welch_df_and_sem(s1, n1, s2, n2):
    """Welch df + standard error of (mean2 − mean1)."""
    n1 = max(int(n1), 1); n2 = max(int(n2), 1)
    s1, s2 = float(s1), float(s2)
    se2 = (s1 ** 2) / n1 + (s2 ** 2) / n2
    se  = np.sqrt(se2)
    num = se2 ** 2
    den = ((s1 ** 2 / n1) ** 2 / (n1 - 1 if n1 > 1 else 1)) \
        + ((s2 ** 2 / n2) ** 2 / (n2 - 1 if n2 > 1 else 1))
    df = num / den if den > 0 else np.nan
    return df, se


def _collect_samples(per_sim_all: pd.DataFrame, model: str, category: str, metric: str) -> np.ndarray:
    """Per-sim sample of `metric` for (model, category). OVERALL = mean across roles within sim."""
    df = per_sim_all.copy()
    df.index = df.index.set_names(["model", "sim", "role"])
    dd = df.reset_index()
    if category == "OVERALL":
        s = (dd[dd["model"] == model]
             .groupby(["model", "sim"], as_index=False)[metric].mean()[metric].to_numpy(float))
    else:
        s = (dd[(dd["model"] == model) & (dd["role"] == category)]
             .groupby(["model", "sim"], as_index=False)[metric].mean()[metric].to_numpy(float))
    return s[np.isfinite(s)]


@nature_style
def plot_baseline_vs_prompt_grouped_flexible(
    per_sim_all: pd.DataFrame,
    pairs: List[Tuple[str, str, str]],         # [(family, baseline_model, prompt_model), ...]
    *,
    metric: str = "Relative_log_MSE",
    categories = ("OVERALL", "retailer", "wholesaler", "distributor", "factory"),
    family_order: Optional[List[str]] = None,
    alternative: str = "two-sided",
    correction: str = "fdr_bh",
    err: str = "ci",                            # 'ci' | 'sem'
    conf_level: float = 0.95,
    ylabel: str = "Relative log-MSE improvement\n(heuristic-aware minus baseline)",
    dot_note: str = "Heuristic-aware session minus baseline mean",
    savepath: Optional[str] = None,
):
    """Grouped-bar comparison Prompt − Baseline (plotted value) with FDR-corrected stars.

    Bars show the difference in means with its Welch standard error (err='sem') or CI (err='ci');
    dots show each prompt session minus the baseline mean.
    """
    assert alternative in {"two-sided", "greater", "less"}
    assert isinstance(per_sim_all.index, pd.MultiIndex)

    per_sim_all = per_sim_all.copy()
    per_sim_all.index = per_sim_all.index.set_names(["model", "sim", "role"])

    family_order = family_order or [fam for fam, _, _ in pairs]
    fams = family_order
    cats = list(categories)
    G, C = len(fams), len(cats)
    alpha = 1.0 - conf_level

    rows = []
    for fam, base_model, prompt_model in pairs:
        for cat in cats:
            xb = _collect_samples(per_sim_all, base_model,   cat, metric)
            yp = _collect_samples(per_sim_all, prompt_model, cat, metric)
            n1, n2 = int(xb.size), int(yp.size)
            m1 = float(np.mean(xb)) if n1 else np.nan
            m2 = float(np.mean(yp)) if n2 else np.nan
            s1 = float(np.std(xb, ddof=1)) if n1 > 1 else np.nan
            s2 = float(np.std(yp, ddof=1)) if n2 > 1 else np.nan
            mean_diff = (m2 - m1) if (np.isfinite(m1) and np.isfinite(m2)) else np.nan

            if n1 >= 3 and n2 >= 3 and np.isfinite(m1) and np.isfinite(m2):
                t_stat, p_raw = ttest_ind(yp, xb, equal_var=False, alternative=alternative)
                df_w, se = _welch_df_and_sem(s1 if np.isfinite(s1) else 0.0, n1,
                                              s2 if np.isfinite(s2) else 0.0, n2)
                if err == "ci":
                    tcrit = float(student_t.ppf(1 - alpha / 2, df_w)) if np.isfinite(df_w) else np.nan
                    halfw = tcrit * se if np.isfinite(tcrit) else np.nan
                else:  # 'sem'
                    halfw = se
            else:
                t_stat = p_raw = np.nan; halfw = np.nan

            rows.append({
                "family": fam, "category": cat,
                "n_base": n1, "n_prompt": n2,
                "baseline_mean": m1, "prompt_mean": m2,
                "mean_diff": mean_diff,
                "half_error": halfw,
                "p_raw": p_raw,
            })

    res = pd.DataFrame(rows)
    _apply_correction(res, "p_raw", "p_corr", correction)
    res["sig"] = res["p_corr"].apply(_stars)

    # Plot
    res["family"]   = pd.Categorical(res["family"],   categories=fams, ordered=True)
    res["category"] = pd.Categorical(res["category"], categories=cats, ordered=True)
    res = res.sort_values(["family", "category"])

    stats = {(row["family"], row["category"]): row for _, row in res.iterrows()}
    models = {fam: (base_model, prompt_model) for fam, base_model, prompt_model in pairs}

    def _differences(fam, cat, _):
        base_model, prompt_model = models[fam]
        xb = _collect_samples(per_sim_all, base_model, cat, metric)
        yp = _collect_samples(per_sim_all, prompt_model, cat, metric)
        return yp - xb.mean()

    fig = role_grouped_figure(
        [(fam, fam) for fam in fams],
        _differences,
        conditions=[("difference", False)],
        center_fn=lambda fam, cat, _: stats[(fam, cat)]["mean_diff"],
        err_fn=lambda fam, cat, _: stats[(fam, cat)]["half_error"],
        star_fn=lambda fam, cat, _: stats[(fam, cat)]["sig"] or "",
        ylabel=ylabel,
        dot_note=dot_note,
        height=70,
        legend_loc="above",   # inside the axes the key covers the stars
    )
    save_figure(fig, savepath)
    plt.show()
    return res

In [ ]:
pairs_bp_pretty = [
    ("Llama-2 7B", "llama2-baseline", "llama2-prompt"),
    ("Llama-3.1 8B", "llama3-baseline", "llama3-prompt"),
    ("Llama-3.3 70B", "llama70b-baseline", "llama70b-prompt"),
    ("Kimi K2", "kimi-baseline", "kimi-prompt"),
    ("Qwen-3.6-35B-A3B", "qwen3-baseline", "qwen3-prompt"),
    ("Mistral Small", "mistral-baseline", "mistral-prompt"),
    ("GPT-OSS 20B", "oss20b-baseline", "oss20b-prompt"),
    ("GPT-OSS 120B", "oss120b-baseline", "oss120b-prompt"),
]
pairs_bp_pretty = filter_pairs_to_available_models(
    pairs_bp_pretty,
    per_sim_all,
)

_ = plot_baseline_vs_prompt_grouped_flexible(
    per_sim_all,
    pairs_bp_pretty,
    metric="Relative_log_MSE",
    alternative="greater",
    correction="fdr_bh",
    err="sem",
    savepath=f"{PLOTS_DIR}/prompt_vs_baseline.pdf",
)

In [ ]:
pairs_bv_alt = [
    ("Llama-2 7B", "llama2-baseline", "llama2-alt"),
    ("Llama-3.1 8B", "llama3-baseline", "llama3-alt"),
    ("Llama-3.3 70B", "llama70b-baseline", "llama70b-alt"),
    ("Kimi K2", "kimi-baseline", "kimi-alt"),
    ("Qwen-3.6-35B-A3B", "qwen3-baseline", "qwen3-alt"),
    ("Mistral Small", "mistral-baseline", "mistral-alt"),
    ("GPT-OSS 20B", "oss20b-baseline", "oss20b-alt"),
    ("GPT-OSS 120B", "oss120b-baseline", "oss120b-alt"),
]
pairs_bv_alt = filter_pairs_to_available_models(
    pairs_bv_alt,
    per_sim_all,
)

_ = plot_baseline_vs_prompt_grouped_flexible(
    per_sim_all,
    pairs_bv_alt,
    metric="Relative_log_MSE",
    alternative="greater",
    correction="fdr_bh",
    err="sem",
    ylabel="Relative log-MSE improvement\n(cognitive-load minus baseline)",
    dot_note="Cognitive-load session minus baseline mean",
    savepath=f"{PLOTS_DIR}/alt_vs_baseline.pdf",
)

## 14. Main paper figure — relative log-MSE overlay


In [ ]:
@nature_style
def plot_overlay_relative_log_mse_for_model_pairs(
    per_sim_all: pd.DataFrame,
    pairs: List[Tuple[str, str, str]],
    *,
    metric: str = "Relative_log_MSE",
    categories: Sequence[str] = ("OVERALL","retailer","wholesaler","distributor","factory"),
    family_order: Optional[List[str]] = None,

    test_reference: str = "baseline",
    alternative: str = "two-sided",
    correction: str = "fdr_bh",

    err_mode: str = "diff",
    err_type: str = "sem",
    conf_level: float = 0.95,

    ylabel: str = "Relative log-MSE\n(positive = lower MSE than LOHO)",
    condition_labels: Tuple[str, str] = ("Baseline", "Heuristic-aware"),
    savepath: Optional[str] = None,
):
    """
    Bars show group means of `metric` for each (family, role), side by side:
      - Baseline bar = mean of baseline model (lighter, hatched)
      - Prompt   bar = mean of prompt model (solid)
    Dots show the individual sessions.

    Stars reflect corrected p-values for:
      * test_reference='baseline' : Welch t-test on Prompt vs Baseline
      * test_reference='human'    : one-sample t-test of Prompt Δ vs 0
    """
    assert isinstance(per_sim_all.index, pd.MultiIndex), "per_sim_all must be indexed by (model, sim, role)"
    assert test_reference in {"baseline","human"}
    assert alternative in {"two-sided","greater","less"}
    assert correction in {"fdr_bh","bonferroni","none"}
    assert err_mode in {"diff","both","prompt","baseline","none"}
    assert err_type in {"sem","ci"}

    def _collect(model: str, cat: str) -> np.ndarray:
        df = per_sim_all.copy()
        df.index = df.index.set_names(["model","sim","role"])
        dd = df.reset_index()
        if cat == "OVERALL":
            s = (dd[dd["model"] == model]
                 .groupby(["model","sim"], as_index=False)[metric]
                 .mean()[metric].to_numpy(float))
        else:
            s = (dd[(dd["model"] == model) & (dd["role"] == cat)]
                 .groupby(["model","sim"], as_index=False)[metric]
                 .mean()[metric].to_numpy(float))
        s = s[np.isfinite(s)]
        return s

    def _bh_fdr(pvals):
        p = np.asarray(pvals, float)
        m = p.size
        order = np.argsort(p)
        ranked = p[order]
        q = ranked * m / np.arange(1, m+1, dtype=float)
        q = np.minimum.accumulate(q[::-1])[::-1]
        out = np.empty_like(q)
        out[order] = np.clip(q, 0, 1)
        return out

    def _stars(p):
        if not np.isfinite(p): return ""
        return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 5e-2 else ""

    if family_order is None:
        family_order = [fam for fam,_,_ in pairs]
    fams = list(family_order)
    cats = list(categories)
    G, C = len(fams), len(cats)

    alpha = 1.0 - conf_level
    rows = []

    for fam, base_m, prompt_m in pairs:
        for cat in cats:
            xb = _collect(base_m, cat)
            yp = _collect(prompt_m, cat)
            n_b, n_p = len(xb), len(yp)
            m_b = float(np.mean(xb)) if n_b else np.nan
            m_p = float(np.mean(yp)) if n_p else np.nan
            s_b = float(np.std(xb, ddof=1)) if n_b > 1 else np.nan
            s_p = float(np.std(yp, ddof=1)) if n_p > 1 else np.nan

            if test_reference == "baseline":
                if n_b >= 2 and n_p >= 2 and np.isfinite(m_b) and np.isfinite(m_p):
                    t_stat, p_raw = ttest_ind(yp, xb, equal_var=False, alternative=alternative)
                else:
                    p_raw = np.nan
            else:
                if n_p >= 2 and np.isfinite(m_p):
                    t_stat, p_raw = ttest_1samp(yp, popmean=0.0, alternative=alternative)
                else:
                    p_raw = np.nan

            half_base = half_prompt = half_diff = np.nan
            if err_mode in {"both","baseline"} and n_b > 1 and np.isfinite(s_b):
                sem_b = s_b / np.sqrt(n_b)
                if err_type == "sem":
                    half_base = sem_b
                else:
                    tcrit_b = float(student_t.ppf(1 - alpha/2, n_b-1))
                    half_base = sem_b * tcrit_b
            if err_mode in {"both","prompt"} and n_p > 1 and np.isfinite(s_p):
                sem_p = s_p / np.sqrt(n_p)
                if err_type == "sem":
                    half_prompt = sem_p
                else:
                    tcrit_p = float(student_t.ppf(1 - alpha/2, n_p-1))
                    half_prompt = sem_p * tcrit_p
            if err_mode == "diff":
                if test_reference == "baseline":
                    if n_b >= 2 and n_p >= 2 and np.isfinite(s_b) and np.isfinite(s_p):
                        se2 = (s_p**2)/n_p + (s_b**2)/n_b
                        se = np.sqrt(se2)
                        if err_type == "sem":
                            half_diff = se
                        else:
                            num = se2**2
                            den = ((s_p**2/n_p)**2/(n_p-1)) + ((s_b**2/n_b)**2/(n_b-1))
                            df = num/den if den > 0 else np.nan
                            tcrit = float(student_t.ppf(1 - alpha/2, df)) if np.isfinite(df) else 1.96
                            half_diff = tcrit * se
                else:
                    if n_p >= 2 and np.isfinite(s_p):
                        sem_p = s_p / np.sqrt(n_p)
                        if err_type == "sem":
                            half_diff = sem_p
                        else:
                            tcrit_p = float(student_t.ppf(1 - alpha/2, n_p-1))
                            half_diff = sem_p * tcrit_p

            rows.append({
                "family": fam, "category": cat,
                "mean_base": m_b, "mean_prompt": m_p,
                "half_base": half_base,
                "half_prompt": half_prompt,
                "half_diff": half_diff,
                "p_raw": p_raw
            })

    stat = pd.DataFrame(rows)
    stat["family"]   = pd.Categorical(stat["family"], categories=fams, ordered=True)
    stat["category"] = pd.Categorical(stat["category"], categories=cats, ordered=True)
    stat = stat.sort_values(["family","category"]).reset_index(drop=True)

    if correction == "bonferroni":
        m = int(stat["p_raw"].notna().sum())
        stat["p_corr"] = np.minimum(stat["p_raw"] * m, 1.0)
    elif correction == "fdr_bh":
        mask = stat["p_raw"].notna().to_numpy()
        q = np.full(len(stat), np.nan)
        if mask.any():
            q[mask] = _bh_fdr(stat.loc[mask, "p_raw"].to_list())
        stat["p_corr"] = q
    else:
        stat["p_corr"] = stat["p_raw"]
    stat["sig"] = stat["p_corr"].apply(_stars)

    models = {fam: (base_m, prompt_m) for fam, base_m, prompt_m in pairs}
    rows_by_key = {(row["family"], row["category"]): row for _, row in stat.iterrows()}

    def _values(fam, cat, cond):
        return _collect(models[fam][0] if cond == "base" else models[fam][1], cat)

    def _error(fam, cat, cond):
        row = rows_by_key[(fam, cat)]
        if cond == "base":
            return row["half_base"]
        return row["half_diff"] if err_mode == "diff" else row["half_prompt"]

    fig = role_grouped_figure(
        [(fam, fam) for fam in fams],
        _values,
        conditions=[("base", True), ("prompt", False)],
        center_fn=lambda fam, cat, cond: rows_by_key[(fam, cat)]["mean_base" if cond == "base" else "mean_prompt"],
        err_fn=_error,
        star_fn=lambda fam, cat, cond: "" if cond == "base" else (rows_by_key[(fam, cat)]["sig"] or ""),
        condition_labels=condition_labels,
        ylabel=ylabel,
        star_position="pair" if test_reference == "baseline" else "bar",
    )
    save_figure(fig, savepath)
    plt.show()
    return stat


# ── Main figure pairs — all 8 families ───────────────────────────
pairs = [
    ("Llama-2 7B",          "llama2-baseline",   "llama2-prompt"),
    ("Llama-3.1 8B",        "llama3-baseline",   "llama3-prompt"),
    ("Llama-3.3 70B",       "llama70b-baseline", "llama70b-prompt"),
    ("Kimi K2",             "kimi-baseline",     "kimi-prompt"),
    ("Qwen-3.6-35B-A3B",    "qwen3-baseline",    "qwen3-prompt"),
    ("Mistral Small",       "mistral-baseline",  "mistral-prompt"),
    ("GPT-OSS 20B",         "oss20b-baseline",   "oss20b-prompt"),
    ("GPT-OSS 120B",        "oss120b-baseline",  "oss120b-prompt"),
]

pairs = filter_pairs_to_available_models(pairs, per_sim_all)

# ── Order families by the prompt model's OVERALL relative log-MSE (best first) ──
_df_order = per_sim_all.copy()
_df_order.index = _df_order.index.set_names(["model", "sim", "role"])
_df_order = _df_order.reset_index()

fam_overall_prompt = {}
for fam, _base_m, prompt_m in pairs:
    vals = (_df_order[_df_order["model"] == prompt_m]
            .groupby("sim")["Relative_log_MSE"]
            .mean()
            .to_numpy(float))
    vals = vals[np.isfinite(vals)]
    fam_overall_prompt[fam] = float(np.mean(vals)) if len(vals) else -np.inf

family_order = sorted(
    [f for f, _, _ in pairs],
    key=lambda f: fam_overall_prompt[f],
    reverse=True,          # highest relative log-MSE on the left; set False to flip
)

_ = plot_overlay_relative_log_mse_for_model_pairs(
    per_sim_all, pairs,
    family_order=family_order,          # <-- ordered by prompt overall relative log-MSE
    metric="Relative_log_MSE",
    test_reference="baseline",          # heuristic-aware vs baseline (one-sided Welch t-test)
    alternative="greater",
    correction="fdr_bh",
    err_mode="prompt", err_type="sem",  # s.e.m. of the heuristic-aware sessions
    ylabel="Relative log-MSE\n(positive = lower MSE than LOHO)",
    savepath=f"{PLOTS_DIR}/main.pdf",
)

## 15. Final paper figure — external Table 7 relative log-MSE comparison


In [ ]:
# Uses T7_ROUNDS=29 and WARMUP=4 from the Table 7 section.
all_model_dirs = resolve_model_dirs(
    [
        "llama2-baseline", "llama2-prompt",
        "llama3-baseline", "llama3-prompt",
        "llama70b-baseline", "llama70b-prompt",
        "kimi-baseline", "kimi-prompt",
        "qwen3-baseline", "qwen3-prompt",
        "mistral-baseline", "mistral-prompt",
        "oss20b-baseline", "oss20b-prompt",
        "oss120b-baseline", "oss120b-prompt",
    ],
    label="Table-7",
)
sim_indices_t7 = range(11)


def _mse(
    observed: np.ndarray,
    predicted: np.ndarray,
    *,
    eps: float = 1e-12,
) -> float:
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    n = min(len(observed), len(predicted))
    observed = observed[:n]
    predicted = predicted[:n]
    mask = np.isfinite(observed) & np.isfinite(predicted)
    if not np.any(mask):
        return np.nan
    return float(
        max(
            np.mean((observed[mask] - predicted[mask]) ** 2),
            eps,
        )
    )


def _collect(
    data: pd.DataFrame,
    model: str,
    category: str,
    metric: str,
) -> np.ndarray:
    """Per-session values; OVERALL averages roles within each session."""
    frame = data.reset_index()
    if category == "OVERALL":
        values = (
            frame[frame["model"] == model]
            .groupby(["model", "sim"], as_index=False)[metric]
            .mean()[metric]
            .to_numpy(float)
        )
    else:
        values = (
            frame[
                (frame["model"] == model)
                & (frame["role"] == category)
            ]
            .groupby(["model", "sim"], as_index=False)[metric]
            .mean()[metric]
            .to_numpy(float)
        )
    return values[np.isfinite(values)]


# Cross-group reference MSE by role.
cross_cr_mse: Dict[str, float] = {}
for role in ROLES:
    low = fig1_low_cr[role][WARMUP:T7_ROUNDS]
    high = fig1_high_cr[role][WARMUP:T7_ROUNDS]
    cross_cr_mse[role] = _mse(low, high)

records: List[Dict[str, Any]] = []
for model_name, model_path in all_model_dirs.items():
    for sim_index in sim_indices_t7:
        for role in ROLES:
            try:
                payload = load_simulated_data(
                    model_path,
                    sim_name=f"sim_{sim_index}",
                )
                llm_orders = np.asarray(
                    payload[ROLE_KEYS[role]],
                    dtype=float,
                )
            except Exception:
                continue

            llm_trim = llm_orders[WARMUP:T7_ROUNDS]
            target_mses = []

            for human_data in (fig1_low_cr, fig1_high_cr):
                human = human_data[role][WARMUP:T7_ROUNDS]
                target_mses.append(_mse(human, llm_trim))

            model_mse = float(np.nanmean(target_mses))
            reference_mse = float(cross_cr_mse[role])
            relative_log_mse = float(
                np.log(reference_mse)
                - np.log(model_mse)
            )

            records.append({
                "model": model_name,
                "sim": f"sim_{sim_index}",
                "role": role,
                "MSE_model": model_mse,
                "MSE_cross_group_reference": reference_mse,
                "Relative_log_MSE": relative_log_mse,
            })

per_sim_df_t7 = (
    pd.DataFrame(records)
    .set_index(["model", "sim", "role"])
)
per_sim_df_t7.index = per_sim_df_t7.index.set_names(
    ["model", "sim", "role"]
)

pairs_t7 = [
    ("Llama-2 7B", "llama2-baseline", "llama2-prompt"),
    ("Llama-3.1 8B", "llama3-baseline", "llama3-prompt"),
    ("Llama-3.3 70B", "llama70b-baseline", "llama70b-prompt"),
    ("Kimi K2", "kimi-baseline", "kimi-prompt"),
    ("Qwen-3.6-35B-A3B", "qwen3-baseline", "qwen3-prompt"),
    ("Mistral Small", "mistral-baseline", "mistral-prompt"),
    ("GPT-OSS 20B", "oss20b-baseline", "oss20b-prompt"),
    ("GPT-OSS 120B", "oss120b-baseline", "oss120b-prompt"),
]
pairs_t7 = filter_pairs_to_available_models(
    pairs_t7,
    per_sim_df_t7,
)

metric = "Relative_log_MSE"
categories = [
    "OVERALL",
    "retailer",
    "wholesaler",
    "distributor",
    "factory",
]
families = [family for family, _, _ in pairs_t7]
group_count = len(families)
category_count = len(categories)

rows: List[Dict[str, Any]] = []
for family, baseline_model, prompt_model in pairs_t7:
    for category in categories:
        baseline_values = _collect(
            per_sim_df_t7,
            baseline_model,
            category,
            metric,
        )
        prompt_values = _collect(
            per_sim_df_t7,
            prompt_model,
            category,
            metric,
        )

        baseline_mean = (
            float(np.mean(baseline_values))
            if len(baseline_values)
            else np.nan
        )
        prompt_mean = (
            float(np.mean(prompt_values))
            if len(prompt_values)
            else np.nan
        )
        prompt_sem = (
            float(np.std(prompt_values, ddof=1) / np.sqrt(len(prompt_values)))
            if len(prompt_values) > 1
            else np.nan
        )

        rows.append({
            "family": family,
            "category": category,
            "mean_base": baseline_mean,
            "mean_prompt": prompt_mean,
            "prompt_sem": prompt_sem,
        })

stat = pd.DataFrame(rows)
stat["family"] = pd.Categorical(
    stat["family"],
    categories=families,
    ordered=True,
)
stat["category"] = pd.Categorical(
    stat["category"],
    categories=categories,
    ordered=True,
)
stat = stat.sort_values(["family", "category"])

# Baseline (hatched) and heuristic-aware (solid) bars side by side, dots = sessions,
# s.e.m. error bars on the heuristic-aware bars.
stat_by_key = {(row["family"], row["category"]): row for _, row in stat.iterrows()}
models_t7 = {family: (baseline_model, prompt_model) for family, baseline_model, prompt_model in pairs_t7}


@nature_style
def plot_table7_relative_log_mse(savepath=None):
    fig = role_grouped_figure(
        [(family, family) for family in families],
        lambda family, category, cond: _collect(
            per_sim_df_t7,
            models_t7[family][0 if cond == "base" else 1],
            category,
            metric,
        ),
        conditions=[("base", True), ("prompt", False)],
        center_fn=lambda family, category, cond: stat_by_key[(family, category)][
            "mean_base" if cond == "base" else "mean_prompt"],
        err_fn=lambda family, category, cond: (
            np.nan if cond == "base" else stat_by_key[(family, category)]["prompt_sem"]),
        ylabel="Relative log-MSE vs cross-group\nhuman reference (positive = closer)",
        legend_loc=("lower left", (0.29, 0.0)),   # clear space under Kimi K2 and Qwen; lower left would cover Llama-3.3 70B
    )
    save_figure(fig, savepath)
    plt.show()


plot_table7_relative_log_mse(f"{PLOTS_DIR}/main_table7.pdf")

## 16. Temperature sensitivity analysis (GPT-OSS 20B, heuristic-aware)


In [ ]:
temp_model_dirs = resolve_model_dirs(
    [
        "oss20b-t02-prompt",
        "oss20b-t06-prompt",
        "oss20b-prompt",
    ],
    label="temperature-sensitivity",
)

# Rename folders to stable analysis keys.
temp_model_dirs = {
    {
        "oss20b-t02-prompt": "oss20b-prompt-t02",
        "oss20b-t06-prompt": "oss20b-prompt-t06",
        "oss20b-prompt": "oss20b-prompt-t10",
    }[folder_name]: path
    for folder_name, path in temp_model_dirs.items()
}

temp_name_map = {
    "oss20b-prompt-t02": "GPT-OSS 20B (τ=0.2)",
    "oss20b-prompt-t06": "GPT-OSS 20B (τ=0.6)",
    "oss20b-prompt-t10": "GPT-OSS 20B (τ=1.0)",
}

(
    per_sim_temp,
    per_model_role_temp,
    leaderboard_temp,
) = evaluate_many_models_vs_human_with_log_mse(
    model_dirs=temp_model_dirs,
    human_param_df=human_df_sims,
    simulate_fn=simulate_beer_game,
    sim_indices=range(11),
    series="order",
    warmup_weeks=4,
    human_baseline="loho",
    rank_by="Relative_log_MSE",
    rank_ascending=False,
)

display(
    leaderboard_temp
    .rename(index=temp_name_map)
    .round(4)
)


In [ ]:
res_temp_vs_human = tests_model_vs_loho_parity(
    per_sim_temp,
    metric="Relative_log_MSE",
    correction="fdr_bh",
    alternative="two-sided",
    min_n=3,
)
display(res_temp_vs_human.round(4))

_ = plot_model_vs_human_grouped_from_results(
    res_temp_vs_human,
    per_sim=per_sim_temp,
    models=None,
    star_source="two-sided",
    ylabel="Relative log-MSE\n(positive = lower MSE than LOHO)",
    model_name_map={"oss20b-prompt-t02": "τ = 0.2",
                    "oss20b-prompt-t06": "τ = 0.6",
                    "oss20b-prompt-t10": "τ = 1.0"},
    savepath=f"{PLOTS_DIR}/temp_sensitivity_vs_human.pdf",
    width=HALF_WIDTH * 1.3,
    height=65,
    legend_loc="above",
)

In [ ]:
temp_pairs = [
    ("τ=0.2 → 0.6", "oss20b-prompt-t02", "oss20b-prompt-t06"),
    ("τ=0.6 → 1.0", "oss20b-prompt-t06", "oss20b-prompt-t10"),
    ("τ=0.2 → 1.0", "oss20b-prompt-t02", "oss20b-prompt-t10"),
]
temp_pairs = filter_pairs_to_available_models(
    temp_pairs,
    per_sim_temp,
)

temp_contrasts = independent_tests_baseline_vs_prompt(
    per_sim_temp,
    temp_pairs,
    metric="Relative_log_MSE",
    correction="fdr_bh",
    alternative="two-sided",
    min_n_per_group=3,
)

display(
    temp_contrasts[
        [
            "family",
            "category",
            "n_base",
            "n_prompt",
            "baseline_mean",
            "prompt_mean",
            "mean_diff",
            "t_welch",
            "p_welch",
            "p_welch_corr",
            "sig",
            "hedges_g",
        ]
    ].round(4)
)


In [ ]:
overall_temperature = (
    res_temp_vs_human[
        res_temp_vs_human["category"] == "OVERALL"
    ]
    .copy()
)
overall_temperature["temperature"] = overall_temperature["model"].map(
    {
        "oss20b-prompt-t02": 0.2,
        "oss20b-prompt-t06": 0.6,
        "oss20b-prompt-t10": 1.0,
    }
)
overall_temperature = overall_temperature.sort_values("temperature")

fig, ax = plt.subplots(figsize=(6, 4))
upper_error = (
    overall_temperature["ci95_high"]
    - overall_temperature["diff_base_minus_model"]
)
ax.errorbar(
    overall_temperature["temperature"],
    overall_temperature["diff_base_minus_model"],
    yerr=upper_error,
    marker="o",
    linewidth=2,
    capsize=4,
)
ax.axhline(
    0,
    linewidth=0.8,
    linestyle="--",
    label="LOHO equal-MSE reference",
)
ax.set_xlabel("Sampling temperature τ")
ax.set_ylabel("Relative log-MSE (overall)")
ax.set_title("GPT-OSS 20B heuristic-aware temperature sensitivity")
ax.set_xticks([0.2, 0.6, 1.0])
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(f"{PLOTS_DIR}/temp_sensitivity_overall.pdf")
plt.show()


In [ ]:
# ============================================================================
# GPT-OSS LOW vs HIGH reasoning-effort sensitivity analysis
# Paste this single cell at the END of the latest notebook.
# ============================================================================

import re
from pathlib import Path
from scipy.stats import ttest_rel

# Optional exact path overrides. Leave None for auto-discovery below DATA_ROOT.
# Example: str(DATA_ROOT / "oss20b-baseline-high-reasoning")
HIGH_REASONING_OVERRIDES = {
    ("GPT-OSS 20B", "Baseline"): None,
    ("GPT-OSS 20B", "Heuristic-aware"): None,
    ("GPT-OSS 120B", "Baseline"): None,
    ("GPT-OSS 120B", "Heuristic-aware"): None,
}

CONFIG = {
    "GPT-OSS 20B": {
        "slug": "oss20b",
        "aliases": ("oss20b", "gptoss20b"),
        "low_baseline": "oss20b-baseline",
        "low_prompt": "oss20b-prompt",
    },
    "GPT-OSS 120B": {
        "slug": "oss120b",
        "aliases": ("oss120b", "gptoss120b"),
        "low_baseline": "oss120b-baseline",
        "low_prompt": "oss120b-prompt",
    },
}


def _compact(text):
    return re.sub(r"[^a-z0-9]+", "", str(text).lower())


def _has_sessions(path):
    path = Path(path)
    return path.is_dir() and any(path.glob("sim_*_extracted.json"))


def _find_high_dir(model, condition, override=None):
    if override is not None:
        path = Path(override).expanduser()
        if not path.is_absolute():
            path = DATA_ROOT / path
        if not _has_sessions(path):
            raise FileNotFoundError(f"No extracted sessions found in {path}")
        return path

    aliases = CONFIG[model]["aliases"]
    condition_terms = (
        ("baseline", "raw")
        if condition == "Baseline"
        else ("prompt", "heuristic", "grounded")
    )
    matches = []

    for path in DATA_ROOT.iterdir():
        if not _has_sessions(path):
            continue

        name = path.name.lower()
        compact = _compact(name)

        if not any(_compact(alias) in compact for alias in aliases):
            continue
        if "high" not in compact:
            continue
        if not any(term in name for term in condition_terms):
            continue

        # Prevent a prompted folder from being selected as baseline.
        if condition == "Baseline" and any(
            term in name for term in ("prompt", "heuristic", "grounded")
        ):
            continue

        score = 3 * ("reasoning" in compact) + 2 * ("high" in compact)
        matches.append((score, path))

    if not matches:
        raise FileNotFoundError(
            f"Could not find HIGH-reasoning data for {model}, {condition}. "
            "Set its exact path in HIGH_REASONING_OVERRIDES."
        )

    matches.sort(key=lambda item: (-item[0], item[1].name))
    best_score = matches[0][0]
    best = [path for score, path in matches if score == best_score]

    if len(best) != 1:
        raise RuntimeError(
            f"Ambiguous HIGH-reasoning folders for {model}, {condition}: "
            f"{best}. Set an exact path in HIGH_REASONING_OVERRIDES."
        )

    return best[0]


# ---------------------------------------------------------------------------
# 1. Resolve directories and run the existing relative log-MSE evaluator
# ---------------------------------------------------------------------------

reasoning_dirs = {}
reasoning_meta = {}

for model, cfg in CONFIG.items():
    low_paths = {
        "Baseline": model_dirs_baseline.get(cfg["low_baseline"]),
        "Heuristic-aware": model_dirs_prompt.get(cfg["low_prompt"]),
    }

    for condition, low_path in low_paths.items():
        if low_path is None:
            raise KeyError(
                f"Missing LOW-reasoning main-run folder for "
                f"{model}, {condition}"
            )

        cond_slug = "baseline" if condition == "Baseline" else "prompt"
        low_key = f"{cfg['slug']}-{cond_slug}-low"
        high_key = f"{cfg['slug']}-{cond_slug}-high"

        high_path = _find_high_dir(
            model,
            condition,
            HIGH_REASONING_OVERRIDES[(model, condition)],
        )

        reasoning_dirs[low_key] = str(low_path)
        reasoning_dirs[high_key] = str(high_path)

        reasoning_meta[low_key] = (
            model,
            condition,
            "Low",
        )
        reasoning_meta[high_key] = (
            model,
            condition,
            "High",
        )

print("Resolved reasoning-effort folders:")
for key, value in reasoning_dirs.items():
    print(f"  {key}: {value}")

(
    per_sim_reasoning,
    per_model_role_reasoning,
    leaderboard_reasoning,
) = evaluate_many_models_vs_human_with_log_mse(
    model_dirs=reasoning_dirs,
    human_param_df=human_df_sims,
    simulate_fn=simulate_beer_game,
    sim_indices=range(11),
    series="order",
    warmup_weeks=4,
    human_baseline="loho",
    verbose=False,
    rank_by="Relative_log_MSE",
    rank_ascending=False,
)

meta_df = pd.DataFrame(
    [
        {
            "model": key,
            "model_size": values[0],
            "condition": values[1],
            "reasoning": values[2],
        }
        for key, values in reasoning_meta.items()
    ]
)

role_scores = (
    per_sim_reasoning
    .reset_index()
    .merge(
        meta_df,
        on="model",
        how="left",
        validate="many_to_one",
    )
)

role_scores["category"] = role_scores["role"]

overall_scores = (
    role_scores
    .groupby(
        [
            "model",
            "model_size",
            "condition",
            "reasoning",
            "sim",
        ],
        as_index=False,
    )["Relative_log_MSE"]
    .mean()
)

overall_scores["role"] = "OVERALL"
overall_scores["category"] = "OVERALL"

score_data = pd.concat(
    [
        role_scores[
            [
                "model",
                "model_size",
                "condition",
                "reasoning",
                "sim",
                "role",
                "category",
                "Relative_log_MSE",
            ]
        ],
        overall_scores[
            [
                "model",
                "model_size",
                "condition",
                "reasoning",
                "sim",
                "role",
                "category",
                "Relative_log_MSE",
            ]
        ],
    ],
    ignore_index=True,
)

score_data.to_csv(
    f"{PLOTS_DIR}/reasoning_effort_relative_log_mse_session_role.csv",
    index=False,
)

CATEGORY_ORDER = [
    "OVERALL",
    *ROLES,
]

CATEGORY_LABEL = {
    "OVERALL": "Overall",
    "retailer": "Retailer",
    "wholesaler": "Wholesaler",
    "distributor": "Distributor",
    "factory": "Factory",
}


# ---------------------------------------------------------------------------
# 2. Generic paired contrasts, blocked by matched Sterman session index
# ---------------------------------------------------------------------------

def _bh_local(values):
    values = np.asarray(values, float)
    result = np.full(len(values), np.nan)

    mask = np.isfinite(values)
    if not mask.any():
        return result

    p = values[mask]
    order = np.argsort(p)
    ranked = p[order]

    q = (
        ranked
        * len(ranked)
        / np.arange(1, len(ranked) + 1)
    )
    q = np.minimum.accumulate(q[::-1])[::-1]

    restored = np.empty_like(q)
    restored[order] = np.clip(q, 0, 1)
    result[mask] = restored

    return result


def _star(p):
    if not np.isfinite(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def paired_contrasts(
    data,
    *,
    group_columns,
    contrast_column,
    low_label,
    high_label,
    value_column,
    difference_name,
):
    rows = []

    for group_values, subset in data.groupby(
        group_columns,
        dropna=False,
    ):
        if not isinstance(group_values, tuple):
            group_values = (group_values,)

        paired = subset.pivot_table(
            index="sim",
            columns=contrast_column,
            values=value_column,
            aggfunc="mean",
        )

        if (
            low_label not in paired.columns
            or high_label not in paired.columns
        ):
            continue

        paired = paired.dropna(
            subset=[
                low_label,
                high_label,
            ]
        )

        low = paired[low_label].to_numpy(float)
        high = paired[high_label].to_numpy(float)

        difference = high - low
        n = len(difference)

        if n >= 2:
            t_value, p_t = ttest_rel(
                high,
                low,
                alternative="two-sided",
            )

            sd = float(
                np.std(
                    difference,
                    ddof=1,
                )
            )
            sem = sd / np.sqrt(n)

            half = (
                float(
                    student_t.ppf(
                        0.975,
                        n - 1,
                    )
                )
                * sem
            )

            dz = (
                float(
                    np.mean(difference)
                    / sd
                )
                if sd > 0
                else np.nan
            )

            try:
                if np.any(
                    ~np.isclose(
                        difference,
                        0.0,
                    )
                ):
                    w_value, p_w = wilcoxon(
                        difference,
                        alternative="two-sided",
                    )
                else:
                    w_value, p_w = np.nan, np.nan
            except ValueError:
                w_value, p_w = np.nan, np.nan

        else:
            t_value = np.nan
            p_t = np.nan
            w_value = np.nan
            p_w = np.nan
            half = np.nan
            dz = np.nan

        row = dict(
            zip(
                group_columns,
                group_values,
            )
        )

        row.update(
            {
                "n_pairs": n,
                f"{low_label}_mean": (
                    float(np.mean(low))
                    if n
                    else np.nan
                ),
                f"{high_label}_mean": (
                    float(np.mean(high))
                    if n
                    else np.nan
                ),
                difference_name: (
                    float(np.mean(difference))
                    if n
                    else np.nan
                ),
                "ci95_low": (
                    float(
                        np.mean(difference)
                        - half
                    )
                    if n >= 2
                    else np.nan
                ),
                "ci95_high": (
                    float(
                        np.mean(difference)
                        + half
                    )
                    if n >= 2
                    else np.nan
                ),
                "cohens_dz": dz,
                "paired_t": (
                    float(t_value)
                    if np.isfinite(t_value)
                    else np.nan
                ),
                "p_paired_t": (
                    float(p_t)
                    if np.isfinite(p_t)
                    else np.nan
                ),
                "wilcoxon_W": (
                    float(w_value)
                    if np.isfinite(w_value)
                    else np.nan
                ),
                "p_wilcoxon": (
                    float(p_w)
                    if np.isfinite(p_w)
                    else np.nan
                ),
            }
        )

        rows.append(row)

    output = pd.DataFrame(rows)

    output["p_paired_t_fdr"] = _bh_local(
        output["p_paired_t"]
    )
    output["p_wilcoxon_fdr"] = _bh_local(
        output["p_wilcoxon"]
    )
    output["sig"] = (
        output["p_paired_t_fdr"]
        .apply(_star)
    )

    return output


# HIGH minus LOW within each model, condition, and role.
low_vs_high_tests = paired_contrasts(
    score_data,
    group_columns=[
        "model_size",
        "condition",
        "category",
    ],
    contrast_column="reasoning",
    low_label="Low",
    high_label="High",
    value_column="Relative_log_MSE",
    difference_name="high_minus_low",
)

low_vs_high_tests.to_csv(
    f"{PLOTS_DIR}/reasoning_effort_low_vs_high_tests.csv",
    index=False,
)


# Heuristic-aware minus baseline within LOW and HIGH reasoning.
prompt_effect_tests = paired_contrasts(
    score_data,
    group_columns=[
        "model_size",
        "reasoning",
        "category",
    ],
    contrast_column="condition",
    low_label="Baseline",
    high_label="Heuristic-aware",
    value_column="Relative_log_MSE",
    difference_name="prompt_gain",
)

prompt_effect_tests.to_csv(
    f"{PLOTS_DIR}/reasoning_effort_prompt_effect_tests.csv",
    index=False,
)


# Session-level prompt gains.
prompt_gain_sessions = (
    score_data
    .pivot_table(
        index=[
            "model_size",
            "reasoning",
            "category",
            "sim",
        ],
        columns="condition",
        values="Relative_log_MSE",
        aggfunc="mean",
    )
    .dropna(
        subset=[
            "Baseline",
            "Heuristic-aware",
        ]
    )
    .reset_index()
)

prompt_gain_sessions["prompt_gain"] = (
    prompt_gain_sessions["Heuristic-aware"]
    - prompt_gain_sessions["Baseline"]
)

prompt_gain_sessions.to_csv(
    f"{PLOTS_DIR}/reasoning_effort_prompt_gain_session.csv",
    index=False,
)


# Difference-in-differences:
# (High prompt gain) minus (Low prompt gain).
interaction_tests = paired_contrasts(
    prompt_gain_sessions,
    group_columns=[
        "model_size",
        "category",
    ],
    contrast_column="reasoning",
    low_label="Low",
    high_label="High",
    value_column="prompt_gain",
    difference_name="interaction_high_minus_low",
)

interaction_tests.to_csv(
    f"{PLOTS_DIR}/reasoning_effort_prompt_interaction_tests.csv",
    index=False,
)


# ---------------------------------------------------------------------------
# 3. Summary tables and primary sensitivity figures
# ---------------------------------------------------------------------------

def _sem(series):
    values = np.asarray(
        series,
        float,
    )
    values = values[
        np.isfinite(values)
    ]

    return (
        float(
            np.std(
                values,
                ddof=1,
            )
            / np.sqrt(len(values))
        )
        if len(values) > 1
        else np.nan
    )


score_summary = (
    score_data
    .groupby(
        [
            "model_size",
            "condition",
            "reasoning",
            "category",
        ],
        as_index=False,
    )
    .agg(
        n=(
            "Relative_log_MSE",
            "count",
        ),
        mean=(
            "Relative_log_MSE",
            "mean",
        ),
        sem=(
            "Relative_log_MSE",
            _sem,
        ),
    )
)

score_summary.to_csv(
    f"{PLOTS_DIR}/reasoning_effort_relative_log_mse_summary.csv",
    index=False,
)


# Figures A and B: bars = mean across sessions ± s.e.m., dots = sessions; stars mark
# FDR-corrected paired t-tests (A: low vs high reasoning; B: whether the prompt gain
# changes with reasoning effort).
REASON_COLORS = {"Low": "#1f77b4", "High": "#ff7f0e"}


@nature_style
def plot_reasoning_panels(data, value_col, tests, panels, *, shape, ylabel, height, savepath=None):
    fig, axes = plt.subplots(*shape, figsize=(FULL_WIDTH, height * MM), squeeze=False)
    for ax, (title, filt, test_filt) in zip(axes.ravel(), panels):
        sub = data
        for k, v in filt.items():
            sub = sub[sub[k] == v]
        tsub = tests
        for k, v in test_filt.items():
            tsub = tsub[tsub[k] == v]
        panel_vals = sub.groupby(["reasoning", "category", "sim"])[value_col].mean().to_numpy(float) \
            if value_col == "Relative_log_MSE" else sub[value_col].to_numpy(float)
        set_dot_spacing(panel_vals)
        star_y = np.nanmax(panel_vals) + (np.nanmax(panel_vals) - np.nanmin(panel_vals)) * 0.06
        for ci, cat in enumerate(CATS):
            for j, level in enumerate(["Low", "High"]):
                v = sub[(sub["reasoning"] == level) & (sub["category"] == cat)]
                if value_col == "Relative_log_MSE":
                    v = v.groupby("sim")[value_col].mean()
                v = np.asarray(v if isinstance(v, pd.Series) else v[value_col], float)
                bar_with_dots(ax, ci + (j - 0.5) * 0.38, v, REASON_COLORS[level], 0.36, dot_size=1.6)
            s = tsub[tsub["category"] == cat]["p_paired_t_fdr"]
            s = star(float(s.iloc[0])) if len(s) else ""
            if s:
                ax.text(ci, star_y, s, ha="center", va="bottom", fontsize=6)
        ax.axhline(0, color="black", lw=0.5)
        ax.set_xticks(range(len(CATS)))
        ax.set_xticklabels([ROLE_LABELS[c] for c in CATS])
        ax.set_title(title)
        ax.set_ylabel(ylabel)
    reason_handles = [Patch(facecolor=REASON_COLORS["Low"], edgecolor="black", label="Low reasoning"),
                      Patch(facecolor=REASON_COLORS["High"], edgecolor="black", label="High reasoning"),
                      Line2D([], [], marker="o", ls="", color=DOT_COLOR, alpha=0.75, markeredgewidth=0,
                             markersize=2.5, label="Session ($n=11$)")]
    fig.legend(handles=reason_handles, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3)
    fig.tight_layout(h_pad=1.5, w_pad=1.5)
    save_figure(fig, savepath)
    plt.show()


# Figure A: LOW vs HIGH reasoning for the main score.
plot_reasoning_panels(
    score_data,
    "Relative_log_MSE",
    low_vs_high_tests,
    [(f"{model} — {condition}", {"model_size": model, "condition": condition},
      {"model_size": model, "condition": condition})
     for model in ["GPT-OSS 20B", "GPT-OSS 120B"] for condition in ["Baseline", "Heuristic-aware"]],
    shape=(2, 2),
    ylabel="Relative log-MSE",
    height=110,
    savepath=f"{PLOTS_DIR}/reasoning_effort_relative_log_mse.pdf",
)


# Figure B: prompt gain at LOW and HIGH reasoning.
# Stars test whether the prompt gain changes with reasoning effort.
plot_reasoning_panels(
    prompt_gain_sessions,
    "prompt_gain",
    interaction_tests,
    [(model, {"model_size": model}, {"model_size": model}) for model in ["GPT-OSS 20B", "GPT-OSS 120B"]],
    shape=(1, 2),
    ylabel="Heuristic-aware minus baseline\n(relative log-MSE)",
    height=60,
    savepath=f"{PLOTS_DIR}/reasoning_effort_prompt_gain.pdf",
)


# ---------------------------------------------------------------------------
# 4. Role-wise average trajectories:
#    LOW vs HIGH, baseline vs heuristic-aware
# ---------------------------------------------------------------------------

for model, cfg in CONFIG.items():
    series = [
        {
            "label": "Baseline — low reasoning",
            "model_key": (
                f"{cfg['slug']}-baseline-low"
            ),
            "color": "#1b9e77",
        },
        {
            "label": "Baseline — high reasoning",
            "model_key": (
                f"{cfg['slug']}-baseline-high"
            ),
            "color": "#d95f02",
            "linestyle": "--",
        },
        {
            "label": "Heuristic-aware — low reasoning",
            "model_key": (
                f"{cfg['slug']}-prompt-low"
            ),
            "color": "#7570b3",
        },
        {
            "label": "Heuristic-aware — high reasoning",
            "model_key": (
                f"{cfg['slug']}-prompt-high"
            ),
            "color": "#c51d7c",
            "linestyle": "--",
        },
    ]

    plot_model_orders_vs_human_with_shading(
        model_dirs=reasoning_dirs,
        human_param_df=human_df_sims,
        simulate_fn=simulate_beer_game,
        series=series,
        sim_indices=range(11),
        distribution="gaussian",
        gaussian_sigma=None,
        conf_level=0.95,
        loho_color="#000000",   # the default benchmark colour is used by a series here
        savepath=(
            f"{PLOTS_DIR}/"
            f"reasoning_effort_trajectories_"
            f"{cfg['slug']}.pdf"
        ),
    )


# ---------------------------------------------------------------------------
# 5. Display compact reviewer-facing results
# ---------------------------------------------------------------------------

overall_low_high = low_vs_high_tests[
    low_vs_high_tests["category"]
    == "OVERALL"
]

overall_prompt = prompt_effect_tests[
    prompt_effect_tests["category"]
    == "OVERALL"
]

overall_interaction = interaction_tests[
    interaction_tests["category"]
    == "OVERALL"
]

print(
    "\nOVERALL relative log-MSE: "
    "HIGH vs LOW reasoning"
)
display(
    overall_low_high.round(4)
)

print(
    "\nOVERALL heuristic-aware prompt effect "
    "within each reasoning setting"
)
display(
    overall_prompt.round(4)
)

print(
    "\nOVERALL reasoning × prompt interaction"
)
display(
    overall_interaction.round(4)
)

reviewer_summary = overall_low_high[
    [
        "model_size",
        "condition",
        "n_pairs",
        "Low_mean",
        "High_mean",
        "high_minus_low",
        "ci95_low",
        "ci95_high",
        "p_paired_t_fdr",
        "p_wilcoxon_fdr",
        "cohens_dz",
    ]
]

reviewer_summary.to_csv(
    f"{PLOTS_DIR}/reasoning_effort_reviewer_summary.csv",
    index=False,
)

print(
    "\nSaved all sensitivity outputs to:",
    PLOTS_DIR,
)

In [ ]:
# ── Main-style overlay: Baseline vs Cognitive-load prompt ─────────
pairs_cog = [
    ("Llama-2 7B", "llama2-baseline", "llama2-alt"),
    ("Llama-3.1 8B", "llama3-baseline", "llama3-alt"),
    ("Llama-3.3 70B", "llama70b-baseline", "llama70b-alt"),
    ("Kimi K2", "kimi-baseline", "kimi-alt"),
    ("Qwen-3.6-35B-A3B", "qwen3-baseline", "qwen3-alt"),
    ("Mistral Small", "mistral-baseline", "mistral-alt"),
    ("GPT-OSS 20B", "oss20b-baseline", "oss20b-alt"),
    ("GPT-OSS 120B", "oss120b-baseline", "oss120b-alt"),
]
pairs_cog = filter_pairs_to_available_models(
    pairs_cog,
    per_sim_all,
)

order_frame = per_sim_all.copy()
order_frame.index = order_frame.index.set_names(
    ["model", "sim", "role"]
)
order_frame = order_frame.reset_index()

family_overall_cognitive: Dict[str, float] = {}
for family, _, cognitive_model in pairs_cog:
    values = (
        order_frame[order_frame["model"] == cognitive_model]
        .groupby("sim")["Relative_log_MSE"]
        .mean()
        .to_numpy(float)
    )
    values = values[np.isfinite(values)]
    family_overall_cognitive[family] = (
        float(np.mean(values))
        if len(values)
        else -np.inf
    )

family_order_cognitive = sorted(
    [family for family, _, _ in pairs_cog],
    key=lambda family: family_overall_cognitive[family],
    reverse=True,
)

stat_cognitive = plot_overlay_relative_log_mse_for_model_pairs(
    per_sim_all,
    pairs_cog,
    family_order=family_order_cognitive,
    metric="Relative_log_MSE",
    test_reference="baseline",   # cognitive-load vs baseline (one-sided Welch t-test)
    alternative="greater",
    correction="fdr_bh",
    err_mode="prompt",           # s.e.m. of the cognitive-load sessions
    err_type="sem",
    ylabel="Relative log-MSE\n(positive = lower MSE than LOHO)",
    condition_labels=("Baseline", "Cognitive-load"),
    savepath=f"{PLOTS_DIR}/main_cognitive.pdf",
)

display(stat_cognitive.round(4))

## 17. Temporal behavioral diagnostics

The relative log-MSE score retains round alignment but compresses the entire
error path into one scalar. The analyses below preserve temporal structure by
reporting peak magnitude and timing, later collapse, zero-order runs,
autocorrelation, roughness, early/middle/late RMSE, and individual-session
trajectories.

The descriptive overshoot-and-collapse indicator uses a transparent threshold:
a post-shock peak of at least 10 units followed by a later trough of at most
6 units. Continuous peak, trough, timing, and error measures remain the primary
diagnostics.

In [ ]:
MODEL_PRETTY_NAMES = {
    "llama2": "Llama-2 7B",
    "llama3": "Llama-3.1 8B",
    "llama70b": "Llama-3.3 70B",
    "kimi": "Kimi K2",
    "qwen3": "Qwen-3.6-35B-A3B",
    "mistral": "Mistral Small",
    "oss20b": "GPT-OSS 20B",
    "oss120b": "GPT-OSS 120B",
}
CONDITION_PRETTY_NAMES = {
    "baseline": "Baseline",
    "prompt": "Heuristic-aware",
    "alt": "Cognitive-load",
}


def split_model_key(model_key: str) -> Tuple[str, str]:
    family_key, condition_key = model_key.rsplit("-", 1)
    return family_key, condition_key


def lag1_autocorrelation(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < 3:
        return np.nan
    if np.std(values[:-1]) == 0 or np.std(values[1:]) == 0:
        return np.nan
    return float(np.corrcoef(values[:-1], values[1:])[0, 1])


def longest_true_run(mask: np.ndarray) -> int:
    longest = 0
    current = 0
    for value in np.asarray(mask, dtype=bool):
        if value:
            current += 1
            longest = max(longest, current)
        else:
            current = 0
    return int(longest)


def best_lagged_correlation(
    target: np.ndarray,
    model: np.ndarray,
    *,
    max_lag: int = 5,
) -> Tuple[float, float]:
    """
    Positive lag means the model is delayed relative to the target.
    Negative lag means the model responds earlier.
    """
    target = np.asarray(target, dtype=float)
    model = np.asarray(model, dtype=float)
    n = min(len(target), len(model))
    target = target[:n]
    model = model[:n]

    candidates: List[Tuple[int, float]] = []

    for lag in range(-max_lag, max_lag + 1):
        if lag > 0:
            target_part = target[:-lag]
            model_part = model[lag:]
        elif lag < 0:
            target_part = target[-lag:]
            model_part = model[:lag]
        else:
            target_part = target
            model_part = model

        mask = np.isfinite(target_part) & np.isfinite(model_part)
        target_part = target_part[mask]
        model_part = model_part[mask]

        if target_part.size < 3:
            continue
        if np.std(target_part) == 0 or np.std(model_part) == 0:
            continue

        candidates.append(
            (
                lag,
                float(np.corrcoef(target_part, model_part)[0, 1]),
            )
        )

    if not candidates:
        return np.nan, np.nan

    best_lag, best_correlation = max(
        candidates,
        key=lambda item: item[1],
    )
    return float(best_lag), float(best_correlation)


def extract_order_shape_features(
    sequence: np.ndarray,
    *,
    shock_round: int = 5,
    equilibrium_order: float = 8.0,
    late_start_round: int = 25,
    overshoot_margin: float = 2.0,
    collapse_margin: float = 2.0,
) -> Dict[str, Any]:
    sequence = np.asarray(sequence, dtype=float)
    rounds = np.arange(1, len(sequence) + 1)
    valid = np.isfinite(sequence)
    post_mask = valid & (rounds >= shock_round)

    if not np.any(post_mask):
        return {
            "peak_order": np.nan,
            "peak_round": np.nan,
            "post_peak_min": np.nan,
            "post_peak_min_round": np.nan,
            "overshoot_amount": np.nan,
            "collapse_depth": np.nan,
            "overshoot_collapse": False,
            "postshock_mean": np.nan,
            "postshock_std": np.nan,
            "postshock_acf1": np.nan,
            "roughness": np.nan,
            "zero_fraction_postshock": np.nan,
            "late_zero_fraction": np.nan,
            "first_zero_round": np.nan,
            "longest_zero_run": 0,
            "late_mean_order": np.nan,
        }

    post_indices = np.flatnonzero(post_mask)
    post_values = sequence[post_indices]
    peak_position = int(np.nanargmax(post_values))
    peak_index = int(post_indices[peak_position])
    peak_order = float(sequence[peak_index])
    peak_round = int(rounds[peak_index])

    after_peak_indices = post_indices[post_indices > peak_index]
    if after_peak_indices.size:
        after_peak_values = sequence[after_peak_indices]
        trough_position = int(np.nanargmin(after_peak_values))
        trough_index = int(after_peak_indices[trough_position])
        post_peak_min = float(sequence[trough_index])
        post_peak_min_round = int(rounds[trough_index])
    else:
        post_peak_min = np.nan
        post_peak_min_round = np.nan

    collapse_depth = (
        peak_order - post_peak_min
        if np.isfinite(post_peak_min)
        else np.nan
    )
    overshoot_collapse = bool(
        peak_order >= equilibrium_order + overshoot_margin
        and np.isfinite(post_peak_min)
        and post_peak_min <= equilibrium_order - collapse_margin
    )

    zero_post = np.isclose(post_values, 0.0)
    first_zero_round = (
        int(rounds[post_indices[np.flatnonzero(zero_post)[0]]])
        if np.any(zero_post)
        else np.nan
    )

    late_mask = valid & (rounds >= late_start_round)
    late_values = sequence[late_mask]

    return {
        "peak_order": peak_order,
        "peak_round": peak_round,
        "post_peak_min": post_peak_min,
        "post_peak_min_round": post_peak_min_round,
        "overshoot_amount": peak_order - equilibrium_order,
        "collapse_depth": (
            float(collapse_depth)
            if np.isfinite(collapse_depth)
            else np.nan
        ),
        "overshoot_collapse": overshoot_collapse,
        "postshock_mean": float(np.mean(post_values)),
        "postshock_std": (
            float(np.std(post_values, ddof=1))
            if post_values.size > 1
            else 0.0
        ),
        "postshock_acf1": lag1_autocorrelation(post_values),
        "roughness": (
            float(np.mean(np.abs(np.diff(post_values))))
            if post_values.size > 1
            else 0.0
        ),
        "zero_fraction_postshock": float(np.mean(zero_post)),
        "late_zero_fraction": (
            float(np.mean(np.isclose(late_values, 0.0)))
            if late_values.size
            else np.nan
        ),
        "first_zero_round": first_zero_round,
        "longest_zero_run": longest_true_run(zero_post),
        "late_mean_order": (
            float(np.mean(late_values))
            if late_values.size
            else np.nan
        ),
    }


def compare_order_trajectories_temporally(
    model: np.ndarray,
    target: np.ndarray,
    *,
    shock_round: int = 5,
    max_lag: int = 5,
) -> Dict[str, float]:
    model = np.asarray(model, dtype=float)
    target = np.asarray(target, dtype=float)
    n = min(len(model), len(target))
    model = model[:n]
    target = target[:n]

    rounds = np.arange(1, n + 1)
    valid = np.isfinite(model) & np.isfinite(target)

    def phase_rmse(start_round: int, end_round: Optional[int]) -> float:
        phase = valid & (rounds >= start_round)
        if end_round is not None:
            phase &= rounds <= end_round
        if not np.any(phase):
            return np.nan
        return float(
            np.sqrt(
                np.mean((model[phase] - target[phase]) ** 2)
            )
        )

    post = valid & (rounds >= shock_round)
    if np.any(post):
        residual = model[post] - target[post]
        max_abs_error = float(np.max(np.abs(residual)))
        integrated_abs_error = float(np.sum(np.abs(residual)))
    else:
        max_abs_error = np.nan
        integrated_abs_error = np.nan

    model_post = model[post]
    target_post = target[post]
    first_difference_rmse = (
        float(
            np.sqrt(
                np.mean(
                    (
                        np.diff(model_post)
                        - np.diff(target_post)
                    ) ** 2
                )
            )
        )
        if model_post.size > 1 and target_post.size > 1
        else np.nan
    )

    best_lag, best_correlation = best_lagged_correlation(
        target_post,
        model_post,
        max_lag=max_lag,
    )

    return {
        "rmse_early": phase_rmse(5, 12),
        "rmse_middle": phase_rmse(13, 24),
        "rmse_late": phase_rmse(25, None),
        "max_abs_error_postshock": max_abs_error,
        "integrated_abs_error_postshock": integrated_abs_error,
        "first_difference_rmse": first_difference_rmse,
        "best_lag": best_lag,
        "best_lag_correlation": best_correlation,
    }


def collect_temporal_diagnostics(
    model_dirs: Dict[str, str],
    human_param_df: pd.DataFrame,
    simulate_fn,
    *,
    score_table: pd.DataFrame,
    sim_indices: Iterable[int] = range(11),
    shock_round: int = 5,
    equilibrium_order: float = 8.0,
    late_start_round: int = 25,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for model_key, model_path in model_dirs.items():
        family_key, condition_key = split_model_key(model_key)

        for sim_index in sim_indices:
            sim_name = f"sim_{sim_index}"
            try:
                llm_payload = load_simulated_data(
                    model_path,
                    sim_name=sim_name,
                )
            except Exception as error:
                print(f"Skipping {model_key}/{sim_name}: {error}")
                continue

            weeks = int(len(llm_payload["retailer_order"]))
            target_df = simulate_fn(
                **_human_kwargs_for_sim(
                    human_param_df,
                    sim_index,
                    weeks=weeks,
                )
            )["data"]
            loho_df = simulate_fn(
                **_kwargs_from_param_series(
                    _loho_series(human_param_df, sim_index),
                    weeks=weeks,
                )
            )["data"]

            for role in ROLES:
                key = f"{role}_order"
                model_sequence = np.asarray(
                    llm_payload[key],
                    dtype=float,
                )
                target_sequence = target_df[key].to_numpy(float)
                loho_sequence = loho_df[key].to_numpy(float)

                model_features = extract_order_shape_features(
                    model_sequence,
                    shock_round=shock_round,
                    equilibrium_order=equilibrium_order,
                    late_start_round=late_start_round,
                )
                target_features = extract_order_shape_features(
                    target_sequence,
                    shock_round=shock_round,
                    equilibrium_order=equilibrium_order,
                    late_start_round=late_start_round,
                )
                loho_features = extract_order_shape_features(
                    loho_sequence,
                    shock_round=shock_round,
                    equilibrium_order=equilibrium_order,
                    late_start_round=late_start_round,
                )
                temporal_comparison = compare_order_trajectories_temporally(
                    model_sequence,
                    target_sequence,
                    shock_round=shock_round,
                )

                try:
                    relative_log_mse = float(
                        score_table.loc[
                            (model_key, sim_name, role),
                            "Relative_log_MSE",
                        ]
                    )
                    mse_ratio = float(
                        score_table.loc[
                            (model_key, sim_name, role),
                            "MSE_ratio_to_LOHO",
                        ]
                    )
                except Exception:
                    relative_log_mse = np.nan
                    mse_ratio = np.nan

                row: Dict[str, Any] = {
                    "model_key": model_key,
                    "family_key": family_key,
                    "family": MODEL_PRETTY_NAMES.get(
                        family_key,
                        family_key,
                    ),
                    "condition_key": condition_key,
                    "condition": CONDITION_PRETTY_NAMES.get(
                        condition_key,
                        condition_key,
                    ),
                    "sim": sim_name,
                    "role": role,
                    "Relative_log_MSE": relative_log_mse,
                    "MSE_ratio_to_LOHO": mse_ratio,
                }

                row.update(
                    {
                        f"model_{name}": value
                        for name, value in model_features.items()
                    }
                )
                row.update(
                    {
                        f"target_{name}": value
                        for name, value in target_features.items()
                    }
                )
                row.update(
                    {
                        f"loho_{name}": value
                        for name, value in loho_features.items()
                    }
                )
                row.update(temporal_comparison)

                row["peak_order_error"] = (
                    model_features["peak_order"]
                    - target_features["peak_order"]
                )
                row["peak_round_error"] = (
                    model_features["peak_round"]
                    - target_features["peak_round"]
                )
                row["acf1_error"] = (
                    model_features["postshock_acf1"]
                    - target_features["postshock_acf1"]
                )
                row["volatility_ratio"] = (
                    model_features["postshock_std"]
                    / target_features["postshock_std"]
                    if (
                        np.isfinite(target_features["postshock_std"])
                        and target_features["postshock_std"] > 0
                    )
                    else np.nan
                )
                row["late_zero_fraction_difference"] = (
                    model_features["late_zero_fraction"]
                    - target_features["late_zero_fraction"]
                )
                rows.append(row)

    output = pd.DataFrame(rows)
    if output.empty:
        raise RuntimeError("No temporal diagnostics were generated.")

    output["abs_peak_round_error"] = np.abs(
        output["peak_round_error"]
    )
    output["late_zero_session"] = (
        output["model_late_zero_fraction"] > 0
    )
    return output

In [ ]:
ALL_MAIN_MODEL_DIRS = {
    **model_dirs_baseline,
    **model_dirs_prompt,
    **model_dirs_alt,
}

temporal_diagnostics = collect_temporal_diagnostics(
    ALL_MAIN_MODEL_DIRS,
    human_df_sims,
    simulate_beer_game,
    score_table=per_sim_all,
    sim_indices=range(11),
    shock_round=5,
    equilibrium_order=8.0,
    late_start_round=25,
)

temporal_diagnostics.to_csv(
    f"{PLOTS_DIR}/temporal_diagnostics_session_role.csv",
    index=False,
)

temporal_summary = (
    temporal_diagnostics
    .groupby(
        ["family", "condition", "role"],
        as_index=False,
    )
    .agg(
        n_sessions=("sim", "nunique"),
        overshoot_collapse_rate=(
            "model_overshoot_collapse",
            "mean",
        ),
        late_zero_session_rate=(
            "late_zero_session",
            "mean",
        ),
        mean_late_zero_fraction=(
            "model_late_zero_fraction",
            "mean",
        ),
        median_peak_order=(
            "model_peak_order",
            "median",
        ),
        median_peak_round=(
            "model_peak_round",
            "median",
        ),
        median_collapse_depth=(
            "model_collapse_depth",
            "median",
        ),
        median_postshock_acf1=(
            "model_postshock_acf1",
            "median",
        ),
        median_postshock_std=(
            "model_postshock_std",
            "median",
        ),
        median_abs_peak_round_error=(
            "abs_peak_round_error",
            "median",
        ),
        median_rmse_early=("rmse_early", "median"),
        median_rmse_middle=("rmse_middle", "median"),
        median_rmse_late=("rmse_late", "median"),
        median_first_difference_rmse=(
            "first_difference_rmse",
            "median",
        ),
        median_best_lag=("best_lag", "median"),
        median_relative_log_mse=(
            "Relative_log_MSE",
            "median",
        ),
    )
)

temporal_summary.to_csv(
    f"{PLOTS_DIR}/temporal_diagnostics_summary.csv",
    index=False,
)
display(temporal_summary.round(3))


@nature_style
def plot_temporal_metric_by_role(
    data: pd.DataFrame,
    *,
    metric: str,
    ylabel: str,
    savepath: str,
    binary: bool = False,
) -> None:
    """Bars = mean across sessions ± s.e.m. per model, condition and role; dots = sessions.

    For a 0/1 metric (binary=True) each bar is labelled with the number of sessions meeting
    the criterion (k/11) instead of showing dots.
    """
    selected = data[
        data["condition"].isin(
            ["Baseline", "Heuristic-aware"]
        )
    ].copy()

    family_order = [
        label
        for label in MODEL_PRETTY_NAMES.values()
        if label in set(selected["family"])
    ]
    conditions = ["Baseline", "Heuristic-aware"]
    condition_colors = {"Baseline": TAB_BLUE, "Heuristic-aware": TAB_ORANGE}

    fig, axes = plt.subplots(2, 2, figsize=(FULL_WIDTH, 105 * MM))
    for ax, role in zip(axes.ravel(), ROLES):
        sub = selected[selected["role"] == role]
        set_dot_spacing(data[data["role"] == role][metric].to_numpy(float))
        for gi, family in enumerate(family_order):
            for j, condition in enumerate(conditions):
                values = sub[(sub["family"] == family) & (sub["condition"] == condition)][metric].to_numpy(float)
                x = gi + (j - 0.5) * 0.38
                bar_with_dots(ax, x, values, condition_colors[condition], 0.36, baseline=(j == 0),
                    dot_size=1.6, show_dots=not binary)
                if binary:
                    values = values[np.isfinite(values)]
                    ax.text(x, np.mean(values) + sem(values) + 0.03 if values.size > 1 else 0.03,
                            f"{int(values.sum())}/{values.size}", ha="center", va="bottom", fontsize=4.5, rotation=90)
        if binary:
            ax.set_ylim(0, 1.32)
        ax.set_title(role.capitalize())
        ax.set_xticks(range(len(family_order)))
        ax.set_xticklabels(family_order, rotation=35, ha="right")
        ax.set_ylabel(ylabel)
    handles = [Patch(facecolor=lighten(TAB_BLUE), edgecolor="#555555", hatch=HATCH, label="Baseline"),
               Patch(facecolor=TAB_ORANGE, edgecolor="black", label="Heuristic-aware"),
               (Line2D([], [], ls="", label="$k$/11: sessions meeting the criterion") if binary else
                Line2D([], [], marker="o", ls="", color=DOT_COLOR, alpha=0.75, markeredgewidth=0,
                       markersize=2.5, label="Session ($n=11$)"))]
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3)
    fig.tight_layout(h_pad=1.5, w_pad=1.5)
    save_figure(fig, savepath)
    plt.show()


plot_temporal_metric_by_role(
    temporal_diagnostics,
    metric="model_overshoot_collapse",
    ylabel="Share of sessions with\novershoot-and-collapse",
    savepath=f"{PLOTS_DIR}/overshoot_collapse_rate_by_role.pdf",
    binary=True,
)
plot_temporal_metric_by_role(
    temporal_diagnostics,
    metric="abs_peak_round_error",
    ylabel="Absolute peak-timing\nerror (rounds)",
    savepath=f"{PLOTS_DIR}/peak_timing_error_by_role.pdf",
)
plot_temporal_metric_by_role(
    temporal_diagnostics,
    metric="rmse_late",
    ylabel="Late-period RMSE\n(rounds 25–35)",
    savepath=f"{PLOTS_DIR}/late_period_rmse_by_role.pdf",
)

In [ ]:
def load_role_matrix(
    model_directory: str,
    role: str,
    *,
    sim_indices: Iterable[int] = range(11),
) -> Tuple[np.ndarray, List[str]]:
    trajectories: List[np.ndarray] = []
    session_names: List[str] = []

    for sim_index in sim_indices:
        sim_name = f"sim_{sim_index}"
        try:
            payload = load_simulated_data(
                model_directory,
                sim_name=sim_name,
            )
        except Exception:
            continue

        trajectories.append(
            np.asarray(payload[f"{role}_order"], dtype=float)
        )
        session_names.append(sim_name)

    if not trajectories:
        return np.empty((0, 0)), []

    maximum_length = max(len(item) for item in trajectories)
    matrix = np.full(
        (len(trajectories), maximum_length),
        np.nan,
        dtype=float,
    )
    for row_index, trajectory in enumerate(trajectories):
        matrix[row_index, :len(trajectory)] = trajectory

    return matrix, session_names


def human_target_role_matrix(
    human_param_df: pd.DataFrame,
    simulate_fn,
    role: str,
    *,
    sim_indices: Iterable[int] = range(11),
    weeks: int = 35,
) -> np.ndarray:
    trajectories = []
    for sim_index in sim_indices:
        target_df = simulate_fn(
            **_human_kwargs_for_sim(
                human_param_df,
                sim_index,
                weeks=weeks,
            )
        )["data"]
        trajectories.append(
            target_df[f"{role}_order"].to_numpy(float)
        )
    return np.vstack(trajectories)


@nature_style
def plot_family_individual_trajectory_atlas(
    *,
    family_key: str,
    baseline_directory: str,
    prompt_directory: str,
    human_param_df: pd.DataFrame,
    simulate_fn,
    sim_indices: Iterable[int] = range(11),
    savepath: Optional[str] = None,
):
    """Every session's order trajectory (grey), the LLM session median, and the Sterman target
    median with its 10th-90th percentile band, per role and condition."""
    conditions = [
        ("Baseline", baseline_directory),
        ("Heuristic-aware", prompt_directory),
    ]
    band_color = lighten("#1f77b4", 0.15)
    figure, axes = plt.subplots(len(ROLES), 2, figsize=(FULL_WIDTH, 175 * MM), sharex=True)

    for role_index, role in enumerate(ROLES):
        target_matrix = human_target_role_matrix(
            human_param_df,
            simulate_fn,
            role,
            sim_indices=sim_indices,
            weeks=35,
        )
        target_median = np.nanmedian(target_matrix, axis=0)
        target_lower = np.nanquantile(target_matrix, 0.10, axis=0)
        target_upper = np.nanquantile(target_matrix, 0.90, axis=0)
        target_rounds = np.arange(1, len(target_median) + 1)

        for condition_index, (condition_label, directory) in enumerate(conditions):
            axis = axes[role_index, condition_index]
            model_matrix, session_names = load_role_matrix(
                directory,
                role,
                sim_indices=sim_indices,
            )

            if model_matrix.size == 0:
                axis.text(0.5, 0.5, "No valid sessions", ha="center", va="center", transform=axis.transAxes)
                continue

            rounds = np.arange(1, model_matrix.shape[1] + 1)
            axis.fill_between(target_rounds, target_lower, target_upper, color=band_color, linewidth=0, zorder=1)
            for trajectory in model_matrix:
                axis.plot(rounds, trajectory, color="#9E9E9E", lw=0.35, zorder=2)
            axis.plot(target_rounds, target_median, color="#3aa85f", ls="--", lw=1.1, zorder=3)
            axis.plot(rounds, np.nanmedian(model_matrix, axis=0), color="#ff7f0e", lw=1.3, zorder=4)
            for round_marker in (5, 25):
                axis.axvline(round_marker, color="#555555", ls=":", lw=0.6, zorder=0)

            axis.set_title(f"{role.capitalize()} — {condition_label} ($n={len(session_names)}$)")
            axis.set_ylabel("Orders")
            if role_index == len(ROLES) - 1:
                axis.set_xlabel("Round")

    handles = [Line2D([], [], color="#9E9E9E", lw=0.5, label="Individual session"),
               Line2D([], [], color="#ff7f0e", lw=1.3, label="LLM session median"),
               Line2D([], [], color="#3aa85f", ls="--", lw=1.1, label="Sterman target median"),
               Patch(facecolor=band_color, label="Sterman target 10th–90th percentile"),
               Line2D([], [], color="#555555", ls=":", lw=0.6, label="Rounds 5 and 25")]
    figure.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=5)
    figure.tight_layout(h_pad=1.0, w_pad=1.5)
    save_figure(figure, savepath)
    return figure


for family_key in ["llama3", "llama70b"]:
    baseline_key = f"{family_key}-baseline"
    prompt_key = f"{family_key}-prompt"

    if (
        baseline_key in model_dirs_baseline
        and prompt_key in model_dirs_prompt
    ):
        figure = plot_family_individual_trajectory_atlas(
            family_key=family_key,
            baseline_directory=model_dirs_baseline[baseline_key],
            prompt_directory=model_dirs_prompt[prompt_key],
            human_param_df=human_df_sims,
            simulate_fn=simulate_beer_game,
            sim_indices=range(11),
            savepath=(
                f"{PLOTS_DIR}/temporal_atlas_{family_key}.pdf"
            ),
        )
        plt.show()
        plt.close(figure)


@nature_style
def plot_two_condition_session_heatmap(
    *,
    baseline_directory: str,
    prompt_directory: str,
    role: str,
    sim_indices: Iterable[int] = range(11),
    savepath: Optional[str] = None,
):
    baseline_matrix, baseline_names = load_role_matrix(
        baseline_directory,
        role,
        sim_indices=sim_indices,
    )
    prompt_matrix, prompt_names = load_role_matrix(
        prompt_directory,
        role,
        sim_indices=sim_indices,
    )

    if baseline_matrix.size == 0 or prompt_matrix.size == 0:
        raise RuntimeError("Both conditions require valid sessions.")

    shared_max = float(
        np.nanquantile(
            np.concatenate(
                [baseline_matrix.ravel(), prompt_matrix.ravel()]
            ),
            0.99,
        )
    )
    shared_max = max(shared_max, 8.0)

    figure, axes = plt.subplots(1, 2, figsize=(FULL_WIDTH, 55 * MM), sharey=True)
    for axis, matrix, names, condition in [
        (axes[0], baseline_matrix, baseline_names, "Baseline"),
        (axes[1], prompt_matrix, prompt_names, "Heuristic-aware"),
    ]:
        image = axis.imshow(matrix, aspect="auto", interpolation="nearest", vmin=0, vmax=shared_max,
                            cmap="viridis")
        axis.set_title(condition)
        axis.set_xlabel("Round")
        axis.set_xticks(np.arange(0, matrix.shape[1], 5))
        axis.set_xticklabels(np.arange(1, matrix.shape[1] + 1, 5))
        axis.set_yticks(np.arange(len(names)))
        axis.set_yticklabels([name.replace("sim_", "Session ") for name in names])
        axis.axvline(23.5, color="white", ls="--", lw=0.8)
        axis.spines[["top", "right"]].set_visible(True)

    colorbar = figure.colorbar(image, ax=axes, shrink=0.9, pad=0.02)
    colorbar.set_label("Order quantity")
    colorbar.outline.set_linewidth(0.5)
    save_figure(figure, savepath)
    return figure


if (
    "llama70b-baseline" in model_dirs_baseline
    and "llama70b-prompt" in model_dirs_prompt
):
    llama70_heatmap = plot_two_condition_session_heatmap(
        baseline_directory=model_dirs_baseline["llama70b-baseline"],
        prompt_directory=model_dirs_prompt["llama70b-prompt"],
        role="retailer",
        sim_indices=range(11),
        savepath=(
            f"{PLOTS_DIR}/llama70b_retailer_session_heatmap.pdf"
        ),
    )
    plt.show()
    plt.close(llama70_heatmap)

    llama70_retailer_table = (
        temporal_diagnostics[
            (
                temporal_diagnostics["family_key"] == "llama70b"
            )
            & (
                temporal_diagnostics["condition_key"].isin(
                    ["baseline", "prompt"]
                )
            )
            & (
                temporal_diagnostics["role"] == "retailer"
            )
        ][
            [
                "condition",
                "sim",
                "Relative_log_MSE",
                "model_peak_order",
                "model_peak_round",
                "model_first_zero_round",
                "model_longest_zero_run",
                "model_late_zero_fraction",
                "rmse_early",
                "rmse_middle",
                "rmse_late",
            ]
        ]
        .sort_values(["condition", "sim"])
    )

    llama70_retailer_table.to_csv(
        f"{PLOTS_DIR}/llama70b_retailer_temporal_diagnostics.csv",
        index=False,
    )
    display(llama70_retailer_table.round(3))

    baseline_late_zero = (
        llama70_retailer_table.loc[
            llama70_retailer_table["condition"] == "Baseline",
            "model_late_zero_fraction",
        ]
        .dropna()
        .to_numpy(float)
    )
    prompt_late_zero = (
        llama70_retailer_table.loc[
            llama70_retailer_table["condition"] == "Heuristic-aware",
            "model_late_zero_fraction",
        ]
        .dropna()
        .to_numpy(float)
    )

    if len(baseline_late_zero) and len(prompt_late_zero):
        u_statistic, p_value = mannwhitneyu(
            prompt_late_zero,
            baseline_late_zero,
            alternative="two-sided",
        )
        print(
            "Llama-3.3 70B retailer late-zero fraction\n"
            f"Baseline mean: {baseline_late_zero.mean():.3f}\n"
            f"Heuristic-aware mean: {prompt_late_zero.mean():.3f}\n"
            f"Mann–Whitney U: {u_statistic:.1f}\n"
            f"p-value: {p_value:.4g}"
        )

In [ ]:
@nature_style
def make_equal_score_counterexample(
    *,
    human_param_df: pd.DataFrame,
    simulate_fn,
    sim_index: int = 0,
    role: str = "retailer",
    warmup_weeks: int = 4,
    savepath: Optional[str] = None,
):
    """
    Construct two visibly different trajectories with exactly the same MSE
    as the LOHO predictor. Both have relative log-MSE equal to zero.
    """
    target_df = simulate_fn(
        **_human_kwargs_for_sim(
            human_param_df,
            sim_index,
            weeks=35,
        )
    )["data"]
    loho_df = simulate_fn(
        **_kwargs_from_param_series(
            _loho_series(human_param_df, sim_index),
            weeks=35,
        )
    )["data"]

    key = f"{role}_order"
    target = target_df[key].to_numpy(float)[warmup_weeks:]
    loho = loho_df[key].to_numpy(float)[warmup_weeks:]
    n = len(target)

    mse_loho = float(np.mean((loho - target) ** 2))

    uniform_deviation = np.sqrt(mse_loho)
    candidate_uniform = target + uniform_deviation

    candidate_localized = target.copy()
    spike_index = max(0, n - 6)
    candidate_localized[spike_index] += np.sqrt(mse_loho * n)

    mse_uniform = float(
        np.mean((candidate_uniform - target) ** 2)
    )
    mse_localized = float(
        np.mean((candidate_localized - target) ** 2)
    )

    score_uniform = float(
        np.log(max(mse_loho, 1e-12))
        - np.log(max(mse_uniform, 1e-12))
    )
    score_localized = float(
        np.log(max(mse_loho, 1e-12))
        - np.log(max(mse_localized, 1e-12))
    )

    rounds = np.arange(warmup_weeks + 1, warmup_weeks + 1 + n)
    figure, axis = plt.subplots(figsize=(HALF_WIDTH * 1.4, 60 * MM))
    axis.plot(rounds, target, color="#1f77b4", lw=1.6, label="Session-specific Sterman target")
    axis.plot(rounds, loho, color="#ff7f0e", ls="--", lw=1.2, label="LOHO predictor")
    axis.plot(rounds, candidate_uniform, color="#3aa85f", lw=1.0,
              label=f"Candidate A: distributed bias (score = {abs(score_uniform):.0f})")
    axis.plot(rounds, candidate_localized, color="#d62728", lw=1.0,
              label=f"Candidate B: localized spike (score = {abs(score_localized):.0f})")
    axis.set_xlabel("Round")
    axis.set_ylabel(f"{role.capitalize()} orders")
    axis.legend(loc="upper left")
    save_figure(figure, savepath)

    print(
        {
            "MSE_LOHO": mse_loho,
            "MSE_uniform_candidate": mse_uniform,
            "MSE_localized_candidate": mse_localized,
            "Relative_log_MSE_uniform": score_uniform,
            "Relative_log_MSE_localized": score_localized,
        }
    )
    return figure


counterexample_figure = make_equal_score_counterexample(
    human_param_df=human_df_sims,
    simulate_fn=simulate_beer_game,
    sim_index=0,
    role="retailer",
    warmup_weeks=4,
    savepath=(
        f"{PLOTS_DIR}/equal_score_different_trajectories.pdf"
    ),
)
plt.show()
plt.close(counterexample_figure)


near_parity_cases = (
    temporal_diagnostics
    .assign(
        absolute_relative_log_mse=lambda frame: np.abs(
            frame["Relative_log_MSE"]
        )
    )
    .sort_values("absolute_relative_log_mse")
    [
        [
            "family",
            "condition",
            "sim",
            "role",
            "Relative_log_MSE",
            "MSE_ratio_to_LOHO",
            "rmse_early",
            "rmse_middle",
            "rmse_late",
            "max_abs_error_postshock",
            "peak_order_error",
            "peak_round_error",
            "model_late_zero_fraction",
            "model_overshoot_collapse",
            "target_overshoot_collapse",
            "best_lag",
            "best_lag_correlation",
        ]
    ]
    .head(20)
)

near_parity_cases.to_csv(
    f"{PLOTS_DIR}/near_loho_parity_temporal_diagnostics.csv",
    index=False,
)
display(near_parity_cases.round(3))


role_score_wide = (
    per_sim_all
    .reset_index()
    .pivot_table(
        index=["model", "sim"],
        columns="role",
        values="Relative_log_MSE",
        aggfunc="mean",
    )
)
role_score_wide["OVERALL"] = role_score_wide[ROLES].mean(axis=1)
role_score_wide["role_score_range"] = (
    role_score_wide[ROLES].max(axis=1)
    - role_score_wide[ROLES].min(axis=1)
)
role_score_wide["max_absolute_role_score"] = (
    role_score_wide[ROLES].abs().max(axis=1)
)

overall_near_parity = (
    role_score_wide
    .assign(
        absolute_overall_score=lambda frame: np.abs(
            frame["OVERALL"]
        )
    )
    .sort_values(
        ["absolute_overall_score", "max_absolute_role_score"],
        ascending=[True, False],
    )
    .head(20)
)

overall_near_parity.to_csv(
    f"{PLOTS_DIR}/near_overall_parity_role_cancellation.csv"
)
display(overall_near_parity.round(3))